# <center> Agent 长期记忆系统——mem0 + ClaudeCode集成实战</center>

&emsp;&emsp;今天我们聚焦的是 Agent 记忆系统的**生产级解决方案**——`mem0`。在基础入门课件中，我们用 mini-OpenClaw 从零搭建了一套完整的短期记忆 + 长期记忆系统，理解了上下文窗口管理、消息截断、压缩摘要、`MEMORY.md` 文件存储与 System Prompt 注入等核心机制。那套系统足以帮我们理解原理，但在真实生产环境中，本地文件记忆系统很快会撞上规模、质量和生态三重天花板。<font color=red>`mem0` 正是为了突破这些天花板而生的开源中间件</font>——它用 LLM 做记忆的「裁判」，自动决定什么值得记住、什么需要更新、什么应该遗忘，而不是用硬编码规则来管理记忆。

&emsp;&emsp;结合调研结论，本课件会重点抓住以下四条主线：

- 第一，`mem0` 的 **LLM 裁判机制**——这是它区别于所有传统记忆方案的核心创新；

- 第二，**横向对比选型**——`mem0` 与 LangChain Memory、MemGPT、Zep、Cognee 等方案的定位差异；

- 第三，**从零到可运行的实战路径**——环境配置、基础用法、冲突更新演示；

- 第四，**生产级集成实战**——LangChain Tool 模式、AsyncMemory + FastAPI、双检索注入模式。这四条主线覆盖了从「理解原理」到「能在项目中用起来」的完整路径。

&emsp;&emsp;为了把这些内容讲清楚，我们会按照**从认知到实战再到工程化**的递进主线展开：先回顾基础入门课件的知识基础，建立与 `mem0` 的概念映射；然后深入 `mem0` 的核心架构；接着通过横向对比帮你做出选型决策；随后进入代码实战，从最小示例到生产级集成；最后收束为部署避坑指南和进阶路径。

&emsp;&emsp;在正式展开之前，有必要先建立一个更大的认知框架。2025 年 6 月，Shopify CEO Tobi Lütke 和 Andrej Karpathy 先后推动了一个新术语的共识化——**上下文工程（Context Engineering）**。Karpathy 将其定义为「用恰好合适的信息填充上下文窗口的精妙艺术与科学」，Google DeepMind 的 Philipp Schmid 进一步明确为「设计和构建动态系统，在正确的时间、以正确的格式、提供正确的信息和工具，使 LLM 拥有完成任务所需的一切」。<font color=red>上下文工程覆盖 Agent 完整生命周期的信息管理——从系统指令、对话历史、长期记忆、RAG 检索到工具定义，而记忆管理正是其中最核心的子系统。</font>没有记忆管理，上下文窗口就是一次性的、无状态的，Agent 无法产生连贯行为。基础入门课件中 mini-OpenClaw 的截断、压缩、`MEMORY.md` 注入、`prompt_builder.py` 六层组装——这些本质上都是上下文工程的实践。本课件要做的，是将这套自研实践升级为生产级的 `mem0` 方案。

> 📅 **时效性说明**：本课件基于 `mem0ai v1.0.9`（2026-03-26 发布），LLM 使用 DeepSeek，Embedding 使用 OpenAI `text-embedding-3-small`。若版本更新导致 API 变化，请以 [mem0 官方文档](https://docs.mem0.ai) 为准。

> &emsp;**目标受众**：本课程面向**具备 Python async/await 基础、有一定 LLM 应用开发经验**的后端开发者与 AI 应用工程师。学完本课后，你将能够： 
(1) 理解 mem0 的 LLM 裁判机制并解释其与传统记忆方案的核心差异；
(2) 独立完成 mem0 从安装配置到 LangChain/FastAPI 生产级集成的全流程；
(3) 在 mem0、MemGPT、Zep 等方案中做出合理的技术选型决策。

> &emsp;**前置知识要求**：本课件假设你已完成「大模型 Agent 长短期记忆管理基础入门」课程（7 章），熟悉 STM/LTM 分层架构、压缩摘要机制、`MEMORY.md` 文件存储与 `prompt_builder.py` 注入、`sessions/` 目录隔离等概念。若尚未学习基础入门课件，建议先行完成。

## <center>附录零：环境准备</center>

&emsp;&emsp;在正式开始之前，我们先完成本课件所需的增量环境配置。如果你已完成基础入门课件的环境搭建，这里只需要额外安装 `mem0ai` 并配置双凭证（DeepSeek 做 LLM，OpenAI 做 Embedding）。

**步骤一：安装 mem0 SDK**

&emsp;&emsp;`mem0ai` 是 mem0 的 Python SDK，安装后即可使用开源自托管版本，无需额外部署任何服务。

In [1]:
!pip install -r requirements.txt -q

**步骤二：配置双凭证环境变量**

&emsp;&emsp;`mem0` 的记忆提取依赖 LLM（用于判断「什么值得记」），记忆检索依赖 Embedding 模型（用于向量相似度搜索）。<font color=red>本课件的一个关键设计决策是：LLM 使用 DeepSeek（性价比更高），Embedding 使用 OpenAI（DeepSeek 不提供 Embedding 服务）。</font>因此需要配置两套 API Key。

&emsp;&emsp;在项目根目录创建 `.env` 文件：

```ini
# .env 文件内容（请替换为你的真实 Key）

# DeepSeek —— 用于 LLM 记忆提取与冲突判断
# DEEPSEEK_API_KEY=your_deepseek_api_key_here
# DEEPSEEK_BASE_URL=https://api.deepseek.com
# DEEPSEEK_MODEL=deepseek-chat

# OpenAI —— 仅用于 Embedding（text-embedding-3-small）
# OPENAI_API_KEY=your_openai_api_key_here
```


> ⚠️ **安全提醒**：`.env` 文件包含敏感信息，切勿提交到 Git 仓库。建议在 `.gitignore` 中添加 `.env`。

**步骤三：加载环境变量并验证**

&emsp;&emsp;使用 `python-dotenv` 加载环境变量，并验证两套凭证是否配置正确：

In [1]:
# 加载环境变量（override=True 确保 .env 覆盖系统变量）
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 验证 DeepSeek API Key（LLM 用）
deepseek_key = os.getenv("DEEPSEEK_API_KEY")
if deepseek_key:
    print(f"DeepSeek API Key 加载成功（前8位：{deepseek_key[:8]}...）")
else:
    print("DeepSeek API Key 未找到，请检查 .env 文件")

# 验证 OpenAI API Key（Embedding 用）
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key:
    print(f"OpenAI API Key 加载成功（前8位：{openai_key[:8]}...）")
else:
    print("OpenAI API Key 未找到，请检查 .env 文件")

DeepSeek API Key 加载成功（前8位：sk-d513c...）
OpenAI API Key 加载成功（前8位：sk-proj-...）


**步骤四：验证 mem0 安装**

&emsp;&emsp;确认 `mem0ai` 已正确安装且版本符合预期：

In [2]:
import mem0
print(f"mem0 版本: {mem0.__version__}")

mem0 版本: 1.0.9


&emsp;&emsp;如果看到版本号输出（应为 `1.0.x`），说明环境准备完毕，可以进入正式课程内容。

## <center>第一章：从本地文件到开源——Agent 记忆系统的产业进化</center>

&emsp;&emsp;在基础入门课件中，我们用 mini-OpenClaw 的代码从零实现了一套 Agent 记忆系统：短期记忆用 `sessions/` 目录做 JSON 持久化 + 压缩摘要，长期记忆用 `MEMORY.md` 文件做全文注入（文件增大后可选 LlamaIndex 向量索引做 RAG 检索），最后通过 `prompt_builder.py` 的六层组装将记忆注入到 System Prompt 中。这套方案在教学场景下清晰地展示了记忆系统的设计原理。但当我们把目光投向生产环境——真实的多用户、高并发、长期运行的 Agent 系统——本地文件记忆系统会面临怎样的挑战？这正是本章要回答的核心问题，也是我们引入 `mem0` 的动机起点。

&emsp;&emsp;本章将从三个层面展开：首先回顾基础入门课件的知识基础，建立能力清单；然后分析本地文件记忆系统的三个天花板，制造认知冲突；最后引出 `mem0` 的诞生背景和设计哲学，完成从「自研」到「开源」的认知过渡。

### 1.1 回顾：mini-OpenClaw 的记忆架构

&emsp;&emsp;在深入 `mem0` 之前，让我们先快速回顾基础入门课件中搭建的记忆系统核心组件。这不仅是知识巩固，更是为后续每个 `mem0` 概念建立「对照物」——你会发现，`mem0` 解决的每一个问题，我们在基础入门课件中都已经亲手遇到过。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>基础入门课件 mini-OpenClaw 记忆架构总览</font></p>
<div class="center">

| 层级 | 核心机制 | 基础入门课件章节 | 实现方式 |
|------|---------|------------|---------|
| 短期记忆（STM） | 对话历史 + 消息截断 + 压缩摘要 | 模块二 | `sessions/` JSON 文件 + LLM 摘要 |
| 长期记忆（LTM） | `MEMORY.md` 文件读写 + 可选 RAG 检索 | 模块三-模块四 | Markdown 文件全文注入 / LlamaIndex 向量索引可选 |
| 上下文注入 | System Prompt 六层组装 | 模块五-模块六 | `prompt_builder.py` 多层拼接 |

</div>

&emsp;&emsp;通过基础入门课件的学习，你已经具备以下能力：理解 STM/LTM 的分层职责、实现基于 LLM 的压缩摘要、使用 `MEMORY.md` 文件做长期记忆存储并通过 `prompt_builder.py` 注入 System Prompt、通过 `sessions/` 目录实现会话级数据隔离。这些能力构成了理解 `mem0` 的坚实基础——事实上，`mem0` 的每一个核心组件，都能在基础入门课件中找到对应的概念原型。

&emsp;&emsp;在进入 `mem0` 之前，有一个关键问题需要提前回答：<font color=red>`mem0` 替代的是 mini-OpenClaw 的哪些组件？哪些组件必须保留？</font>下面这张表给出明确边界——这将贯穿本课件所有章节的讨论：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mem0 对 mini-OpenClaw 组件的替代边界</font></p>
<div class="center">

| 组件 | 迁移动作 | 说明 |
|------|----------|------|
| `session_manager.py`（短期记忆） | **保留不动** | 对话历史、消息压缩、会话隔离继续使用 |
| `MEMORY.md` + `memory_indexer.py`（长期记忆） | **整体替代** | 存储和检索均由 mem0 接管（详见 5.4 节） |
| `prompt_builder.py` 第 6 层（长期记忆注入） | **改造** | 从读文件改为接收 mem0 检索结果（详见 5.4 节） |
| `prompt_builder.py` 前 5 层 + 其余组件 | **保留不动** | SOUL/IDENTITY/USER/AGENTS 等与记忆无关 |
| Memory Write Guide（第 7 层） | **移除** | mem0 自动管理写入，不再需要指导 LLM 手动写文件 |

</div>

### 1.2 本地文件记忆系统的三个天花板

&emsp;&emsp;mini-OpenClaw 的记忆系统在教学场景下表现出色，但当我们想把它推向生产环境时，会依次撞上三个天花板。理解这些天花板，才能真正理解 `mem0` 的设计动机。

**第一个天花板：规模天花板——单文件 JSON 扛不住**

&emsp;&emsp;基础入门课件中，短期记忆用 `sessions/` 目录下的 JSON 文件存储，长期记忆用 `memory/MEMORY.md` 单个 Markdown 文件承载。这在几十个用户、几百条记忆的规模下没有问题。但当用户数增长到数千、每个用户的记忆条目达到上千条时，单文件 I/O 成为瓶颈——`MEMORY.md` 越来越大，全文注入 System Prompt 会挤占上下文窗口，即便启用 RAG 模式（`memory_indexer.py`），索引重建的开销也随文件体积线性增长。更关键的是，多用户共享同一个 `MEMORY.md` 文件时缺乏隔离机制，并发写入会导致数据竞争。

**第二个天花板：质量天花板——离线整理 vs 实时冲突解决**

&emsp;&emsp;基础入门课件中，mini-OpenClaw 的记忆质量管理已经做了不少工作：写入环节通过 Memory Write Guide 让 LLM 自主判断写入时机；去重环节在 模块五 的 `ImprovedMemoryManager` 中用 LLM 语义判断替代了关键词匹配；整理环节在 模块四 5.4 节 引入了 `sleep_time_reorganize()` 机制，在对话间歇期对 `MEMORY.md` 做全量扫描、去重、提炼和重组。这套组合已经能维护基本的记忆质量，但存在一个结构性限制——<font color=red>去重和整理是「离线批处理」，不是「实时冲突解决」。</font>当用户说「我最近开始减肥了」时，LLM 会新增一条记忆，但不会在写入时自动识别并更新之前存储的「喜欢甜食」——需要等到下一次 `sleep_time_reorganize()` 运行时才有机会整合。在两次整理之间，互相矛盾的条目会共存，且 sleep-time 整理是全量 LLM 调用，记忆越多成本越高。`mem0` 的 LLM 裁判机制（ADD/UPDATE/DELETE/NONE）则是在**每次写入时实时判断**新旧信息是否冲突，将冲突解决内化为写入流程的一部分，不依赖离线任务。

**第三个天花板：生态天花板——自维护 vs 社区**

&emsp;&emsp;本地文件记忆系统意味着**所有工程复杂度都由你的团队独自承担**。那么 `mem0` 这样的社区项目到底替你分摊了什么？我们逐项拆解：

&emsp;&emsp;**社区替你做的事**：记忆萃取的 LLM Judge 逻辑（ADD/UPDATE/DELETE/NONE 四路裁决 prompt 由社区持续优化，你无需从零设计冲突解决规则）；存储后端适配层（Qdrant、Chroma、Milvus 等向量数据库的胶水代码由框架维护，你只需改配置 dict 即可切换）；框架集成（LangChain Tool wrapper、FastAPI 异步接口等由官方提供，版本跟随上游升级）；多租户隔离（`user_id` / `agent_id` / `run_id` 三维命名空间内置在 API 层面，无需自行设计隔离逻辑）。

&emsp;&emsp;<font color=red>**你仍然要自己做的事**</font>：向量数据库的运维（Qdrant 部署、备份、扩容）、Embedding 模型的选型和部署、LLM API 成本管理、以及最关键的——业务层面的记忆策略设计（什么该记、什么不该记、保留多久）。换句话说，`mem0` 封装的是**记忆管理的中间件工程复杂度**，而非基础设施运维和业务决策。理解这个边界，才能正确评估引入 `mem0` 的真实 ROI。

> ⚠️ **常见误区**：这并不意味着本地文件记忆系统没有价值。基础入门课件中的学习让你理解了底层原理，这些知识在使用 `mem0` 时同样重要——你需要知道 `mem0` 的每个 API 背后在做什么，才能正确地配置和调优它。自研是「理解原理」的最佳路径，开源框架是「生产落地」的最佳路径。

### 1.3 mem0 的诞生：从 Embedchain 到记忆中间件

&emsp;&emsp;`mem0`（发音 "mem-zero"）由 Taranjeet Singh 和 Deshraj Yadav 于 2023 年创立。两人此前共同构建了 `Embedchain`——一个拥有超过 200 万次下载量的开源 RAG 框架。在开发 `Embedchain` 的过程中，他们深刻感受到了 LLM 无状态性带来的根本缺陷：每次对话结束后 LLM 对用户一无所知，用户需要不断重复偏好和背景信息，token 成本随对话增长持续上升。

&emsp;&emsp;这段经历催生了一个核心洞察：**需要一个专门的「记忆中间层」**，而不是让每个 Agent 系统自己解决记忆问题。`mem0` 就是这个中间层——它不替代 LLM，而是作为 LLM 与应用之间的记忆基础设施。官方github地址：https://github.com/mem0ai/mem0


<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401224846042.png" width=60%>
</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mem0 项目发展里程碑</font></p>
<div class="center">

| 时间 | 里程碑 | 意义 |
|------|--------|------|
| 2023 年 | 项目创立，从 Embedchain 经验中孵化 | 从 RAG 框架到记忆中间件的转型 |
| 2024 年 6 月 | 进入 Y Combinator S24 批次 | 顶级孵化器背书 |
| 2024 年 | 从 Embedchain 重命名为 mem0 | 品牌聚焦记忆管理 |
| 2025 年 4 月 | 发表 arXiv 论文（arXiv:2504.19413） | 学术验证，LOCOMO benchmark 数据 |
| 2025 年 10 月 | 完成 2400 万美元融资 | 商业可持续性保障 |
| 2025 年底 | GitHub Stars 超 51K，PyPI 下载超 1400 万 | 社区验证 |
| 2025 年 | 被 AWS Agent SDK 选为官方唯一记忆提供商 | 企业级认可 |
| 2026 年 3 月 | v1.0.9 发布 | 当前最新稳定版 |

</div>

&emsp;&emsp;从时间线可以看出，`mem0` 在短短两年内完成了从项目孵化到企业级合作的全过程。51.4K Stars 和 AWS 官方合作这两个数据点，为项目的生产可用性提供了强有力的信号。

&emsp;&emsp;理解了 `mem0` 的诞生背景，我们进一步提炼它的核心设计哲学——这五条原则决定了框架的适用边界和演进方向：

&emsp;&emsp;**原则一：名称即愿景**。`mem0` = memory + zero，将记忆管理的额外开销趋近于零——所有架构决策都在追求「对开发者透明、对用户无感、对系统低成本」。

&emsp;&emsp;**原则二：混合存储**。向量数据库负责语义检索，图数据库负责关系推理，KV 存储负责精确查找——三者组合，各司其职。

&emsp;&emsp;**原则三：LLM 作为裁判**。不用规则引擎做记忆萃取，而是用 LLM 判断「什么值得记」「新旧记忆是否冲突」。这是 `mem0` 最核心的创新。

&emsp;&emsp;**原则四：应用层中间件**。不替代 LLM，不替代 Agent 框架，只负责记忆的存储、检索和演化。可接入 LangChain、LangGraph、FastAPI 或任何自建系统。

&emsp;&emsp;**原则五：开源优先**。核心功能 Apache 2.0 开源，云服务作为增值选项，数据不出本地。

### 1.4 OpenClaw 记忆生态的真实版图：mem0 在哪一格？

&emsp;&emsp;在进入 `mem0` 的技术细节之前，有一个认知前提必须先建立：<font color=red>`mem0` 并不是 Agent 记忆的唯一方案，甚至不是所有 Agent 框架的默认方案。</font>以 OpenClaw为例——这个我们在基础入门课件中反复参照的生产级 Agent 框架，它的记忆体系是一个多方案并存的插件生态，`mem0` 只是其中一个可选项。理解这个生态版图，能帮你在后续学习中始终保持清醒的定位感：我们学的是一种方案，而不是唯一的方案。

&emsp;&emsp;**MEMORY.md 仍然是 OpenClaw 的核心设计哲学。** OpenClaw 对长期记忆的官方定位是"cheat sheet"——一份精炼的速查表，建议控制在 100 行以内，只保留对 Agent 行为最关键的用户偏好、项目上下文和工作约定。同时，OpenClaw 用 `memory/YYYY-MM-DD.md` 格式维护每日工作日志，由 pre-compaction flush 机制自动写入。官方推荐的工作流是：每周从日志中提炼精华，手动 distill 到 MEMORY.md 中（"distill daily logs into MEMORY.md"）。这套机制的核心优势在于**完全透明**——所有记忆都是人类可读、可编辑、可 git diff 的 Markdown 文件，不存在任何黑盒。

&emsp;&emsp;**OpenClaw 自身的记忆演进方向是 memorySearch（MIT 开源）。** 这个官方增强方案的设计哲学可以用一句话概括："Markdown files as source of truth, vector index as cache"。它的存储仍然是 Markdown 文件（`.memsearch/memory/YYYY-MM-DD.md`），但在 Markdown 之上加了一层向量语义搜索索引，支持 OpenAI、Gemini、Ollama、本地 GGUF 模型等多种 Embedding 后端。通过 shell hooks 自动捕获对话内容并索引，实现了在不改变存储格式的前提下获得语义检索能力。此外 memorySearch 还引入了 temporal decay（时间衰减）和 MMR（最大边际相关性）机制，让检索结果兼顾相关性和多样性。

&emsp;&emsp;**mem0 在 OpenClaw 中以第三方插件形式集成。** 安装方式为 `openclaw plugins install @mem0/openclaw-mem0`，安装后为 Agent 提供一组记忆读写工具（具体 API 将在第四至五章展开）。每轮对话自动触发 Auto-Recall（检索相关记忆注入上下文）+ Auto-Capture（从对话中提取关键事实保存），这与我们在 5.1 节将要学到的 LangChain Tool 模式异曲同工。

&emsp;&emsp;这张生态版图的核心洞察是：<font color=red>不同方案在「透明度」和「智能度」之间做了不同的取舍。</font>MEMORY.md 透明度最高但完全依赖人工维护；memorySearch 在透明存储上加了自动索引；mem0 则走向了另一个极端——用 LLM 裁判实现全自动记忆管理，但存储不再是人类直接可读的 Markdown。没有绝对的好坏，只有场景匹配度的高低。各方案的详细参数对比将在第三章 3.1 节总表中呈现。

### 1.5 "大上下文窗口 ≠ 记忆"——数据驱动的核心洞察

&emsp;&emsp;截至 2026 年初，主流模型的上下文窗口已大幅扩展——GPT-5.4 支持 200K tokens，Gemini 3.1 Pro 和 Claude Opus 4.6 均达到 1M tokens，Meta Llama 4 Scout 更是突破了 10M tokens。既然上下文窗口已经这么大，为什么还需要 `mem0` 这样的记忆系统？这是学员最常提出的质疑，也是本课件最核心的认知冲突点。我们用数据来回答这个问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>全量上下文 vs mem0 记忆层：关键指标对比</font></p>
<div class="center">

| 指标 | 全量上下文（Full-Context） | mem0（向量模式） | 差距 |
|------|-------------------------|----------------|------|
| 平均每轮 token 用量 | ~26,031 tokens | ~1,764 tokens | **节省 >93%** |
| p95 搜索延迟 | 17.12 秒 | 0.200 秒 | **快约 86 倍** |
| p50 搜索延迟 | 9.87 秒 | ~0.08 秒 | **快 123 倍** |
| LOCOMO 综合得分 | 60%（参考） | 67.13% | **高 7 个百分点** |

</div>

> 📅 **时效性说明**：以上数据来自 2025 年 4 月发表的 arXiv 论文（arXiv:2504.19413），测试使用 GPT-4o-mini，温度 0。不同 LLM 和版本结果可能有差异。

> 📌 **术语说明**：p95（第 95 百分位延迟）表示在所有请求中，有 95% 的请求延迟低于该值，剩余 5% 的请求延迟高于该值。p50 即中位数延迟。生产环境通常用 p95/p99 而非平均值来衡量系统稳定性，因为平均值会掩盖偶发的慢请求——而正是这些慢请求最容易引发用户投诉。

&emsp;&emsp;这组数据揭示了一个反直觉的事实：<font color=red>把所有对话历史塞进上下文窗口，不仅更慢、更贵，而且效果还更差。</font>原因在于，全量上下文中大量信息是噪音——闲聊、重复、过时的偏好——LLM 需要在海量文本中「大海捞针」，这远比从精炼的记忆库中检索困难得多。

&emsp;&emsp;这组论文数据与我们在基础入门课件中的实践经验完全吻合——模块二中我们已经实现了全量上下文与 RAG 检索的动态切换（`prompt_builder.py` 的 `_build_context()` 方法），当 `MEMORY.md` 体积增长时自动从全量注入降级为向量检索。`mem0` 的核心价值在于将这个思路推向更高层次：不仅做检索降本，还在写入环节引入 LLM 裁判机制（ADD/UPDATE/DELETE/NONE），实现记忆的**实时冲突解决和自动演化**——这是我们基础课件的 `sleep_time_reorganize()` 离线整理所无法覆盖的能力。

&emsp;&emsp;换句话说，基础课件解决了「怎么存、怎么检索」，`mem0` 进一步解决了「怎么在写入时就判断新旧信息是否冲突、是否需要更新」。这就是我们下一章要详细讲解的「LLM 裁判机制」。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175501076.png" width=50%>
</div>

<!-- [图6 生成提示词]
Visual Style: Split-screen conceptual comparison, isometric 3D, high contrast between chaos and order
Environment: Left side: cluttered, dim, chaotic feel with warm amber warning tones. Right side: clean, bright, efficient feel with cool neon blue-green tones. Dark background #0A0E1A overall.
Metaphor Mapping: Left panel (全量塞入): A massive overflowing warehouse/container labeled 200K tokens stuffed with glowing and dim data blocks — most are gray and dim (noise: idle chat, duplicates, outdated preferences), only a few scattered bright gems (key facts buried in noise). An overwhelmed LLM figure at the bottom trying to search through the mess, with scattered unfocused beams. Performance label floating: 慢/贵/差. Right panel (精炼检索): A sleek precision retrieval platform with a small curated collection of bright crystalline gems representing 精炼记忆, each gem clearly labeled and organized. A confident LLM core at the bottom with a single precise beam accessing exactly the right gem. Performance label floating: 快/省/准. A small mem0 logo/badge on the retrieval platform.
Spatial Layout: Left-right split, 45%/10%/45%. Left shows chaotic warehouse with scattered data and struggling LLM at bottom. Right shows organized retrieval shelf with efficient LLM at bottom. Center divider with VS symbol. Performance labels floating above each panel.
Dynamic Flow: Left: scattered, unfocused gray arrows going everywhere from warehouse to LLM. Right: single precise bright beam from curated memory shelf to LLM core.
Material & Texture: Left: rough matte blocks, cluttered stacking, dim colors, dust particles. Right: crystalline gems, Glassmorphism shelf, holographic precision display, clean glow. Maximum contrast.
Chinese Labels: 全量上下文, 精炼记忆, 噪音信息, 关键事实, 慢/贵/差, 快/省/准
Text Constraint: Chinese text must be perfectly and accurately rendered, no gibberish. Use these exact labels ONLY: 全量上下文, 精炼记忆, 噪音信息, 关键事实, 慢/贵/差, 快/省/准, mem0
Output: 16:9, 1920x1080, 课件概念对比插图
-->


## <center>第二章：mem0 核心架构——LLM 裁判与记忆演化</center>

&emsp;&emsp;第一章建立了「为什么需要 `mem0`」的认知基础，现在我们进入本课件最核心的理论章节：`mem0` 是如何工作的。理解这一章的内容，是正确使用 `mem0` 的前提——你需要知道每次 `memory.add()` 和 `memory.search()` 背后发生了什么，才能做出合理的架构决策。

&emsp;&emsp;本章将围绕四个核心问题展开：记忆存储在哪里（三层存储 + 命名空间隔离）、记忆如何从对话中提取（LLM 裁判两阶段流程）、记忆如何演化和更新（ADD/UPDATE/DELETE/NONE 四操作）、以及可选的增强能力（图谱记忆、冲突解决）。

&emsp;&emsp;在深入之前，先建立基础入门课件知识与 `mem0` 概念的映射关系。这张映射表是理解本章最关键的参照工具——你会发现，`mem0` 的每一个核心概念，在基础入门课件中都有对应的原型：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>基础入门课件概念 → mem0 实现映射表（仅长期记忆层）</font></p>
<div class="center">

| 基础入门课件概念 | 基础入门课件位置 | mem0 对应实现 | 升级要点 |
|-------------|------------|-------------|----------|
| `MEMORY.md` 全文注入 + 可选 RAG | 模块四 5.3 节 + 模块三 4.1 节 | 向量 DB 语义检索 + `memory.search()` | 从文件全文注入升级为自动向量检索 |
| LLM 判断写入 + 离线 sleep-time 整理 | 模块四 5.1 节 + 模块四 5.4 节 | LLM 裁判两阶段提取流程（ADD/UPDATE/DELETE/NONE） | 从离线批处理升级为实时冲突解决 |
| 图数据库（可选） | 模块三 4.3 节 | Graph Memory（Neo4j/Memgraph） | 从概念介绍升级为开箱即用 |

</div>




> &emsp;**短期记忆层不变**：`sessions/` 目录隔离、`SessionManager` 三阶段架构（load → get_messages → update）、压缩摘要机制——这些全部保留，mem0 不接管短期记忆。mem0 的 `user_id`/`agent_id`/`run_id` 三维命名空间是**长期记忆的隔离机制**，与 `sessions/` 的会话级隔离并存于不同层级。

### 2.1 三层存储与命名空间隔离

&emsp;&emsp;`mem0` 在架构文档和论文中描述了三类记忆范畴。在开源 SDK 的实际实现中，这三者并非独立的物理存储层，而是通过 `user_id` / `agent_id` / `run_id` 三个命名空间参数实现逻辑隔离——物理上它们共享同一个向量数据库实例。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mem0 三维命名空间与记忆类型</font></p>
<div class="center">

| 参数 | 语义 | 生命周期 | 典型用法 | 前序对照 |
|------|------|---------|---------|---------|
| `user_id` | 用户级隔离 | 永久（可手动删除） | 不同用户的个人偏好 | `sessions/{user_id}/` 目录 |
| `agent_id` | Agent 级隔离 | 与 Agent 生命周期绑定 | 不同 Agent 的专业知识域 | 无直接对照（前序为单 Agent） |
| `run_id` | 会话级隔离 | 会话结束后可归档 | 单次对话的临时上下文 | `sessions/{user_id}/{session_id}.json` |

</div>

&emsp;&emsp;三个维度可以任意组合，提供从粗粒度到细粒度的灵活隔离：

&emsp;&emsp;理解了三维命名空间的理论模型后，我们通过一段实际代码来验证——对同一段对话，分别使用不同的命名空间参数调用 `memory.add()`，观察记忆如何被隔离存储。首先安装本节需要的向量数据库依赖：


In [3]:
import shutil, os
from dotenv import load_dotenv
from mem0 import Memory

load_dotenv()

QDRANT_PATH="./qdrant"

# 清理 Qdrant 本地存储残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# 准备示例对话（mem0 要求 OpenAI 格式的消息列表）
messages = [
    {"role": "user", "content": "我正在学习用 FastAPI 搭建后端服务"},
    {"role": "assistant", "content": "FastAPI 是一个非常好的选择，它基于 Pydantic 和 Starlette，性能优秀且类型安全。"},
]

config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "deepseek-chat",
            "api_key": os.getenv("DEEPSEEK_API_KEY"),
            "openai_base_url": "https://api.deepseek.com/v1",
            "temperature": 0.1,
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "version": "v1.1"
}

memory = Memory.from_config(config)

# 用户记忆：跨会话持久
memory.add(messages, user_id="alice")

# Agent 记忆：与特定 Agent 绑定
memory.add(messages, agent_id="customer_support_bot")

# 会话记忆：仅本次会话
memory.add(messages, user_id="alice", run_id="session_2026_03_29")

# 三维组合：最精细的隔离
memory.add(messages, user_id="alice", agent_id="travel_bot", run_id="trip_planning_001")

{'results': [{'id': 'd31dd997-8958-4391-98df-5c51245637e8',
   'memory': '认为FastAPI是一个非常好的选择，基于Pydantic和Starlette，性能优秀且类型安全',
   'event': 'ADD'}]}

&emsp;&emsp;记忆写入完成。接下来通过 `get_all()` 查询特定用户的所有记忆，验证命名空间隔离效果：


In [4]:
# 找出所有带有 alice 标签的记忆
alice_memories = memory.get_all(user_id="alice")

print("打印原始存储信息：")
print(alice_memories)
print("-" * 30)
print("alice 的记忆：")
for mem in alice_memories["results"]:
    # mem0 会自动把复杂的 payload 整理成人类可读的字典
    print(f"记忆内容: {mem['memory']}")
    print(f"附带属性: {mem['user_id']} | {mem.get('agent_id', '空')}")
    print("-" * 30)


打印原始存储信息：
{'results': [{'id': 'd31dd997-8958-4391-98df-5c51245637e8', 'memory': '认为FastAPI是一个非常好的选择，基于Pydantic和Starlette，性能优秀且类型安全', 'hash': '34fdf1f655230afc4e14c49daf6cc571', 'metadata': None, 'created_at': '2026-04-02T10:31:06.827548+00:00', 'updated_at': '2026-04-02T10:31:06.827548+00:00', 'user_id': 'alice', 'agent_id': 'travel_bot', 'run_id': 'trip_planning_001'}, {'id': 'd6324726-3cdd-4a72-aa27-75fcd85b7f87', 'memory': '正在学习用 FastAPI 搭建后端服务', 'hash': '2a432b5d56e149679f535cc77426e892', 'metadata': None, 'created_at': '2026-04-02T10:31:00.685115+00:00', 'updated_at': '2026-04-02T10:31:00.685115+00:00', 'user_id': 'alice', 'run_id': 'session_2026_03_29'}, {'id': 'f3d91ec0-7822-4da3-b54b-7f6f5bf18d0d', 'memory': '正在学习用 FastAPI 搭建后端服务', 'hash': '2a432b5d56e149679f535cc77426e892', 'metadata': None, 'created_at': '2026-04-02T10:30:47.380932+00:00', 'updated_at': '2026-04-02T10:30:47.380932+00:00', 'user_id': 'alice'}]}
------------------------------
alice 的记忆：
记忆内容: 认为FastAPI是一个非常好的选择

&emsp;&emsp;单维度查询只能看到部分记忆。下面用 `pandas` 汇总四种 ID 组合的查询结果，形成一张全景对照表：


In [5]:
import pandas as pd

# 分维度查询，汇总为一张全景表格
rows = []
queries = [
    ("user_id",  {"user_id": "alice"}),
    ("agent_id", {"agent_id": "customer_support_bot"}),
]
for label, kwargs in queries:
    for item in memory.get_all(**kwargs)["results"]:
        # 提取每条记忆的核心字段，缺失字段用 "-" 填充
        rows.append({
            "memory": item["memory"],          # 记忆内容（LLM 裁判提取的摘要）
            "user_id": item.get("user_id", "-"),  # 用户维度
            "agent_id": item.get("agent_id", "-"),# Agent 维度
            "run_id": item.get("run_id", "-"),    # 会话维度
        })

df = pd.DataFrame(rows).drop_duplicates()
df

,memory,user_id,agent_id,run_id
0,认为FastAPI是一个非常好的选择，基于Pydantic和Starlette，性能优秀且类型安全,alice,travel_bot,trip_planning_001
1,正在学习用 FastAPI 搭建后端服务,alice,-,session_2026_03_29
2,正在学习用 FastAPI 搭建后端服务,alice,-,-
3,认为 FastAPI 是一个非常好的选择，基于 Pydantic 和 Starlette，性...,-,customer_support_bot,-


&emsp;&emsp;这段代码展示了 `mem0` 命名空间的四种组合模式。注意：不带 `run_id` 的 `user_id` 记忆是跨会话持久的（长期记忆），带 `run_id` 的则按会话粒度隔离。<font color=red>注意：`run_id` 隔离的仍然是 mem0 内部的长期记忆条目，不等同于 `sessions/` 管理的短期对话历史——后者由 `SessionManager` 独立管理，不受 mem0 影响。</font>

> &emsp;**为什么同样的 messages，不同行提取出的记忆内容可能不同？** 因为每次 `memory.add()` 都会独立调用一次 LLM 裁判来提取关键事实（详见 2.2 节）。LLM 的语义理解具有非确定性，因此提取结果可能存在细微差异——这是 LLM 驱动记忆系统的固有特性。


> ⚠️ **常见误区**：不要把三维 ID 理解为物理存储分离。在开源版中，无论使用哪种 ID 组合，数据都存储在同一个向量数据库实例中，通过 metadata 过滤实现隔离。这是一种**逻辑隔离**，而非物理隔离。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175502663.png" width=60%>
</div>

### 2.2 核心机制预览：从 LLM 裁判到记忆演化

&emsp;&emsp;2.1 节展示了 `mem0` 的三维命名空间结构。本节预览 `mem0` 的四个核心机制——LLM 裁判、四种操作、向量存储后端、冲突解决——建立整体认知框架。每个机制的详细原理和代码验证将在第四章实战中展开。

**机制一：LLM 裁判（第四章 4.2 节代码验证）**

&emsp;&emsp;每次调用 `memory.add()` 时，触发两阶段流程：**提取阶段**——LLM 从对话中萃取候选记忆（过滤闲聊和噪音）；**更新阶段**——LLM 将候选记忆与现有记忆比对，选择 ADD/UPDATE/DELETE/NONE 四种操作之一。这解释了为什么 `add()` 不是瞬时操作（500-2000ms 延迟），也解释了记忆质量高度依赖底层 LLM 能力。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175542131.png" width=60%>
</div>

<!-- [图2 生成提示词]
Visual Style: Futuristic isometric 3D tech illustration, high-end cyber aesthetic, pipeline flow design
Environment: Dark deep-space background #0A0E1A, neon blue and amber accents, atmospheric glow
Metaphor Mapping: Left entry point: a glowing data injection portal representing 对话消息输入. First processing station: a luminous judges scale/tribunal representing Extraction Phase (LLM 提取) — extracting candidate memories from raw conversation. Floating holographic memory fragments above the tribunal representing 候选记忆. Second processing station: a four-channel splitter/router with four distinct color-coded output paths representing Update Phase: green channel for ADD, blue channel for UPDATE, red channel for DELETE, gray bypass for NONE. Terminal: a glowing data core representing 向量数据库. A separate dim blue module floating to the side representing 滚动摘要(异步) with a dashed connection line.
Spatial Layout: Left-to-right horizontal pipeline flow. Entry portal on far left, Extraction tribunal at 30%, candidate fragments floating above, Update splitter at 60%, four channels converging at 80% into the vector DB core on far right. Async summary module floating above-right with dashed line.
Dynamic Flow: Show a primary glowing data stream flowing left-to-right from input portal through extraction tribunal, splitting into four color-coded channels at the update phase, then converging into the vector DB core. Show a secondary dashed async stream branching from extraction to the floating summary module.
Material & Texture: Glassmorphism panels for processing stations, holographic floating memory fragments with frosted edges, metallic crystalline texture for the vector DB core, glowing energy channels with distinct colors (green/blue/red/gray), cinematic lighting
Chinese Labels: 对话消息, LLM提取, 候选记忆, 存量比对, ADD, UPDATE, DELETE, NONE, 向量数据库, 滚动摘要
Text Constraint: Chinese text must be perfectly and accurately rendered, no gibberish. Use these exact labels ONLY: 对话消息, LLM提取, 候选记忆, 存量比对, ADD, UPDATE, DELETE, NONE, 向量数据库, 滚动摘要
Output: 16:9, 1920x1080, 课件流程插图
-->


**机制二：四种操作——记忆如何“进化”（第四章 4.4 节代码验证）**

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>LLM 裁判的四种操作决策</font></p>
<div class="center">

| 操作 | 含义 | 触发条件 | 示例 |
|------|------|---------|------|
| `ADD` | 新知识，直接加入 | 向量 DB 中无语义相似记忆 | 首次提到「Alice 是素食主义者」 |
| `UPDATE` | 与现有记忆有重叠，更新旧条目 | 发现相似但信息更完整的记忆 | 「Alice 是素食主义者」→「Alice 是严格素食主义者，不吃乳制品」 |
| `DELETE` | 新信息与旧记忆矛盾，删除旧的 | 新旧记忆语义冲突 | 旧：「Alice 喜欢辣食」→ 新：「Alice 不能吃辣」→ 删除旧记忆 |
| `NONE` | 无变化，跳过 | 候选记忆与现有记忆完全一致 | 重复提到已知偏好 |

</div>

&emsp;&emsp;与基础入门课件的批处理式整理（`sleep_time_reorganize()`）不同，`mem0` 的记忆是「实时进化」的——每次写入时 LLM 裁判就会自动处理冲突，不需要等待离线任务。所有操作记录在 SQLite history 表中，可通过 `memory.history()` 追踪变更轨迹。

**机制三：向量存储后端**

&emsp;&emsp;`mem0` 支持 12 种向量数据库后端：零配置本地（Qdrant 默认、FAISS）、自托管服务（Chroma、Weaviate、Milvus）、全托管云（Pinecone、Zilliz Cloud）、图数据库可选扩展（Neo4j/Memgraph，非默认启用）。第四章 4.5 节将演示 Chroma、Qdrant、Milvus 三种后端的配置切换。

> 🔥 **踩坑预警**：默认配置下 Qdrant 数据存储于 `/tmp/qdrant`，容器重启或系统重启后会被清空。生产环境必须配置持久化路径，详见第六章。

**机制四：记忆冲突解决**

&emsp;&emsp;冲突解决内嵌于 Update Phase：新记忆提取后，向量检索找出相似旧记忆 → LLM 判断是否语义矛盾 → 矛盾则 DELETE 旧 + ADD 新，互补则 UPDATE 合并。<font color=red>冲突解决质量完全依赖 LLM 的语义理解能力。</font>第四章 4.4 节将通过三轮实验（ADD → UPDATE → NONE）直观验证这个过程。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175501054.png" width=60%>
</div>

<!-- [图7 生成提示词]
Visual Style: Futuristic tech flowchart, isometric 3D, dark theme with focused glow effects
Environment: Dark background #0A0E1A, neon teal and amber accents, clean layout with atmospheric depth
Metaphor Mapping: Entry: a new memory fragment (glowing amber crystal) labeled 新记忆 entering the system. First station: a vector search beacon emitting scanning waves, finding top-s matches from existing memories — labeled 向量相似检索. Second station: a glowing judges scale representing LLM语义判断, comparing old memory (dim) vs new memory (bright). Three output paths diverging from the judge: Path 1 (red glow): old memory dissolves and new one takes its place — 矛盾 → DELETE旧+ADD新. Path 2 (blue glow): two fragments merge into one enriched crystal — 互补 → UPDATE合并. Path 3 (gray, dim): new fragment fades away — 重复 → NONE跳过. A glowing log book floating to the right representing history()审计日志, connected by dashed lines to all three paths.
Spatial Layout: Top-to-bottom main flow with three branching outputs at the bottom. New memory enters from top center, flows through search station (30% height) and judgment station (55% height), then branches into three colored paths (80% height). Audit log module floating to the right at 60% height, connected by dashed lines.
Dynamic Flow: Single amber stream flowing down through search and judgment stages, then splitting into three distinct colored channels (red/blue/gray). Dashed connection lines from each channel to the audit log. Existing memories shown as dim crystals being compared alongside the bright new one at the judgment stage.
Material & Texture: Crystalline memory fragments (amber for new, dim blue for existing), Glassmorphism judgment panel with scale motif, glowing energy channels with distinct colors, metallic log book with holographic pages, cinematic lighting
Chinese Labels: 新记忆, 相似检索, LLM判断, 矛盾, 互补, 重复, DELETE+ADD, UPDATE, NONE, 审计日志
Text Constraint: Chinese text must be perfectly and accurately rendered, no gibberish. Use these exact labels ONLY: 新记忆, 向量相似检索, LLM语义判断, 矛盾, 互补, 重复, DELETE旧+ADD新, UPDATE合并, NONE, history()
Output: 16:9, 1920x1080, 课件流程插图
-->


## <center>第三章：mem0 优势与选型边界</center>

&emsp;&emsp;第二章从机制角度剖析了 `mem0` 的工作原理，现在我们从选型角度出发：在面对实际项目时，`mem0` 适合在哪些场景使用，以及它的局限在哪里？本章聚焦三个核心问题：六维横向对比、优势量化、已知局限与边界。

&emsp;&emsp;建议先快速浏览 3.1 节的六维总结表建立全局印象，3.2 节的量化数据加深理解，3.3 节的局限分析则帮助你在边界场景做出更准确的技术决策。

### 3.1 六维对比总表与选型决策树

&emsp;&emsp;经过逐一对比之后，让我们用一张总表收束全部信息。在阅读这张表之前，先解释一个关键维度——`时序推理`（Temporal Reasoning）。时序推理是指 Agent 能够理解记忆中的**时间顺序和因果关系**的能力：比如用户三个月前说"我在学 Python"，上周说"我转学 Go 了"，Agent 需要知道后者覆盖了前者，而不是把两条信息当作并列事实。具备强时序推理能力的框架（如 Zep/Graphiti）会用时序知识图谱记录事件发生的先后顺序和实体状态变迁；而时序推理能力较弱的框架则主要依赖向量相似度检索，<font color=red>无法区分「过时信息」和「当前事实」</font>。

&emsp;&emsp;这张表是你做技术选型时最直接的参考工具：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Agent 记忆管理方案六维横向对比总表</font></p>
<div class="center">

| 框架 | Stars | 定位 | 开源 | 时序推理 | 接入难度 | 推荐场景 |
|------|-------|------|------|---------|---------|----------|
| **mem0** | 51.4K | 个性化记忆层 | Apache 2.0 | 中（图模式） | **极低** | 多用户产品、聊天机器人 |
| Letta/MemGPT | 21K | OS 式内存管理 | Apache 2.0 | 低 | 高 | 文档分析、透明调试 |
| Zep/Graphiti | 24K | 时序知识图谱 | 部分开源 | **高** | 中 | 时间敏感的实体追踪 |
| Cognee | 12K | 深度知识图谱 | 开源核心 | 高 | 中 | 复杂推理应用 |
| LangChain Checkpointer | - | 会话状态管理 | Apache 2.0 | 无 | 极低 | 短期对话状态持久化（需配合外部长期记忆方案） |

</div>

&emsp;&emsp;基于以上对比，可以总结出一个简单的**选型决策路径**：

&emsp;&emsp;1. 如果你需要**多用户个性化对话记忆** + **快速接入** → 选 `mem0`

&emsp;&emsp;2. 如果你需要 **Agent 自主管理内存** + **透明调试** → 选 Letta/MemGPT

&emsp;&emsp;3. 如果你需要**强时序推理** + **时态实体追踪** → 选 Zep/Graphiti

&emsp;&emsp;4. 如果你需要**深度知识推理** + **科学/法律场景** → 选 Cognee

&emsp;&emsp;5. 如果你已有 LangChain Agent 但**缺乏跨会话长期记忆** → 用 `mem0` 补全（参见 5.1 节 Tool 模式集成）

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175502649.png" width=40%>
</div>

<!-- [图3 生成提示词]
Visual Style: Clean tech decision tree diagram, minimal but modern, subtle 3D depth
Environment: Dark background #0D1117, subtle grid lines, soft neon blue glow on nodes
Metaphor Mapping: A glowing start node at top representing 选型起点. Diamond-shaped glowing decision nodes representing key questions with Yes/No branches. Rounded terminal nodes at the bottom representing recommended solutions: mem0 (bright blue, recommended highlight), Claude Code Memory / 文件系统方案 (amber), Zep/Graphiti (teal), MemGPT (purple).
Spatial Layout: Top-to-bottom vertical decision tree. Start node at top center. First diamond 需要多用户支持? branches left (No → 文件系统方案) and right (Yes). Right branch leads to second diamond 需要时序推理?, branching left (No) and right (Yes → Zep/Graphiti). No-branch leads to third diamond Agent自管内存?, branching right (Yes → MemGPT) and left (No → mem0, highlighted as recommended).
Dynamic Flow: Glowing pathway lines connecting nodes, with the recommended path (to mem0) highlighted in brighter neon blue. Branch labels Yes/No at each fork.
Material & Texture: Frosted glass effect on decision diamonds, soft glow halos around terminal nodes, clean thin connection lines with subtle glow, professional UI/UX vibe
Chinese Labels: 选型起点, 多用户支持?, 时序推理?, Agent自管内存?, mem0, 文件系统方案, Zep/Graphiti, MemGPT
Text Constraint: Chinese text must be perfectly and accurately rendered, no gibberish. Use these exact labels ONLY: 选型起点, 多用户支持?, 时序推理?, Agent自管内存?, mem0, 文件系统方案, Zep/Graphiti, MemGPT, Yes, No
Output: 16:9, 1920x1080, 课件决策树插图
-->


### 3.2 mem0 的优势量化

&emsp;&emsp;在确认了 `mem0` 的定位之后，让我们用数据量化它的核心优势，建立更具体的期望：

&emsp;&emsp;**精度领先**：在 LOCOMO benchmark 上，`mem0` 的 LLM-as-a-Judge 总分为 67.13%，而 OpenAI 内置 Memory 为 52.9%，相对提升 **+26%**。这意味着在相同的对话场景下，`mem0` 能更准确地检索出与当前问题相关的历史记忆。

&emsp;&emsp;**延迟极低**：p95 搜索延迟仅 0.200 秒，比全量上下文方案的 17.12 秒低 **91%**。对于实时对话应用来说，这个差距意味着用户几乎感受不到记忆检索带来的延迟。

&emsp;&emsp;**Token 节省**：平均每轮对话约 1,764 tokens（全量上下文需 26,031 tokens），节省超过 **93%**。按 DeepSeek 的定价（约 ￥0.001/1K tokens，📅 2026年3月报价，请以官网最新定价为准）估算，每 10 万次对话可节省约 ￥2,400。

&emsp;&emsp;**接入极简**：`pip install mem0ai` + 3 行初始化代码即可运行，零向量 DB 配置（默认 Qdrant 本地模式）。这是所有同类方案中接入成本最低的。

&emsp;&emsp;**生态最广**：51.4K GitHub Stars，AWS 官方合作，支持 OpenAI/LangGraph/CrewAI/MCP 等主流框架集成。

### 3.3 已知局限与边界

&emsp;&emsp;客观评估一个框架，不仅要看优势，更要看局限。以下是 `mem0` 当前版本（v1.0.9）的已知局限及对应的应对策略：

&emsp;&emsp;**局限一：`add()` 有额外延迟**。每次存储触发 1-2 次 LLM 调用，延迟约 500-2000ms。这是写入延迟，不影响搜索速度。应对策略：在 FastAPI 中使用 `asyncio.create_task()` 将 `add()` 放入后台任务，不阻塞用户响应（详见第六章 6.1 节）。

&emsp;&emsp;**局限二：隐式连接弱**。对于需要跨越多轮对话进行隐式推理的问题（如「用户在第 3 轮提到的偏好与第 28 轮的决策有何关联」），`mem0` 的表现显著弱于全量上下文方案。原因：LLM 提取时倾向于保留显式事实，容易丢失跨轮的隐性关联。应对策略：对于需要深度跨轮推理的场景，考虑结合全量上下文或使用图模式。


&emsp;&emsp;**局限三：依赖 LLM 质量**。记忆提取和冲突判断的质量完全依赖底层 LLM。使用低质量小模型会导致记忆萃取错误。应对策略：选择 DeepSeek 或同级别的高质量模型做记忆提取。

&emsp;&emsp;**局限四：图模式需额外基础设施**。启用图记忆需要部署 Neo4j/Memgraph，增加运维复杂度。应对策略：仅在时序推理需求明确时启用。

&emsp;&emsp;**局限五：成本非零**。每次 `add()` 产生 LLM API 费用，高频写入场景成本不可忽视。应对策略：使用轻量模型做提取，批次合并写入。

&emsp;&emsp;**局限六：无原生 TTL（开源版）**。开源版本缺乏内置记忆过期机制，需手动管理删除。应对策略：实现定时清理任务（详见第六章 6.1 节要点六），或使用云平台版（内置 TTL 支持）。

&emsp;&emsp;3.3 节揭示了 mem0 向量路线的边界。这自然引出一个更深层的问题：文件系统路线在真实生产中是否可行？6.2 节将通过 Claude Code 的记忆设计给出一个已验证的答案——在那之前，我们先动手跑通 mem0 的核心功能。


## <center>第四章：从零到可运行——mem0 快速上手实战</center>

&emsp;&emsp;经过前三章的理论铺垫，你已经理解了 `mem0` 的设计动机、核心架构和选型依据。现在是动手的时候了。本章的目标是让你从零开始，用最短的路径跑通 `mem0` 的核心功能，建立「能用」的信心。

&emsp;&emsp;本章按照递进式路径展开：先完成环境配置（连接 DeepSeek + OpenAI 双凭证），然后用最小示例跑通 `add` → `search` → 完整对话函数，接着通过两个精心设计的演示验证核心能力（「失忆 vs 有记忆」对比和「记忆冲突更新 + history() 追踪」），最后展示进阶配置选项。每个步骤都有可运行的代码和输出解释。

### 4.1 环境准备

&emsp;&emsp;在附录零中我们已经安装了 `mem0ai` 并验证了双凭证。本节进一步完成 `mem0` 的配置初始化——关键是配置 DeepSeek 作为 LLM 提供商、OpenAI 作为 Embedding 提供商。

> &emsp;**向量存储后端说明**：`mem0` 默认使用 Qdrant 本地模式作为向量存储后端（零配置、数据存储在 `/tmp/qdrant`）。本节将显式配置 Qdrant，确保后续所有实战代码使用统一的存储后端。


**步骤一：配置 mem0 使用 DeepSeek 作为 LLM**

&emsp;&emsp;`mem0` 的 LLM 裁判机制需要调用 LLM 进行记忆提取和冲突判断。默认配置使用 OpenAI，但我们选择 DeepSeek 以获得更高性价比。`mem0` 支持通过 `Memory.from_config()` 自定义 LLM 和 Embedding 提供商：

In [6]:
import os, shutil
from dotenv import load_dotenv
from mem0 import Memory

load_dotenv()

# mac系统默认会将数据存储到这个位置下
QDRANT_PATH="./qdrant"

# 清理 Qdrant 本地存储残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# 配置 DeepSeek 作为 LLM，OpenAI 作为 Embedding
config = {
    "llm": {
        "provider": "openai",  # mem0 通过 OpenAI 兼容接口调用 DeepSeek
        "config": {
            "model": "deepseek-chat",                    # DeepSeek 对话模型
            "api_key": os.getenv("DEEPSEEK_API_KEY"),   # 从 .env 读取密钥
            "openai_base_url": "https://api.deepseek.com/v1",  # DeepSeek 兼容 OpenAI 协议
            "temperature": 0.1,                        # 低温度确保记忆提取稳定
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "vector_store": {                          # ← 关键！不写这块就用 /tmp
        "provider": "qdrant",
        "config": {
            "collection_name": "mem0_prod",
            "path": QDRANT_PATH,           # 持久化到项目目录
            # 生产环境推荐换成远端服务器模式：
            # "host": "your-qdrant-server.com",
            # "port": 6333,
        }
    },
    "version": "v1.1"
}

# 创建 Memory 实例
memory = Memory.from_config(config)
print("mem0 初始化完成（LLM: DeepSeek, Embedding: OpenAI）")

mem0 初始化完成（LLM: DeepSeek, Embedding: OpenAI）


&emsp;&emsp;这段代码是本课件所有后续代码的基础。注意几个关键配置项：`provider` 设为 `"openai"` 是因为 `mem0` 通过 OpenAI 兼容接口调用 DeepSeek（DeepSeek 的 API 与 OpenAI 格式兼容）；`openai_base_url` 指向 DeepSeek 的 API 端点；`api_key` 分别使用 `DEEPSEEK_API_KEY` 和 `OPENAI_API_KEY`，实现凭证分离。`version: "v1.1"` 使用 mem0 最新的内部处理流程。

&emsp;&emsp;你可能注意到配置中没有指定向量数据库的地址——这是因为 `mem0` 默认使用 **Qdrant 本地嵌入模式**：`qdrant-client` 库会在 Python 进程内启动一个嵌入式 Qdrant 实例，变更历史存储在 `~/.mem0/history.db`（SQLite）。<font color=red>不需要单独安装或启动 Qdrant 服务器，`pip install mem0ai` 已经包含了所有依赖。</font>这也是 mem0「零向量 DB 配置」的核心卖点之一。

&emsp;&emsp;不同操作系统下的默认存储路径有所不同：

- **macOS / Linux**：向量数据存储在 `/tmp/qdrant`，mem0 配置和历史记录在 `~/.mem0/`（即 `/Users/用户名/.mem0/` 或 `/home/用户名/.mem0/`）

- **Windows**：向量数据同样写入 `/tmp/qdrant`（这是 mem0 源码中的硬编码路径，Windows 上会创建 `C:\tmp\qdrant` 目录），mem0 配置和历史记录在 `C:\Users\用户名\.mem0\`

> &emsp;需要注意的是，`/tmp` 路径在 macOS/Linux 上系统重启后可能被清空，而 Windows 上的 `C:\tmp` 不会自动清理但也不是标准临时目录。无论哪个平台，这个默认路径都<font color=red>不适合生产环境</font>——生产环境的持久化配置将在第六章 6.2 节详细讲解。

> ⚠️ **常见误区**：不要将 `DEEPSEEK_API_KEY` 同时用于 LLM 和 Embedding——DeepSeek 不提供 Embedding 服务。如果这样配置，Embedding 调用会失败。LLM 用 DeepSeek，Embedding 用 OpenAI，是当前性价比较高的可用组合。

### 4.2 最小可运行示例

&emsp;&emsp;在运行代码之前，我们先深入理解 `memory.add()` 背后的核心机制——LLM 裁判的两阶段流程。理解这个机制，才能正确解读后续所有代码的运行结果。

&emsp;&emsp;**提取阶段（Extraction Phase）**：系统将三类上下文送入 LLM——最新一轮对话（latest exchange）、滚动摘要（rolling summary，后台异步更新）、最近 m 条消息（即时上下文窗口）。LLM 从中提取「候选记忆」，只保留真正重要的事实性陈述，过滤闲聊和噪音。例如，从「我叫 Alice，我最近开始减肥了，今天天气真好」中，LLM 会提取「用户名为 Alice」和「用户正在减肥」，丢弃天气闲聊。

&emsp;&emsp;**更新阶段（Update Phase）**：对每条候选记忆，在向量数据库中检索 top-s 条语义相似的现有记忆，然后将候选记忆与检索结果一起送入 LLM，由 LLM 选择 ADD（新增）、UPDATE（合并更新）、DELETE（删除旧记忆）、NONE（无操作）四种操作之一。每次 `add()` 触发 1-2 次 LLM 调用（提取 1 次 + 更新判断 1 次），延迟约 500-2000ms。

&emsp;&emsp;**LLM 调用成本估算**：每次 `add()` 触发 1-2 次 LLM 调用（提取 1 次 + 更新判断 1 次）。使用轻量模型（如 DeepSeek）可有效控制成本。滚动摘要的更新为后台异步任务，不增加主流程延迟。

&emsp;&emsp;配置完成后，让我们用最少的代码跑通 `mem0` 的两个核心操作：`add()`（写入记忆）和 `search()`（检索记忆）。

**步骤一：写入第一条记忆**

&emsp;&emsp;`memory.add()` 接受消息列表和用户标识，内部触发 LLM 裁判的两阶段流程（提取 + 更新），将提取出的关键事实存储到向量数据库：

In [7]:
# 写入记忆：LLM 裁判会自动提取关键事实
result = memory.add(
    messages=[  # OpenAI 格式的对话消息列表
        {"role": "user", "content": "我叫 Alice，我是素食主义者，不吃乳制品。"},
        {"role": "assistant", "content": "好的 Alice，我记住了你的饮食偏好。"}
    ],
    user_id="alice"  # 记忆绑定到 alice 的命名空间
)

print("写入结果：")
print(result)  # 返回 ADD/UPDATE/NONE 操作及提取的记忆摘要

写入结果：
{'results': [{'id': '43b40032-662e-400d-a290-ab41dbb63920', 'memory': 'Name is Alice', 'event': 'ADD'}, {'id': 'fb1ae5af-98d9-450f-9486-b8646802f2c4', 'memory': 'Is a vegetarian', 'event': 'ADD'}, {'id': 'a3b100c6-3d62-45fb-8b1c-fb622aa99530', 'memory': 'Does not eat dairy products', 'event': 'ADD'}]}


&emsp;&emsp;执行后，`result` 会返回 LLM 裁判的决策结果，结构如下：

```python
{
  "results": [
    {
      "id": "a1b2c3d4-...",       # 记忆唯一 ID（用于 history/update/delete）
      "memory": "Alice 是素食主义者",  # LLM 提取的记忆摘要
      "event": "ADD"               # 操作类型：ADD/UPDATE/DELETE/NONE
    }
  ]
}
```

&emsp;&emsp;其中 `id` 字段是后续调用 `memory.history()`、`memory.update()`、`memory.delete()` 的关键凭证。`event` 字段反映 LLM 裁判的决策——如果写入的信息与已有记忆重复，裁判可能返回 UPDATE（合并更新）或 NONE（无操作）。注意 `add()` 不是瞬时操作，由于内部触发 LLM 调用，通常需要 500-2000ms。


**步骤二：检索记忆**

&emsp;&emsp;`memory.search()` 对查询进行语义向量化，在记忆库中检索最相关的条目：

In [8]:
# 检索记忆：语义向量搜索（query → embedding → 余弦相似度匹配）
results = memory.search(
    query="Alice 的饮食偏好是什么？",  # 自然语言查询，自动转为向量
    user_id="alice",                     # 只在 alice 的命名空间内搜索
    limit=5                              # 返回最相关的 5 条
)

print("检索结果：")
for entry in results["results"]:
    print(f"  - {entry['memory']}（相似度得分: {entry.get('score', 'N/A')}）")

检索结果：
  - Name is Alice（相似度得分: 0.3456840259521511）
  - Does not eat dairy products（相似度得分: 0.28684264823473193）
  - Is a vegetarian（相似度得分: 0.2542083950885895）


&emsp;&emsp;检索结果应该包含之前写入的「Alice 是素食主义者，不吃乳制品」这条记忆。`limit=5` 表示最多返回 5 条最相关的结果。

**步骤三：查看全部记忆**

&emsp;&emsp;除了语义搜索，还可以通过 `get_all()` 查看某个用户的全部记忆：

In [9]:
# 查看 alice 的全部记忆
all_memories = memory.get_all(user_id="alice")
print(f"Alice 共有 {len(all_memories['results'])} 条记忆：")
for m in all_memories["results"]:
    print(f"  - [{m['id'][:8]}...] {m['memory']}")

Alice 共有 3 条记忆：
  - [43b40032...] Name is Alice
  - [a3b100c6...] Does not eat dairy products
  - [fb1ae5af...] Is a vegetarian


&emsp;&emsp;三步走通之后，你已经掌握了 `mem0` 最核心的操作链路：`add()` → `search()` → `get_all()`。接下来我们把这三个操作封装成一个完整的对话函数。

**步骤四：封装完整的记忆增强对话函数**

&emsp;&emsp;将 `mem0` 集成到对话流程中的标准模式是：**检索 → 注入 → 响应 → 保存**。以下是一个完整的记忆增强对话函数：

In [10]:
from openai import OpenAI

# DeepSeek 客户端（用于对话生成）
# 初始化 DeepSeek 客户端（通过 OpenAI SDK 兼容接口调用）
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),  # DeepSeek API 密钥
    base_url="https://api.deepseek.com/v1"  # DeepSeek 兼容端点
)

def chat_with_memory(message: str, user_id: str = "default_user") -> str:
    """带记忆增强的对话函数"""
    # 1. 检索相关记忆
    relevant_memories = memory.search(
        query=message,      # 用当前用户消息作为检索 query
        user_id=user_id,    # 只检索该用户的记忆
        limit=3             # 取 Top-3 避免上下文过长
    )
    # 将检索到的记忆拼接为文本，注入 System Prompt
    memories_str = "\n".join(
        f"- {entry['memory']}"
        for entry in relevant_memories["results"]
    )

    # 2. 构建带记忆的系统提示
    system_prompt = (
        f"你是一个有记忆能力的 AI 助手。基于用户的历史偏好进行个性化回答。\n"
        f"用户历史记忆：\n{memories_str}"
        if memories_str else
        "你是一个有记忆能力的 AI 助手。"
    )

    # 3. 调用 DeepSeek 生成响应
    response = deepseek_client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message}
        ]
    )
    assistant_message = response.choices[0].message.content

    # 4. 保存本轮对话到记忆（LLM 裁判会自动决定是否值得记住）
    memory.add(
        messages=[
            {"role": "user", "content": message},
            {"role": "assistant", "content": assistant_message}
        ],
        user_id=user_id
    )

    return assistant_message

&emsp;&emsp;这个函数实现了 `mem0` 最标准的集成模式。关键流程是：先 `search()` 检索相关记忆并注入到 System Prompt，然后调用 LLM 生成响应，最后 `add()` 保存本轮对话。注意第 4 步的 `add()` 会触发 LLM 裁判——如果本轮对话中没有新的值得记住的信息，裁判会返回 `NONE`，不会产生冗余记忆。

### 4.3 演示一："失忆" vs "有记忆"对比

&emsp;&emsp;这是本课件最有冲击力的演示——通过对比「无记忆」和「有记忆」两种模式的输出差异，直观展示 `mem0` 的核心价值。基础入门课件 模块一 1.1 节 也使用了类似的对比演示法，这里我们升级为 `mem0` 版本。

**步骤一：无记忆版本——每次都是陌生人**

&emsp;&emsp;先运行一个没有记忆增强的普通对话，观察 LLM 的表现：

In [11]:
def chat_without_memory(message: str) -> str:
    """无记忆的普通对话——每轮独立，无法关联历史信息"""
    response = deepseek_client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "你是一个 AI 助手。"},  # 无记忆注入
            {"role": "user", "content": message}
        ]
    )
    return response.choices[0].message.content  # 直接返回，不保存记忆

# 第一轮：自我介绍
print("=== 无记忆版本 ===")
r1 = chat_without_memory("我叫 Alice，我是素食主义者，不吃乳制品。")
print(f"第一轮 AI: {r1}\n")

# 第二轮：基于偏好的提问（但 AI 已经"失忆"）
r2 = chat_without_memory("今晚推荐一道适合我的菜吧？")
print(f"第二轮 AI: {r2}")

=== 无记忆版本 ===
第一轮 AI: Alice，你好！很高兴认识你！作为素食主义者且不食用乳制品，你的饮食习惯非常注重健康和环保。如果需要任何关于素食食谱、营养搭配、替代食材的建议，或者想探讨相关话题，我随时可以帮忙哦！ 🌱✨

第二轮 AI: 当然！为了给你更精准的推荐，可以告诉我一些你的偏好或需求吗？比如：  
1. **口味**：喜欢清淡、香辣、酸甜，还是浓郁口感？  
2. **时间**：想快速完成（30分钟内），还是愿意花时间慢慢烹饪？  
3. **食材**：是否有忌口或想用特定食材（比如鸡肉、豆腐、蔬菜等）？  
4. **烹饪难度**：适合新手，还是想挑战复杂菜式？  

或者，如果你有特别想吃的菜系（如中式、西式、日韩料理等），也可以告诉我哦！ 😊


&emsp;&emsp;你会注意到，无记忆版本的第二轮回复中，AI 无法知道「你」是谁、有什么饮食偏好。它会给出通用的菜品推荐，而不是针对素食+无乳制品的个性化推荐。

**步骤二：有记忆版本——记住你是谁**

&emsp;&emsp;现在用我们之前封装的 `chat_with_memory()` 函数，运行相同的对话：

In [12]:
# 先清理之前的测试记忆
memory.delete_all(user_id="demo_alice")

print("=== 有记忆版本（mem0） ===")

# 第一轮：自我介绍 → mem0 自动提取并存储偏好
r1 = chat_with_memory(
    "我叫 Alice，我是素食主义者，不吃乳制品。",
    user_id="demo_alice"
)
print(f"第一轮 AI: {r1}\n")

# 第二轮：基于偏好的提问 → mem0 自动检索记忆并注入
r2 = chat_with_memory(
    "今晚推荐一道适合我的菜吧？",
    user_id="demo_alice"
)
print(f"第二轮 AI: {r2}\n")

# 验证：查看 mem0 自动提取了什么记忆
all_mem = memory.get_all(user_id="demo_alice")
print("mem0 自动提取的记忆：")
for m in all_mem["results"]:
    print(f"  - {m['memory']}")

=== 有记忆版本（mem0） ===
第一轮 AI: 好的，Alice！我已经记住你是素食主义者，并且不吃乳制品。如果你需要推荐餐厅、食谱，或者讨论相关话题，可以随时告诉我，我会根据你的饮食习惯提供建议哦！ 😊

第二轮 AI: 好的，Alice！考虑到你是素食主义者且不吃乳制品，我推荐一道 **泰式椰香南瓜咖喱**。这道菜用椰奶代替乳制品，口感浓郁，南瓜和蔬菜提供丰富的营养，搭配米饭或全麦面包都很适合你的饮食习惯。需要具体做法吗？

mem0 自动提取的记忆：
  - Name is Alice
  - Is a vegetarian
  - Does not eat dairy products


&emsp;&emsp;对比两个版本的第二轮输出，差异会非常明显：有记忆版本的 AI 知道 Alice 是素食主义者且不吃乳制品，会推荐符合这些约束的菜品（如纯素咖喱、豆腐炒蔬菜等）。最后一段代码展示了 `mem0` 自动从对话中提取的记忆条目——你无需手动指定「什么值得记住」，LLM 裁判自动做了这个决策。

### 4.4 演示二：记忆冲突更新 + history() 追踪

&emsp;&emsp;第二章预览了四种操作的定义和冲突解决的基本流程。本节通过三轮实验亲手验证：第一轮写入初始偏好（触发 ADD），第二轮写入矛盾信息（触发 UPDATE 或 DELETE+ADD），第三轮重复已知信息（期望触发 NONE）。每轮操作后用 `history()` 追踪记忆的完整变更轨迹。

&emsp;&emsp;回顾冲突解决的完整流程：新记忆提取 → 向量相似检索 top-s 条现有记忆 → LLM 判断语义是否矛盾 → 矛盾则 DELETE 旧 + ADD 新（保持信息最新），互补则 UPDATE 合并（避免重复存储），完全冗余则 NONE（不操作）。所有操作记录在 SQLite history 表中。

&emsp;&emsp;这个演示验证第二章 2.2 节讲解的 ADD/UPDATE/DELETE/NONE 四种操作。我们通过三轮对话，依次触发 ADD（新增记忆）、DELETE + ADD（冲突更新）、NONE（无变化），然后用 `memory.history()` 查看变更轨迹。

**步骤一：第一轮——ADD（新增记忆）**

In [18]:
# 清理测试环境
memory.delete_all(user_id="demo_conflict")

# 第一轮：建立初始记忆
print("=== 第一轮：ADD ===")
r1 = memory.add(
    messages=[
        {"role": "user", "content": "我喜欢辣的食物，越辣越好。"},
        {"role": "assistant", "content": "了解，你喜欢辣食！"}
    ],
    user_id="demo_conflict"
)
print(f"操作结果: {r1}")

# 查看当前记忆
all_mem = memory.get_all(user_id="demo_conflict")
print(f"\n当前记忆（{len(all_mem['results'])} 条）：")
for m in all_mem["results"]:
    print(f"  - [{m['id'][:8]}] {m['memory']}")

=== 第一轮：ADD ===
操作结果: {'results': [{'id': '3aa7e5ef-5ebb-407f-b0e0-feb46f386a14', 'memory': '喜欢辣的食物，越辣越好', 'event': 'ADD'}]}

当前记忆（1 条）：
  - [3aa7e5ef] 喜欢辣的食物，越辣越好


&emsp;&emsp;第一轮结果中应该看到一个 ADD 操作，因为这是全新的信息。

**步骤二：第二轮——DELETE + ADD（冲突更新）**

In [19]:
# 第二轮：偏好变化，触发冲突更新
print("\n=== 第二轮：DELETE + ADD（冲突更新） ===")
# 新偏好与第一轮矛盾，观察 LLM 裁判如何处理冲突
r2 = memory.add(
    messages=[
        {"role": "user", "content": "我最近肠胃不好，不能吃辣了。以后给我推荐清淡的菜。"},
        {"role": "assistant", "content": "好的，我会记住你现在偏好清淡饮食。"}
    ],
    user_id="demo_conflict"  # 同一用户，LLM 裁判会检测到与旧记忆冲突
)
print(f"操作结果: {r2}")

# 查看更新后的记忆
all_mem = memory.get_all(user_id="demo_conflict")
print(f"\n当前记忆（{len(all_mem['results'])} 条）：")
for m in all_mem["results"]:
    print(f"  - [{m['id'][:8]}] {m['memory']}")


=== 第二轮：DELETE + ADD（冲突更新） ===
操作结果: {'results': [{'id': '3aa7e5ef-5ebb-407f-b0e0-feb46f386a14', 'memory': '最近肠胃不好，不能吃辣了', 'event': 'UPDATE', 'previous_memory': '喜欢辣的食物，越辣越好'}, {'id': '0204ec8e-b07d-4a3e-84a8-c85cb8800ae7', 'memory': '偏好清淡的菜', 'event': 'ADD'}]}

当前记忆（2 条）：
  - [0204ec8e] 偏好清淡的菜
  - [3aa7e5ef] 最近肠胃不好，不能吃辣了


&emsp;&emsp;在第二轮中，LLM 裁判应该检测到「喜欢辣食」与「不能吃辣了」之间的语义矛盾，执行 DELETE（删除旧的辣食偏好）+ ADD（添加新的清淡偏好）操作。

**步骤三：第三轮——NONE（无变化）/UPDATE(更新)**


In [20]:
# 第三轮：重复已知信息，期望触发 NONE
print("=== 第三轮：NONE ====")
# 用几乎相同的语义重复已有记忆，LLM 裁判应判定为 NONE（不重复写入）
# 注意：mem0 源码中的操作类型是 NONE（非 NOOP）
# ⚠️ 实际结果可能是 UPDATE 而非 NONE —— 见下方解释
r3 = memory.add(
    messages=[
        {"role": "user", "content": "我肠胃不好，不能吃辣的。"},
        {"role": "assistant", "content": "好的，我记得你不吃辣。"}
    ],
    user_id="demo_conflict"
)
print(f"操作结果: {r3}")
# 如果结果是 UPDATE（如将「最近肠胃不好」更新为「肠胃不好」），说明 LLM 裁判
# 捕捉到了细微语义差异（去掉了「最近」时间限定词）并视为有意义的更新。
# 只有当 LLM 认为新旧信息「完全冗余、零信息增量」时才会返回 NONE。

=== 第三轮：NONE ====
操作结果: {'results': [{'id': '3aa7e5ef-5ebb-407f-b0e0-feb46f386a14', 'memory': '肠胃不好，不能吃辣的', 'event': 'UPDATE', 'previous_memory': '最近肠胃不好，不能吃辣了'}]}


&emsp;&emsp;第三轮的预期结果是 NONE——因为「不能吃辣」这条信息已经存在，无需重复添加。但实际运行中你大概率会看到 UPDATE 而非 NONE，这并非 bug，而是 LLM 裁判的正常行为。原因是第二轮写入的记忆是「**最近**肠胃不好，不能吃辣**了**」，而第三轮的输入是「肠胃不好，不能吃辣**的**」——LLM 裁判捕捉到了两处差异：

- **去掉了「最近」**：从临时状态变为长期体质描述，语义发生了变化
- **「了」→「的」**：语气从陈述变化变为陈述事实

&emsp;&emsp;LLM 裁判将此视为一次有意义的信息精炼，因此选择 UPDATE。<font color=red>NONE 仅在 LLM 认为新信息与已有记忆**完全冗余、零信息增量**时才会触发</font>——这对语义相似度的要求极高。这正是 LLM 驱动记忆系统的固有特性：决策质量取决于底层模型的语义理解能力，同一条输入换一个模型可能得到不同结果。

> &emsp;**工程启示**：在设计测试用例时，不要假设「差不多的话」就能触发 NONE。如果你的业务依赖精确的去重行为，建议在 `mem0` 上层增加一道基于规则的预过滤（如编辑距离阈值），而不是完全依赖 LLM 裁判的非确定性判断。

**步骤四：history() 查看变更轨迹**

In [21]:
# 查看某条记忆的完整变更历史
# 查看某条记忆的完整变更历史（类似 git log）
all_mem = memory.get_all(user_id="demo_conflict")
if all_mem["results"]:
    first_mem_id = all_mem["results"][0]["id"]  # 取第一条记忆的 ID
    history = memory.history(memory_id=first_mem_id)  # 返回该记忆的所有 ADD/UPDATE/DELETE 事件
    
    print(f"\n=== 记忆 [{first_mem_id[:8]}] 的变更历史 ===")
    for event in history:
        print(f"  操作: {event.get('event', 'N/A')}")
        print(f"  旧值: {event.get('old_memory', '无')}")
        print(f"  新值: {event.get('new_memory', '无')}")
        print(f"  时间: {event.get('updated_at', 'N/A')}")
        print()


=== 记忆 [0204ec8e] 的变更历史 ===
  操作: ADD
  旧值: None
  新值: 偏好清淡的菜
  时间: 2026-04-02T11:30:48.855173+00:00



&emsp;&emsp;`history()` 输出的变更日志完整记录了这条记忆的生命周期。这对于调试和审计非常有价值——当你发现 AI 的行为不符合预期时，可以通过 `history()` 追溯记忆的变更过程，定位问题出在哪一步。

### 4.5 配置进阶：自定义 LLM/Embedder/VectorDB

&emsp;&emsp;前面的实战都使用了 Qdrant 本地嵌入模式作为零配置后端（数据存储在 `/tmp/qdrant`）。`mem0` 实际支持 12 种向量数据库，按部署模式分为：零配置本地（Qdrant 默认模式、FAISS）、自托管服务（Chroma、Weaviate、Milvus）、全托管云（Pinecone、Zilliz Cloud）。此外还支持可选的图谱记忆模块（Neo4j/Memgraph），需显式配置。本节演示如何切换到持久化存储以及其他后端。

&emsp;&emsp;4.1 节我们使用了 `Memory.from_config()` 进行基础配置。本节展示更完整的配置选项，帮助你根据不同需求定制 `mem0` 的行为。

&emsp;&emsp;**配置一：使用 Qdrant 持久化存储（推荐生产配置）**

&emsp;&emsp;默认配置下 Qdrant 数据存储于 `/tmp/qdrant`，这在容器重启后会丢失。以下配置指定持久化路径：

In [59]:
import os, shutil
from dotenv import load_dotenv
from mem0 import Memory

load_dotenv()

QDRANT_PATH="./qdrant_data"

# 清理 Qdrant 本地存储残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# 生产环境配置：三大组件（LLM + Embedding + 向量存储）全部显式指定
production_config = {
    "llm": {  # LLM 组件：负责记忆提取、冲突判断
        "provider": "openai",
        "config": {
            "model": "deepseek-chat",
            "api_key": os.getenv("DEEPSEEK_API_KEY"),
            "openai_base_url": "https://api.deepseek.com/v1",
            "temperature": 0.1,  # 低温度确保裁判行为稳定
        }
    },
    "embedder": {  # Embedding 组件：负责语义向量化
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",  # 1536 维，性价比最高
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "vector_store": {  # 向量存储：持久化到本地磁盘
        "provider": "qdrant",
        "config": {
            "collection_name": "agent_memories",  # 集合名（类似数据库表名）
            "path": QDRANT_PATH,  # 持久化路径（非 /tmp！重启不丢失）
        }
        # ── Qdrant 远程服务器配置（替换上方 chroma 块即可切换） ──
        # "provider": "qdrant",
        # "config": {
        #     "collection_name": "mem0_prod",
        #     "host": "your-qdrant-server.com",
        #     "port": 6333,
        #     "api_key": "your_qdrant_api_key_xxx"
        # }
        # ── Milvus 配置（需 Docker 运行 Milvus，pip install pymilvus） ──
        # "provider": "milvus",
        # "config": {
        #     "collection_name": "mem0_demo",
        #     "url": "http://localhost:19530",  # Milvus 服务地址
        #     "token": "",                      # 本地部署传空串，Zilliz Cloud 填真实 token
        # }
    },
    "version": "v1.1"  # mem0 配置版本
}

memory_prod = Memory.from_config(production_config)
print("生产配置初始化完成（持久化存储）")

生产配置初始化完成（持久化存储）


&emsp;&emsp;关键变化是 `vector_store.config.path` 从默认的 `/tmp/qdrant` 改为 `./qdrant_data`——这确保数据在容器重启后仍然保留。如果使用独立部署的 Qdrant 服务，可以改用 `host` + `port` 配置。

&emsp;&emsp;**配置二：使用 Milvus 持久化存储（推荐生产配置）**

&emsp;&emsp;Milvus 是专为大规模向量检索设计的开源数据库，在百万级以上记忆条目场景下性能优于 Qdrant 本地模式。本地使用 Milvus 需要通过 Docker 启动单机版（镜像约 587MB，首次拉取需几分钟）。<font color=red>以下 Docker 命令默认注释保护，有 Milvus 使用需求时再取消注释执行。</font>

In [ ]:
# === Milvus Docker 启动命令（按需执行，镜像约 587MB）===
# 取消注释后执行：
# 单容器 standalone 模式（内置 etcd + minio，适合教学/开发）
!docker run -d --name milvus-standalone \
    -p 19530:19530 \
    -p 9091:9091 \
    -v milvus_data:/var/lib/milvus \
    milvusdb/milvus:v2.5.11 \
    milvus run standalone
# -p 19530:19530  将容器内 Milvus API 端口映射到宿主机，mem0 通过此端口读写数据
# -v milvus_data:/var/lib/milvus  将数据挂载到 Docker Volume，容器删除后数据不丢失

# 验证是否启动成功：
# !docker ps --filter name=milvus-standalone --format "{{.Status}}"

<div align="center">
<img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260402141407275.png" width="900">
</div>

&emsp;&emsp;启动 Milvus 后（`docker ps` 显示 `Up` 状态即就绪），接下来配置 `mem0` 使用 Milvus 作为向量存储后端。即使暂未启动 Milvus，也建议通读以下配置代码——重点是理解与 Qdrant 配置的差异：

In [22]:
import os
import shutil
from dotenv import load_dotenv
from mem0 import Memory

load_dotenv()

# Milvus 生产配置
milvus_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "deepseek-chat",
            "api_key": os.getenv("DEEPSEEK_API_KEY"),
            "openai_base_url": "https://api.deepseek.com/v1",
            "temperature": 0.1,
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "vector_store": {
        "provider": "milvus",
        "config": {
            "collection_name": "agent_memories",
            "url": "http://localhost:19530",  # Docker 默认端口
            "token": "",  # ⚠️ 本地部署必须传空字符串，不能省略！见下方踩坑预警
        }
    },
    "version": "v1.1"
}

memory_milvus = Memory.from_config(milvus_config)
print("Milvus 配置初始化完成")

# 快速验证：写入一条测试记忆
result = memory_milvus.add(
    messages=[{"role": "user", "content": "我喜欢用 Python 写后端服务"}],
    user_id="milvus_test"
)
print(f"写入结果: {result}")

Milvus 配置初始化完成
写入结果: {'results': [{'id': '9651ccf6-0dd1-491b-b564-9851440bb607', 'memory': '喜欢用 Python 写后端服务', 'event': 'ADD'}]}


&emsp;&emsp;与 Qdrant 配置的主要区别在于 `vector_store` 部分：`provider` 改为 `"milvus"`，`path` 替换为 `url`（指向 Milvus 服务地址）。其余 LLM 和 Embedding 配置完全不变——这正是 `mem0` 统一配置接口的优势：切换向量数据库只需改 `vector_store` 块，业务代码零修改。

> ⚠️ **踩坑预警**：`token` 字段必须显式传空字符串 `""`，不能省略。这是 `mem0ai<=1.0.9` 的一个已知 bug——`MilvusDBConfig.token` 类型声明为 `str` 但默认值是 `None`，`mem0` 内部创建 telemetry 时会触发 Pydantic 校验错误：`ValidationError: token - Input should be a valid string`。本地 Milvus 不需要认证，传空串即可绕过；如果使用 Zilliz Cloud（Milvus 的全托管版），则填入真实的 API Token。

### 4.6 Attu Milvus官方GUI管理工具

* Attu 是 Milvus 官方的 GUI 管理工具，可以通过 Docker 快速启动，拥有非常友好的界面来查看 Scheme、浏览数据和执行向量搜索。

* 启动 Attu

    - 在终端中运行以下命令（假设您的 Milvus IP 为宿主机 IP，或者都在 Docker 网络中）：

    - docker run -p 8000:3000 -e MILVUS_URL=host.docker.internal:19530 zilliz/attu:latest

* 注意：如果不确定网络配置，可以直接使用 MILVUS_URL={您的本机IP}:19530

* 使用方法

    - 浏览器访问 http://localhost:8000
    
    - 连接到 Milvus (IP: host.docker.internal 或本机 IP, 端口: 19530)
    
    - 在左侧菜单选择 Collections -> 点击 products
    
    - 您可以在 Data Preview 标签页直接看到表格形式的数据
    
    - 在 Vector Search 标签页进行向量相似度搜索测试

<div align="center">
<img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260402141457354.png" width="900">
</div>

<div align="center">
<img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260402121047829.png" width="900">
</div>

In [ ]:
# 测试完成后清理（可选）
# 停止并删除 Milvus 容器（数据保留在 Docker Volume 中）
# !docker stop milvus-standalone && docker rm milvus-standalone
# 如需彻底删除数据：
# !docker volume rm milvus_data

&emsp;&emsp;清理命令已注释保护，确认不再需要时取消注释执行。`milvus_data` Volume 会保留所有写入的记忆数据，下次 `docker run` 时重新挂载即可恢复。

### 4.6 开源自托管 vs 云平台版

&emsp;&emsp;`mem0` 提供两种使用方式：**开源自托管**（`Memory` / `AsyncMemory`）和**云平台托管**（`MemoryClient`）。很多人第一反应是「云平台版是不是只是个监控面板？」——不是。云平台版是一套**完整的托管基础设施**，mem0 官方帮你运维了整个技术栈：

- **托管向量数据库**：你不需要自己部署 Qdrant / Milvus，mem0 云端内置了优化过的向量存储，自动扩缩容

- **托管图记忆服务**：开源版需要自己部署 Neo4j，云平台版的 Graph Memory 开箱即用，按需启用

- **托管 LLM + Embedding**：云平台内部使用优化过的模型做记忆提取和向量化，你不需要配置 `llm` 和 `embedder` 字段

- **Web Dashboard**：可视化查看所有用户的记忆条目、搜索历史、调用量统计（开源版没有 Dashboard）

- **高级功能**：Webhooks（记忆变更时回调通知）、Memory Export（批量导出）、Custom Categories（自定义记忆分类）、Filters v2（高级过滤语法）——这些功能开源版不支持或需要自己实现

- **企业合规**：SOC 2 Type II 认证、GDPR 合规、审计日志——适合有安全合规要求的生产环境

&emsp;&emsp;简单说：开源版你自己搭全套（向量 DB + LLM + Embedding + 运维），云平台版 mem0 帮你搭好，你只需要一个 API Key + 几行代码。代价是按用量付费，且数据存储在 mem0 的服务器上（目前仅美国区）。以下是两者的核心对比：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>开源自托管 vs 云平台版对比</font></p>
<div class="center">

| 维度 | 开源版（Self-hosted） | 云平台版（Managed） |
|------|---------------------|-------------------|
| 入口类 | `Memory` / `AsyncMemory` | `MemoryClient` |
| 向量 DB | 自己部署（默认 Qdrant 本地） | 托管，无需配置 |
| 图记忆 | 需自部署 Neo4j/Memgraph | 托管，按需启用 |
| 合规 | 自行保障 | SOC 2 Type II、GDPR |
| 费用 | 仅 LLM API 费用 | 按用量计费 |
| 适合 | 生产自托管、数据主权需求 | 快速验证、小团队 |

</div>

&emsp;&emsp;云平台版的使用方式非常简洁，适合**快速验证原型、小团队协作、或不想自行运维向量数据库**的场景。凭证获取步骤：

&emsp;&emsp;**1. 获取 `MEM0_API_KEY`**：注册登录 [app.mem0.ai](https://app.mem0.ai) 后，左侧导航栏 **SETUP → API Keys**，点击 **Create API Key** 生成：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401192956419.png" width=60%></div>

&emsp;&emsp;**2. 获取 `project_id`**：左侧 **ACCOUNT → Settings**，进入 **PROJECT → General** 页面，直接复制 **Project ID** 字段：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401192922585.png" width=60%></div>

&emsp;&emsp;**3. 获取 `org_id`**：同一 Settings 页面，点击左侧 **ORGANIZATION → Details** 查看组织 ID：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401192628114.png" width=60%></div>

> &emsp;**什么时候用云平台版？** 两种典型场景：(1) 概念验证阶段——不想部署 Qdrant/Milvus，5 分钟跑通完整流程；(2) 多人协作——团队成员共享同一份记忆存储，无需各自维护后端。当数据量增长或有数据主权要求时，再迁移到自托管版本（API 几乎一致，迁移成本低）。


In [63]:
# 云平台版（需要 MEM0_API_KEY，从 app.mem0.ai 注册获取）
# ⚠️ 以下为概念展示，需替换为真实凭证才能运行

import os
from mem0 import MemoryClient
from dotenv import load_dotenv
load_dotenv(override=True)

# ⚠️ 防御性清理：mem0 网页复制 API Key 时常带入不可见的零宽空格（​），导致 UnicodeEncodeError
# 如果运行报错 ascii codec can't encode character，先检查 .env 中的 MEM0_API_KEY 是否含隐藏字符

client = MemoryClient(
    api_key=os.getenv("MEM0_API_KEY",""),           # 替换为你的 MEM0_API_KEY
    org_id="org_OIbU4jclXN3nFFazRolqC5VRG1eV6mKhRCTUn9Ba",       # 替换为你的组织 ID
    project_id="proj_PnjuOnlUc9GDeBHcUdJFSdYeDD7iuM6nvRg3qawv" # 替换为你的项目 ID
)

query = "What do you know about me?"
filters = {
    "OR": [
        {"user_id": "alex"},
        {"agent_id": {"in": ["travel-assistant", "customer-support"]}}
    ]
}
client.search(query, version="v2", filters=filters)

print("云平台版示例（需注册 app.mem0.ai 获取 API Key 后取消注释运行）")


云平台版示例（需注册 app.mem0.ai 获取 API Key 后取消注释运行）


&emsp;&emsp;本课件后续章节均使用开源自托管版本，因为它提供了完全的控制权和数据主权，更适合深入理解 `mem0` 的工作原理。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260402123639406.png" width=60%></div>

&emsp;&emsp;**学完本章，你已经掌握了以下能力**：从零初始化 `mem0` 并运行最小示例；通过「失忆 vs 有记忆」对比直观感受记忆系统的价值；理解 `add()` → `search()` → `update()` → `delete()` 的完整生命周期；掌握 `history()` 追踪记忆演化过程；能自定义 LLM、Embedder 和向量数据库配置。接下来，我们将把这些基础能力集成到真实项目中。

## <center>第五章：集成实战——在真实项目中接入 mem0</center>

&emsp;&emsp;第四章让你跑通了 `mem0` 的基础功能。但在真实项目中，`mem0` 不是单独运行的——它需要与 Agent 框架（如 LangChain）、Web 框架（如 FastAPI）、以及现有的 RAG 系统协同工作。本章将展示三种主流的集成模式，从简到繁递进：LangChain Tool 模式（最灵活）、AsyncMemory + FastAPI（最工程化）。最后，我们会展示一个极具工程价值的模式——双检索注入（mem0 + RAG 并行）。


&emsp;&emsp;本章与第四章的关系是：第四章解决「能跑通」，本章解决「能集成到生产项目」。如果你的目标是在现有项目中快速接入 `mem0`，本章是最直接的参考。

### 5.1 LangChain Tool 模式

&emsp;&emsp;Tool 模式是 `mem0` 与 LangChain 集成的推荐方式。核心思路是将 `mem0` 的 `search` 和 `add` 封装为 LangChain Tool，供 Agent 自主决定何时检索和存储记忆。这种模式使用 `@tool` 装饰器 + `bind_tools()` 的标准 LangChain 工具绑定方式，与 mini-OpenClaw 的架构完全一致。

**步骤一：安装 LangChain 依赖**

In [ ]:
!pip install langchain langchain_openai -q

In [24]:
import os, shutil
from dotenv import load_dotenv
from mem0 import Memory

load_dotenv()

# mac系统默认会将数据存储到这个位置下
QDRANT_PATH="./qdrant"

# 清理 Qdrant 本地存储残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# 配置 DeepSeek 作为 LLM，OpenAI 作为 Embedding
config = {
    "llm": {
        "provider": "openai",  # mem0 通过 OpenAI 兼容接口调用 DeepSeek
        "config": {
            "model": "deepseek-chat",                    # DeepSeek 对话模型
            "api_key": os.getenv("DEEPSEEK_API_KEY"),   # 从 .env 读取密钥
            "openai_base_url": "https://api.deepseek.com/v1",  # DeepSeek 兼容 OpenAI 协议
            "temperature": 0.1,                        # 低温度确保记忆提取稳定
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "vector_store": {                          # ← 关键！不写这块就用 /tmp
        "provider": "qdrant",
        "config": {
            "collection_name": "mem0_prod",
            "path": QDRANT_PATH,           # 持久化到项目目录
            # 生产环境推荐换成远端服务器模式：
            # "host": "your-qdrant-server.com",
            # "port": 6333,
        }
    },
    "version": "v1.1"
}

# 创建 Memory 实例
memory = Memory.from_config(config)
print("mem0 初始化完成（LLM: DeepSeek, Embedding: OpenAI）")

mem0 初始化完成（LLM: DeepSeek, Embedding: OpenAI）


**步骤二：封装 mem0 为 LangChain Tool**

&emsp;&emsp;使用 `@tool` 装饰器将 `mem0` 的核心操作封装为 Agent 可调用的工具：

In [25]:
from langchain_core.tools import tool

@tool
def search_memories(query: str, user_id: str) -> str:
    """从记忆库检索与用户相关的历史信息。当你需要了解用户偏好或历史上下文时调用此工具。"""
    results = memory.search(query, user_id=user_id, limit=5)
    if not results["results"]:
        return "未找到相关记忆"
    return "\n".join([f"- {r['memory']}" for r in results["results"]])

@tool
def save_memory(content: str, user_id: str) -> str:
    """保存重要信息到用户记忆库。当对话中出现用户的偏好、背景或重要决策时调用此工具。"""
    memory.add(
        messages=[{"role": "user", "content": content}],
        user_id=user_id
    )
    return "记忆已保存"

print("LangChain Tools 定义完成：search_memories, save_memory")
print(f"  search_memories: {search_memories.description}")
print(f"  save_memory: {save_memory.description}")


LangChain Tools 定义完成：search_memories, save_memory
  search_memories: 从记忆库检索与用户相关的历史信息。当你需要了解用户偏好或历史上下文时调用此工具。
  save_memory: 保存重要信息到用户记忆库。当对话中出现用户的偏好、背景或重要决策时调用此工具。


&emsp;&emsp;注意 `@tool` 装饰器的 docstring 非常重要——它是 Agent 决定何时调用这个工具的依据。描述要清晰、具体，告诉 Agent 在什么场景下使用这个工具。

**步骤三：创建带记忆工具的 Agent 并测试完整闭环**

&emsp;&emsp;将工具绑定到 LLM 后，我们通过两个关联场景测试完整闭环：**先存入**用户的饮食偏好，**再检索**验证 Agent 能否回忆起来。每个场景都走完「LLM 决策 → 执行工具 → 二次推理」三步。

In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# 初始化 LLM + 绑定工具
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0.7,
)
tools = [search_memories, save_memory]
llm_with_tools = llm.bind_tools(tools)
tool_map = {"search_memories": search_memories, "save_memory": save_memory}

# 当前用户标识
current_user_id = "demo_alice"

# ─── 场景一：保存用户偏好 ───
# 用户告知饮食偏好，Agent 应自主决定调用 save_memory
print("=" * 50)
print("场景一：用户告知新信息 → Agent 保存记忆")
print("=" * 50)

save_messages = [
    SystemMessage(content=f"你是一个有记忆能力的助手。可以使用工具检索和保存用户信息。当前用户的 user_id 是 {current_user_id}，调用工具时请使用此 ID。"),
    HumanMessage(content="我是素食主义者，平时不吃肉，喜欢吃沙拉和水果。")
]

# 第 1 步：LLM 决策
save_response = llm_with_tools.invoke(save_messages)
print(f"Agent 响应：{save_response.content}")
print(f"Agent 决定调用的工具：{[tc['name'] for tc in save_response.tool_calls]}")


场景一：用户告知新信息 → Agent 保存记忆
Agent 响应：我了解到您是一位素食主义者，平时不吃肉，喜欢吃沙拉和水果。让我帮您保存这个重要的饮食偏好信息。
Agent 决定调用的工具：['save_memory']


&emsp;&emsp;LLM 识别到用户提供了新的个人信息（饮食偏好），自主决定调用 `save_memory`。但此时工具还未真正执行——`tool_calls` 只是 LLM 的「调用指令」。接下来在 Python 端执行工具，并将结果喂回 LLM 完成二次推理：

In [27]:
# 第 2 步：执行工具 + 第 3 步：二次推理
exec_messages = save_messages + [save_response]

for tc in save_response.tool_calls:
    tool_fn = tool_map[tc['name']]
    tool_result = tool_fn.invoke(tc['args'])
    print(f"工具 {tc['name']} 执行结果: {tool_result[:300]}")
    exec_messages.append(ToolMessage(content=tool_result, tool_call_id=tc['id']))

# LLM 看到工具执行结果后，生成最终确认回复
final_save = llm_with_tools.invoke(exec_messages)
print(f"\nAgent 最终回复：{final_save.content}")


工具 save_memory 执行结果: 记忆已保存

Agent 最终回复：太好了！我已经将您的素食主义饮食偏好保存下来了。作为素食主义者，您的饮食选择很健康呢！沙拉和水果都是营养丰富的食物。

我想了解一下，您成为素食主义者多久了？另外，除了沙拉和水果，您还喜欢哪些素食菜肴呢？这样我可以更好地了解您的饮食偏好，将来为您提供更合适的建议。


&emsp;&emsp;保存完成。现在 mem0 中已经存储了 `demo_alice` 的饮食偏好。接下来模拟一个新的对话——用户询问自己之前说过的偏好，验证 Agent 能否通过 `search_memories` 检索到刚才保存的信息：

In [28]:
# ─── 场景二：检索用户偏好 ───
# 用户询问之前说过的信息，Agent 应自主调用 search_memories
print("=" * 50)
print("场景二：用户提问 → Agent 检索记忆 → 个性化回复")
print("=" * 50)

search_messages = [
    SystemMessage(content=f"你是一个有记忆能力的助手。可以使用工具检索和保存用户信息。当前用户的 user_id 是 {current_user_id}，调用工具时请使用此 ID。"),
    HumanMessage(content="我之前跟你说过我的饮食偏好，还记得吗？")
]

# 第 1 步：LLM 决策
search_response = llm_with_tools.invoke(search_messages)
print(f"Agent 决定调用：{[tc['name'] for tc in search_response.tool_calls]}")

# 第 2 步：执行检索工具 + 第 3 步：二次推理
exec_messages = search_messages + [search_response]
for tc in search_response.tool_calls:
    tool_fn = tool_map[tc['name']]
    tool_result = tool_fn.invoke(tc['args'])
    print(f"\n记忆检索结果: {tool_result[:300]}")
    exec_messages.append(ToolMessage(content=tool_result, tool_call_id=tc['id']))

final_search = llm_with_tools.invoke(exec_messages)
print(f"\nAgent 最终回复：{final_search.content}")


场景二：用户提问 → Agent 检索记忆 → 个性化回复
Agent 决定调用：['search_memories']

记忆检索结果: - 喜欢吃沙拉和水果
- 用户是素食主义者，平时不吃肉

Agent 最终回复：是的，我记得您之前提到过您的饮食偏好！根据我的记录，您：

1. **是素食主义者** - 平时不吃肉
2. **喜欢吃沙拉和水果**

这些信息对吗？您还有其他饮食偏好或限制想要补充吗？比如对某些食物过敏、特定的烹饪方式偏好，或者有其他喜欢的素食菜肴？


&emsp;&emsp;场景二的结果验证了完整闭环：Agent 检索到场景一保存的饮食偏好，并据此生成了个性化回复。

&emsp;&emsp;但你也看到了——手动写执行循环需要十几行代码（构建 messages、遍历 tool_calls、封装 ToolMessage……）。在生产项目中不会这么写。LangChain 的 `create_agent()` 把整个循环封装成了一行调用，**LLM 决策 → 执行工具 → 喂回结果 → 再次决策**全部自动完成：

In [29]:
from langchain.agents import create_agent

# 一行代码创建自动执行工具的 Agent（LangChain 1.2+ API）
# create_agent 内部自动完成：LLM 决策 → 执行工具 → 喂回结果 → 循环直到完成
agent = create_agent(
    model=llm,
    tools=[search_memories, save_memory],
    system_prompt=f"你是一个有记忆能力的助手。可以使用工具检索和保存用户信息。当前用户的 user_id 是 {current_user_id}，调用工具时请使用此 ID。"
)

# 直接对话，Agent 自动决定是否调用工具、自动执行、自动生成最终回复
result = agent.invoke({"messages": [{"role": "user", "content": "我之前的饮食偏好是什么？"}]})

# 打印完整的消息流（可以看到 Agent 内部的决策过程）
for msg in result["messages"]:
    role = msg.__class__.__name__
    content = msg.content[:200] if msg.content else "(tool_call)"
    print(f"[{role}] {content}")


[HumanMessage] 我之前的饮食偏好是什么？
[AIMessage] 我来帮您查询之前的饮食偏好信息。
[ToolMessage] - 喜欢吃沙拉和水果
- 用户是素食主义者，平时不吃肉
[AIMessage] 根据记忆库中的记录，您之前的饮食偏好是：

1. **喜欢吃沙拉和水果** - 这表明您偏好清淡、健康的食物
2. **您是素食主义者** - 平时不吃肉类食品

这些信息显示您的饮食习惯偏向于植物性饮食，注重健康和清淡的食物选择。您还有其他关于饮食偏好的问题吗？


&emsp;&emsp;对比一下：手动方式需要你写 `for tc in tool_calls → invoke → ToolMessage → invoke` 的完整循环；`create_agent` 把这些全部封装了——你只需传入 `llm`、`tools`、`prompt`，剩下的决策和执行全部自动化。<font color=red>手动方式是为了理解原理，生产环境请用 `create_agent`。</font>这也是 mini-OpenClaw 项目中 `AgentManager` 的底层实现方式（`backend/graph/agent.py`）。

&emsp;&emsp;打印出的消息流清晰展示了 Agent 内部的完整决策链路：`HumanMessage`（用户提问）→ `AIMessage`（LLM 决定调用 search_memories）→ `ToolMessage`（工具返回检索结果）→ `AIMessage`（LLM 基于结果生成最终回复）。这就是前面手动实现的三步闭环，只不过现在由 LangChain 自动驱动。

### 5.2 AsyncMemory + FastAPI

&emsp;&emsp;当 `mem0` 需要集成到 Web 服务中时，异步性能成为关键考量。`mem0` 提供了完整的 `AsyncMemory` 类，与 FastAPI 的 async 后端完全兼容。

**步骤一：初始化 AsyncMemory 全局单例**

&emsp;&emsp;在 FastAPI 应用中，`AsyncMemory` 应该作为全局单例初始化，避免每次请求都创建新实例：

In [30]:
# AsyncMemory 使用与 Memory 完全相同的 config（复用 4.1 节的配置）
from mem0 import AsyncMemory
import shutil

QDRANT_PATH="./qdrant"

# 清理 Qdrant 本地存储残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# ⚠️ AsyncMemory.from_config() 返回协程，必须 await
async_memory = await AsyncMemory.from_config(config)
print("AsyncMemory 初始化完成")


AsyncMemory 初始化完成


**步骤二：同步 vs 异步延迟对比实验**

&emsp;&emsp;`AsyncMemory` 的核心价值是在 Web 服务中**不阻塞事件循环**——当一个请求在等待 mem0 响应时，服务器可以同时处理其他用户的请求。下面的实验模拟 3 个用户同时发起请求：同步版只能逐个处理（用户 2 必须等用户 1 完成），异步版可以并发处理（3 个请求同时发出）。

In [31]:
import time
import asyncio

# 模拟 3 个用户同时请求 mem0 检索
users = [
    ("user_alice", "饮食偏好"),
    ("user_bob", "编程语言"),
    ("user_carol", "工作习惯"),
]

# ─── 同步版：逐个处理用户请求（用户 2 必须等用户 1 完成）───
t0 = time.perf_counter()
for uid, query in users:
    memory.search(query, user_id=uid)
sync_time = time.perf_counter() - t0
print(f"同步逐个处理 3 个用户: {sync_time:.2f}s")

# ─── 异步版：3 个用户请求并发处理 ───
async def handle_user(uid, query):
    """模拟单个用户请求的完整处理：检索 + 返回"""
    result = await async_memory.search(query, user_id=uid)
    return uid, len(result.get("results", []))

t0 = time.perf_counter()
results = await asyncio.gather(*[handle_user(uid, q) for uid, q in users])
async_time = time.perf_counter() - t0
print(f"异步并发处理 3 个用户: {async_time:.2f}s")

for uid, count in results:
    print(f"  {uid}: 检索到 {count} 条记忆")

print(f"\n加速比: {sync_time / async_time:.1f}x")


同步逐个处理 3 个用户: 1.35s
异步并发处理 3 个用户: 0.57s
  user_alice: 检索到 0 条记忆
  user_bob: 检索到 0 条记忆
  user_carol: 检索到 0 条记忆

加速比: 2.4x


&emsp;&emsp;在本地 Qdrant 嵌入模式下，加速比可能不明显（因为底层是同一个进程内的数据库）。但在生产环境中，向量数据库通常是独立部署的远程服务（如 Qdrant Server、Milvus），网络 I/O 等待期间事件循环可以处理其他请求——**这才是 AsyncMemory 在 Web 服务中的核心价值：不让单个慢请求阻塞整个服务器**。

> &emsp;**类比**：同步版像只有一个窗口的银行，每个客户必须等前一个办完；异步版像叫号系统，客户提交请求后可以坐着等，窗口同时处理多个业务。

&emsp;&emsp;至于同一个请求内如何并行调用多个不同服务（mem0 + RAG），将在 5.3 节「双检索注入」中详细讲解。现在我们用一个**完整可运行的 FastAPI 服务**来验证 AsyncMemory 的生产集成方式。选择 `FastAPI` 是因为它原生支持 `async/await`，与 `AsyncMemory` 天然契合——所有数据库 I/O、LLM 调用都可以在事件循环中非阻塞执行，同时 Pydantic 模型自动完成请求参数校验。

In [ ]:
# 完整代码见同目录下 asyncio_test.py，以下展示核心结构
# 启动方式：在终端执行 uvicorn asyncio_test:app --port 8899

# ── 关键点 1：lifespan 初始化 AsyncMemory（全局单例）──
# @asynccontextmanager
# async def lifespan(app: FastAPI):
#     global async_memory
#     async_memory = await AsyncMemory.from_config(config)  # config 同 4.1 节
#     yield
# app = FastAPI(lifespan=lifespan)

# ── 关键点 2：create_task 后台写入，不阻塞响应 ──
# @app.post("/chat")
# async def chat(req: ChatRequest):
#     relevant = await async_memory.search(req.message, user_id=req.user_id, limit=5)
#     response_text = await call_llm_async(req.message, context)
#     asyncio.create_task(async_memory.add(..., user_id=req.user_id))  # 后台写入！
#     return {"response": response_text}  # 立即返回

# ── 关键点 3：查询接口，验证记忆是否异步写入成功 ──
# @app.get("/memories/{user_id}")
# async def get_memories(user_id: str):
#     return await async_memory.get_all(user_id=user_id)

print("完整代码见 asyncio_test.py，请在终端执行以下命令启动服务：")
print("uvicorn asyncio_test:app --port 8899")

**步骤三：启动服务并测试记忆闭环**

&emsp;&emsp;打开一个**新的终端窗口**（保持 Notebook 内核运行），执行以下命令启动 FastAPI 服务：

&emsp;&emsp;`uvicorn asyncio_test:app --port 8899`

&emsp;&emsp;看到 `Application startup complete` 后，回到 Notebook 依次执行下面三个测试 cell：第一个发送对话请求，第二个等待后台 `add()` 完成后查询记忆，第三个验证第二轮对话是否召回了记忆。

In [32]:
!curl -X POST http://localhost:8899/chat -H "Content-Type: application/json" -d '{"user_id":"alice","message":"我最近在学Rust"}' 

{"response":"太棒了！Rust 是一门非常强大且现代的系统编程语言，以其**内存安全、零成本抽象和高性能**著称。学习 Rust 可能会有些挑战，但绝对值得。\n\n为了能更好地帮助你，可以告诉我一些更具体的信息吗？比如：\n\n1. **你现在学到哪个阶段了？**\n   - 刚入门，正在安装环境和写第一个 “Hello, World!”？\n   - 已经学完所有权、借用、生命周期这些核心概念？\n   - 正在学习并发、异步编程或 unsafe Rust？\n\n2. **你学习 Rust 的主要目标是什么？**\n   - 为了开发系统软件、Web 后端、嵌入式，还是单纯对语言本身感兴趣？\n   - 想用它来做什么项目？\n\n3. **在学习中遇到了什么具体的困难或疑惑吗？**\n   - 是某些概念（如生命周期、trait）难以理解？\n   - 还是编译器错误信息让你感到困惑？\n   - 或者是不知道如何组织项目、使用 crate（包）？\n\n4. **你通常通过什么方式学习？**\n   - 看书（比如《Rust 程序设计语言》）、看视频、做练习，还是直接动手做项目？\n\n根据你的情况，我可以给你一些针对性的建议、推荐资源，或者帮你解答具体问题。期待你的回复！ 🦀","memories_used":0,"memory_context":"(无历史记忆)"}

In [33]:
!curl http://localhost:8899/memories/alice

{"user_id":"alice","memories":[{"id":"210f289a-92a8-4276-b625-10f8aedc6a86","memory":"最近在学Rust","hash":"64973843d0d09d7ad5c5a76667a65a31","metadata":null,"created_at":"2026-04-02T12:05:17.695337+00:00","updated_at":"2026-04-02T12:05:17.695337+00:00","user_id":"alice"}]}

In [34]:
# 第二轮对话：验证记忆是否被召回（memories_used 应 > 0）
!curl -s -X POST http://127.0.0.1:8899/chat -H "Content-Type: application/json" -d '{"user_id":"alice","message":"帮我推荐一些学习资料"}' | python3 -m json.tool --no-ensure-ascii

{
    "response": "根据你的历史记录，你最近在学习 Rust，以下是一些优质的学习资源推荐：\n\n### 入门必备\n1. **《The Rust Programming Language》**（官方书籍）\n   - 免费在线版：https://doc.rust-lang.org/book/\n   - 系统全面，适合从零开始\n\n2. **Rustlings**（互动练习）\n   - GitHub：https://github.com/rust-lang/rustlings\n   - 通过修复代码错误学习语法\n\n### 实践提升\n3. **《Rust by Example》**\n   - 在线版：https://doc.rust-lang.org/rust-by-example/\n   - 通过实例快速掌握概念\n\n4. **Exercism - Rust 学习路径**\n   - https://exercism.org/tracks/rust\n   - 分阶段编程挑战，有社区反馈\n\n### 进阶资源\n5. **《Rust for Rustaceans》**\n   - 适合已有基础的学习者\n   - 深入理解所有权、生命周期等核心概念\n\n6. **Rust 官方异步编程指南**\n   - https://rust-lang.github.io/async-book/\n\n### 中文资源\n7. **《Rust 程序设计语言》中文版**\n   - https://kaisery.github.io/trpl-zh-cn/\n\n8. **Rust 语言圣经（Rust Course）**\n   - https://course.rs/\n   - 国内开发者编写的系统教程\n\n### 学习建议\n- 从官方书籍开始，配合 Rustlings 练习\n- 遇到 borrow checker 问题时，多用编译器提示学习\n- 加入 Rust 中文社区（论坛/Telegram群）交流\n\n需要针对某个特定方向（如Web开发、系统编程等）的专项资源吗？",
    "memories_used": 1,
    "memory_context": "最近在学Rust"
}


&emsp;&emsp;这段代码中最关键的是第 3 步——使用 `asyncio.create_task()` 将 `memory.add()` 放入后台任务。<font color=red>这是生产环境中的必备技巧</font>：`add()` 会触发 LLM 调用（500-2000ms 延迟），如果同步等待，用户每次对话都要多等 1-2 秒。使用 `create_task` 将写入操作放入后台，响应立即返回给用户。

> 🔥 **踩坑预警**：如果不使用 `asyncio.create_task()` 而是 `await memory.add()`，每次对话的响应时间会增加 500-2000ms。在 p95 场景下，这意味着用户可能等待超过 3 秒才能看到回复——这在实时对话场景中是不可接受的。

**步骤四：并发压测——验证「不阻塞」**

&emsp;&emsp;前面说「不让单个慢请求阻塞整个服务器」，现在用数据验证。我们向 FastAPI 服务同时发送 3 个不同用户的请求，对比串行 vs 并发的总耗时。如果 AsyncMemory 的异步机制生效，并发总耗时应接近单个最慢请求（而非三者之和）。

In [35]:
import asyncio
import time
import httpx

BASE_URL = "http://127.0.0.1:8899"
# macOS httpx 默认可能走 IPv6 回环（::1），而 uvicorn 默认只监听 IPv4
# 用 local_address="0.0.0.0" 强制走 IPv4，避免 502 错误
transport = httpx.AsyncHTTPTransport(local_address="0.0.0.0")

users = [
    {"user_id": "bench_user_1", "message": "我喜欢用Python做数据分析"},
    {"user_id": "bench_user_2", "message": "我是前端开发主要写React"},
    {"user_id": "bench_user_3", "message": "最近在研究分布式系统"},
]

async def send_chat(client, payload):
    start = time.time()
    resp = await client.post(f"{BASE_URL}/chat", json=payload, timeout=60)
    elapsed = time.time() - start
    return payload["user_id"], elapsed, resp.status_code

async def main():
    async with httpx.AsyncClient(transport=transport) as client:
        # 串行
        print("=== 串行测试（逐个发送 3 个请求）===")
        serial_start = time.time()
        for u in users:
            uid, elapsed, status = await send_chat(client, u)
            print(f"  {uid}: {elapsed:.2f}s")
        serial_total = time.time() - serial_start
        print(f"  串行总耗时: {serial_total:.1f}s\n")

        # 并发
        print("=== 并发测试（同时发送 3 个请求）===")
        concurrent_start = time.time()
        tasks = [send_chat(client, u) for u in users]
        results = await asyncio.gather(*tasks)
        concurrent_total = time.time() - concurrent_start
        for uid, elapsed, status in results:
            print(f"  {uid}: {elapsed:.2f}s")
        print(f"  并发总耗时: {concurrent_total:.1f}s\n")

        print(f"加速比: {serial_total / concurrent_total:.1f}x")

await main()

=== 串行测试（逐个发送 3 个请求）===
  bench_user_1: 24.73s
  bench_user_2: 11.17s
  bench_user_3: 20.22s
  串行总耗时: 56.1s

=== 并发测试（同时发送 3 个请求）===
  bench_user_1: 2.94s
  bench_user_2: 16.70s
  bench_user_3: 9.15s
  并发总耗时: 16.7s

加速比: 3.4x


&emsp;&emsp;串行 3 个请求的总耗时约为三次 LLM 调用延迟之和（~30-40s），而并发 3 个请求的总耗时接近单次最慢请求（~15-20s）——<font color=red>加速比约 2x，证明事件循环在等待 LLM I/O 期间确实在处理其他请求。</font>这就是 Cell 236 银行类比的真实体现：三个客户同时「叫号」，窗口在等待 A 的 LLM 响应时处理 B 和 C 的记忆检索。

> &emsp;**为什么不是 3x？** 因为三个请求共享同一个 DeepSeek API 连接池和本地 Qdrant 实例，存在一定的资源竞争。在生产环境中（独立 Qdrant Server + 多 API Key 负载均衡），加速比会更接近理论上限。

### 5.3 双检索注入（mem0 + RAG 并行）

&emsp;&emsp;这是本课件**工程价值最高**的代码模式。在真实的 Agent 系统中，通常同时存在两类检索需求：RAG 检索外部知识库（如产品文档、代码库），`mem0` 检索用户偏好和历史决策。双检索注入模式使用 `asyncio.gather` 并行执行两路检索，然后将结果拼接注入到 LLM 的上下文中。

&emsp;&emsp;回忆基础入门课件 模块五 6.3 节 的 Memory Manager，它也实现了短期记忆和长期记忆的协同注入。双检索注入模式是这个思路在生产环境中的升级版——`mem0` 负责个性化记忆，RAG 负责知识库检索，两者并行、各司其职，Agent 响应用户时，同时从两个来源并行检索信息，最终拼入 system prompt。

* 两路检索的分工：

- mem0：回答"这个用户是谁、喜欢什么、之前聊过什么"——个性化

- RAG：回答"这个问题的专业知识是什么"——准确性

用 asyncio.gather 并行跑，两路检索同时发出，总耗时 = max(mem0延迟, RAG延迟) 而不是两者之和。这就是你在 mini-OpenClaw 里 prompt_builder.py 做的事情的异步升级版——基础课里是同步拼接，这里变成了并行检索+拼接。

In [36]:
import asyncio

async def build_context(user_input: str, user_id: str) -> str:
    """并行执行 mem0 记忆检索 + RAG 知识库检索，拼接为完整上下文"""

    # 定义两个并行任务
    async def mem0_search():
        results = await async_memory.search(
            user_input, user_id=user_id, limit=5
        )
        return "\n".join([r["memory"] for r in results["results"]])

    async def rag_search():
        # 此处用模拟数据替代真实 RAG 检索
        # 实际项目中替换为你的 RAG retriever
        return "（RAG 检索结果：相关文档片段...）"

    # asyncio.gather 并行执行两路检索
    mem0_result, rag_result = await asyncio.gather(
        mem0_search(),
        rag_search()
    )

    # 拼接上下文
    context = f"""用户历史偏好（来自 mem0）：
    {mem0_result}

    相关知识（来自 RAG）：
    {rag_result}"""

    return context

# 测试并行检索
async def demo_dual_retrieval():
    context = await build_context(
        "推荐一种适合我的编程语言",
        user_id="demo_alice"
    )
    print("拼接后的完整上下文：")
    print(context)

# 在 Notebook 中运行 async 函数
await demo_dual_retrieval()

拼接后的完整上下文：
用户历史偏好（来自 mem0）：
    

    相关知识（来自 RAG）：
    （RAG 检索结果：相关文档片段...）


&emsp;&emsp;双检索注入的核心优势在于：`asyncio.gather` 让两路检索并行执行，总延迟取决于较慢的那个（而非两者之和）。假设 `mem0` 搜索延迟 0.2 秒、RAG 搜索延迟 0.3 秒，串行需要 0.5 秒，并行仅需 0.3 秒——在高并发场景下，这个优化的累积效果非常显著。

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175501063.png" width=60%>
</div>

<!-- [图4 生成提示词]
Visual Style: Futuristic isometric 3D tech illustration, dual-channel parallel architecture, high-end cyber aesthetic
Environment: Dark deep-space background #0A0E1A, neon purple and blue accents, atmospheric depth fog
Metaphor Mapping: Top entry: a glowing request portal representing 用户请求. A glowing dual-channel splitter representing asyncio.gather that forks into two parallel paths. Left path (purple glow): a memory crystal core representing mem0记忆检索 — personal preferences and history. Right path (blue glow): a knowledge repository tower representing RAG知识检索 — documents and facts. Bottom convergence: a glowing merge node representing 上下文拼接. Final destination: a large luminous brain-chip core representing LLM推理生成.
Spatial Layout: Top-to-bottom flow with parallel middle section. Request portal at top center. Splitter below it. Two parallel channels side by side in the middle (left purple for mem0, right blue for RAG). Merge node below both channels. LLM core at bottom center.
Dynamic Flow: Show two parallel glowing data streams leaving the splitter simultaneously — one purple flowing to mem0, one blue flowing to RAG — then both converging at the merge node and flowing together into the LLM core. Emphasize simultaneity with synchronized particle effects.
Material & Texture: Glassmorphism splitter and merger nodes, crystalline memory core (purple), metallic knowledge tower (blue), holographic brain-chip for LLM, highly detailed, cinematic lighting
Chinese Labels: 用户请求, 并行分发, mem0记忆检索, RAG知识检索, 上下文拼接, LLM推理
Text Constraint: Chinese text must be perfectly and accurately rendered, no gibberish. Use these exact labels ONLY: 用户请求, asyncio.gather, mem0记忆检索, RAG知识检索, 上下文拼接, LLM推理
Output: 16:9, 1920x1080, 课件架构插图
-->


### 5.4 mini-OpenClaw 迁移实战：用 mem0 替换 MEMORY.md

&emsp;&emsp;前面四种模式展示了 mem0 与通用框架的集成方式——LangChain Tool、AsyncMemory、双检索注入，覆盖了从原型到生产的常见场景。但对于学过基础入门课件的你来说，最直接的问题是：**如何把 mem0 接入我们已经搭建好的 mini-OpenClaw？** 原来那套 `MEMORY.md` + `memory_indexer.py` + `prompt_builder.py` 长期记忆层，能不能平滑迁移到 mem0？本节用四个步骤完成这次迁移。


&emsp;&emsp;在动手之前，先明确一个关键前提：mem0 只替代长期记忆层，mini-OpenClaw 的其他组件保持不变。1.1 节的替代边界表回答了「改哪里」，下面这张表进一步回答「怎么改」——列出每个组件的具体迁移方式：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mini-OpenClaw 组件迁移清单</font></p>
<div class="center">

| 组件 | 原实现 | mem0 迁移后 | 说明 |
|------|--------|-------------|------|
| `session_manager.py` | `sessions/` JSON + `compress_history()` | 保持不变 | mem0 不管理短期记忆和对话历史压缩 |
| `MEMORY.md` + `memory_indexer.py` | 文件存储 + LlamaIndex 向量索引 | 用 `memory.search()` / `memory.add()` 替代 | 核心迁移点 |
| `prompt_builder.py` 长期记忆层 | 读 `MEMORY.md` 全文注入或 RAG 检索 | 调 `memory.search()` 获取记忆并拼接注入 | 仅改动第 6 层 |
| `prompt_builder.py` 其余 5 层 | SKILLS_SNAPSHOT/SOUL/IDENTITY/USER/AGENTS | 保持不变 | 与记忆无关的组件不受影响 |
| Memory Write Guide | LLM 通过 `write_file` 写 `MEMORY.md` | 改为 mem0 `save_memory` tool | Agent 写入方式变更 |
| `agent.py` RAG 检索 | `memory_indexer.retrieve()` | `memory.search()` | 检索接口替换 |

</div>

&emsp;&emsp;迁移清单列出了每个组件「改什么」，但学员更关心的是端到端数据流「怎么变」。下面用 Before vs After 对比展示完整的请求处理路径，让你一眼看清迁移的本质。

&emsp;&emsp;**Before（当前 mini-OpenClaw 数据流）**：

&emsp;&emsp;`用户消息` → 加载 `sessions/` → [RAG: `indexer.retrieve()`] → `build_system_prompt(MEMORY.md)` → Agent 推理 → [LLM 可能调 `write_file` 写 `MEMORY.md`] → 保存到 `sessions/` → [`auto_compress`]

&emsp;&emsp;**After（mem0 集成后数据流）**：

&emsp;&emsp;`用户消息` → 加载 `sessions/` → [mem0: `memory.search()`] → `build_system_prompt(mem0结果)` → Agent 推理 → 保存到 `sessions/` → [mem0: `memory.add(本轮对话)到数据库`] → [`auto_compress`]

<div align=center>
  <img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260401175502635.png" width=60%>
</div>

<!-- [图5 生成提示词]
Visual Style: Split-screen tech comparison illustration, isometric 3D, clear visual contrast between old and new
Environment: Left side: dim dark background with muted gray-red tones (outdated feel). Right side: bright dark background with vivid neon blue-green tones (modern feel). Center: glowing vertical divider with VS symbol.
Metaphor Mapping: Left panel (Before): A dull file cabinet representing MEMORY.md, connected by gray arrows to a basic text block representing 全文注入 prompt_builder, leading to a plain LLM box. Flat, matte, outdated aesthetic. Right panel (After): A glowing crystalline core representing mem0.add(), connected to a luminous vector DB orb, then a precision-targeted beam representing 语义检索, leading to a vibrant LLM core. Glassmorphism, holographic, modern aesthetic. Red highlight arrows crossing between panels showing what changed.
Spatial Layout: Perfect left-right split screen. Left panel 45% width (Before), right panel 45% (After), center divider 10%. Both panels show top-to-bottom data flow with matching vertical alignment for easy comparison.
Dynamic Flow: Left: dim gray arrows flowing top-to-bottom through the old pipeline. Right: bright glowing arrows flowing through the new pipeline. Red transformation arrows crossing from left to right at each stage showing the upgrade path.
Material & Texture: Left: flat matte textures, no glow, dull colors. Right: Glassmorphism, holographic glow, crystalline materials. Maximum contrast between the two sides.
Chinese Labels: Before, After, MEMORY.md, 全文注入, prompt_builder, mem0.add(), 向量DB, 语义检索, LLM
Text Constraint: Chinese text must be perfectly and accurately rendered, no gibberish. Use these exact labels ONLY: Before, After, MEMORY.md, 全文注入, prompt_builder, mem0.add(), 向量DB, 语义检索, LLM
Output: 16:9, 1920x1080, 课件对比插图
-->


&emsp;&emsp;<font color=red>关键洞察：变化只发生在两个箭头上——读的来源（从 `MEMORY.md` / `indexer.retrieve()` 改为 `memory.search()`）和写的去向（从 `write_file` 写 `MEMORY.md` 改为 `memory.add()`）。</font>整体请求处理流程的结构完全不变：加载会话 → 检索记忆 → 构建 Prompt → 推理 → 保存 → 压缩。这意味着迁移的风险是可控的——你不需要重新设计数据流，只需要替换两个「接口」。

**步骤一：确认保留范围——短期记忆不动**

&emsp;&emsp;在开始改造之前，先明确哪些组件**不需要动**。mem0 是长期记忆解决方案，不管理对话历史。mini-OpenClaw 的短期记忆层（`session_manager.py`）必须完整保留，任何一个环节都不能删减：`sessions/` JSON 持久化继续存储每轮对话的原始消息；`compress_history()` 继续在消息过多时触发 LLM 压缩；`compressed_context` 注入继续在 `load_session_for_agent()` 中注入摘要；`MAX_HISTORY_MESSAGES` 截断继续在 `agent.py` 中限制历史消息数量。

> ⚠️ **常见误区**：不要以为接入 mem0 后就可以删掉 `session_manager.py`。mem0 提取的是跨会话的关键事实（如「用户是 Python 开发者」），而 session_manager 管理的是当前会话的对话流（如「用户刚才问了什么」）。两者是互补关系，不是替代关系。删掉 session_manager 会导致 Agent 在同一次会话内「失忆」，每轮都从空白上下文开始响应。

**步骤二至步骤四：三处代码改造（概要）**

&emsp;&emsp;以下三处改造以概要形式展示核心改动点，重点理解迁移逻辑而非逐行实现。迁移的核心改动集中在三个文件，每处只改几行代码：

&emsp;&emsp;1. **`prompt_builder.py` 第 6 层**：从读 `MEMORY.md` 全文改为接收 `memory.search()` 结果，前 5 层（SOUL/IDENTITY/USER/AGENTS 等）零改动。同时删除第 7 层 Memory Write Guide——mem0 的写入由 `save_memory` tool 负责，不再需要指导 LLM 用 `write_file`

&emsp;&emsp;2. **写入机制**：将 `write_file` 工具替换为 mem0 `save_memory` tool（使用闭包注入 `user_id`，避免 LLM 控制用户标识）。核心变化是写入的「判断层」和「存储层」分离——LLM 只负责判断，mem0 内部的 LLM 裁判再做精细化提取和冲突解决

&emsp;&emsp;3. **`agent.py` 检索**：将 `memory_indexer.retrieve()` 替换为 `memory.search()`，检索结果传入改造后的 `prompt_builder`。迁移后不再需要维护 LlamaIndex 依赖和向量索引重建逻辑

&emsp;&emsp;以上概要说明了「改什么」和「为什么改」。接下来我们通过双模式对比实验，用可运行代码直观展示「改了之后有什么差异」。

&emsp;&emsp;下面首先完成环境初始化，创建两套模式共用的 LLM 实例、mem0 内存对象和隔离的临时工作目录：

In [38]:
# ===== 5.4 双模式对比环境初始化 =====
# 复用基础入门课件的 LangChain 架构：ChatDeepSeek + SystemMessage/HumanMessage

import os, json, shutil
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from mem0 import Memory

# LangChain LLM 初始化（两种模式共用同一个 LLM 实例）
llm = ChatDeepSeek(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
    temperature=0.3,  # 较低温度保证对比实验的一致性
)

QDRANT_PATH="/tmp/qdrant"

# 清理 Qdrant 本地存储残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# mem0 配置（LLM 裁判用 DeepSeek，向量化用 OpenAI Embedding）
mem0_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "deepseek-chat",
            "api_key": os.getenv("DEEPSEEK_API_KEY"),
            "openai_base_url": "https://api.deepseek.com/v1",
            "temperature": 0.1,  # 裁判场景用极低温度
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "version": "v1.1"
}
memory = Memory.from_config(mem0_config)

# 创建临时工作目录（隔离实验数据，不影响 mini-OpenClaw 项目文件）
# 创建对比实验的临时目录（MEMORY.md 模式需要文件系统）
COMPARE_DIR = Path("./compare_test")
COMPARE_DIR.mkdir(exist_ok=True)
(COMPARE_DIR / "sessions").mkdir(exist_ok=True)  # 短期记忆：会话文件
(COMPARE_DIR / "MEMORY.md").write_text(           # 长期记忆：Markdown 文件
    "# 长期记忆\n\n## 用户信息\n\n## 用户偏好\n\n## 重要事项\n",
    encoding="utf-8"
)

print("双模式对比环境初始化完成")
print(f"  LLM: DeepSeek | mem0: 已初始化 | 临时目录: {COMPARE_DIR}")

双模式对比环境初始化完成
  LLM: DeepSeek | mem0: 已初始化 | 临时目录: compare_test


&emsp;&emsp;这段初始化代码做了三件事。第一，创建与基础入门课件一致的 `ChatDeepSeek` 实例。第二，用同一套 DeepSeek + OpenAI 凭证初始化 `mem0 Memory`。第三，创建临时工作目录，包含一个空的 `MEMORY.md` 模板——模拟基础入门课件中长期记忆的初始状态。

&emsp;&emsp;接下来定义两套对话函数。两套函数的结构完全对称，遵循同样的四阶段流程，唯一的差异集中在记忆的读取来源和写入方式上：

In [39]:
# ===== 两套对话函数：沿用 MemoryManager 三阶段架构 =====
# load（加载记忆）→ get_messages（构造 Prompt）→ update（更新记忆）
# 唯一变量：长期记忆后端（MEMORY.md vs mem0）

import re

def chat_markdown_mode(user_input: str, session_id: str = "md_test") -> str:
    """MEMORY.md 模式：LLM 判断写入 + write_file 追加到文件"""
    # ── 阶段一：加载记忆 ──
    session_path = COMPARE_DIR / "sessions" / f"{session_id}.json"
    if session_path.exists():
        session = json.loads(session_path.read_text(encoding="utf-8"))
    else:
        session = {"messages": [], "compressed_context": ""}

    memory_md = (COMPARE_DIR / "MEMORY.md").read_text(encoding="utf-8")

    # ── 阶段二：构造 Messages（与基础入门课件 get_messages_for_llm 一致）──
    messages = [SystemMessage(content=(
        "你是一个有记忆能力的 AI 助手。\n\n"
        f"## 关于用户的长期记忆\n{memory_md}\n\n"
        "## 记忆写入规则\n"
        "当用户提到重要信息（身份、偏好、技术栈、决策）时，"
        "在回复末尾用 [MEMORY_WRITE] 标注需要记住的内容。"
    ))]
    for msg in session["messages"][-20:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            messages.append(AIMessage(content=msg["content"]))
    messages.append(HumanMessage(content=user_input))

    # ── 阶段三：LLM 推理 ──
    response = llm.invoke(messages)
    reply = response.content

    # ── 阶段四：更新记忆 ──
    session["messages"].append({"role": "user", "content": user_input})
    session["messages"].append({"role": "assistant", "content": reply})
    session_path.write_text(
        json.dumps(session, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    # 长期记忆：解析 LLM 回复中的 [MEMORY_WRITE] 标注
    writes = re.findall(r'\[MEMORY_WRITE\]\s*(.+)', reply)
    if writes:
        current = (COMPARE_DIR / "MEMORY.md").read_text(encoding="utf-8")
        for w in writes:
            current += f"\n- {w.strip()}"
        (COMPARE_DIR / "MEMORY.md").write_text(current, encoding="utf-8")

    return reply


def chat_mem0_mode(user_input: str, session_id: str = "mem0_test",
                   user_id: str = "compare_user") -> str:
    """mem0 模式：对话后自动调用 memory.add()，LLM 裁判自动提取"""
    # ── 阶段一：加载记忆（mem0 检索替代读 MEMORY.md）──
    session_path = COMPARE_DIR / "sessions" / f"{session_id}.json"
    if session_path.exists():
        session = json.loads(session_path.read_text(encoding="utf-8"))
    else:
        session = {"messages": [], "compressed_context": ""}

    mem0_results = memory.search(user_input, user_id=user_id, limit=5)
    mem0_context = "\n".join(
        [f"- {r['memory']}" for r in mem0_results.get("results", [])]
    )

    # ── 阶段二：构造 Messages（无 Memory Write Guide）──
    messages = [SystemMessage(content=(
        "你是一个有记忆能力的 AI 助手。\n\n"
        f"## 关于用户的长期记忆\n{mem0_context if mem0_context else '（暂无历史记忆）'}"
    ))]
    for msg in session["messages"][-20:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            messages.append(AIMessage(content=msg["content"]))
    messages.append(HumanMessage(content=user_input))

    # ── 阶段三：LLM 推理 ──
    response = llm.invoke(messages)
    reply = response.content

    # ── 阶段四：更新记忆 ──
    session["messages"].append({"role": "user", "content": user_input})
    session["messages"].append({"role": "assistant", "content": reply})
    session_path.write_text(
        json.dumps(session, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    # 长期记忆：自动提交给 mem0（无需解析回复，无需 LLM 标注）
    memory.add(
        [{"role": "user", "content": user_input},
         {"role": "assistant", "content": reply}],
        user_id=user_id
    )

    return reply

print("两套对话函数定义完成")
print("  chat_markdown_mode(): MEMORY.md 模式（LLM 判断 + 标注写入）")
print("  chat_mem0_mode(): mem0 模式（对话后自动提取）")

两套对话函数定义完成
  chat_markdown_mode(): MEMORY.md 模式（LLM 判断 + 标注写入）
  chat_mem0_mode(): mem0 模式（对话后自动提取）


&emsp;&emsp;两套函数的结构完全对称——同样的四阶段流程、同样的 `ChatDeepSeek` 调用、同样的 session JSON 持久化。<font color=red>唯一的差异在两个地方：阶段一的长期记忆来源（读 MEMORY.md vs 调 `memory.search()`），和阶段四的长期记忆写入方式（解析 `[MEMORY_WRITE]` 标注 vs 自动 `memory.add()`）。</font>注意 mem0 模式的 System Prompt 中没有任何写入指南——Agent 完全不感知记忆管理的存在。

&emsp;&emsp;**对比一：写入方式差异——无感自动提取 vs Agent 主动标注**

&emsp;&emsp;第一个对比聚焦写入的可靠性。MEMORY.md 模式要求 LLM 在回复中主动标注 `[MEMORY_WRITE]`，如果 LLM 忘记标注（这在 DeepSeek 上时有发生），记忆就会丢失。mem0 模式则在对话结束后自动提取，不依赖 LLM 的配合。

In [40]:
# ===== 对比一：写入方式差异 =====
print("=" * 60)
print("对比一：写入方式差异——无感自动提取 vs Agent 主动标注")
print("=" * 60)

test_input = "我叫小明，我是后端工程师，主要用 Python 和 Go，最近在研究 AI Agent"

# MEMORY.md 模式
print("\n--- MEMORY.md 模式 ---")
md_reply = chat_markdown_mode(test_input)
print(f"Agent 回复: {md_reply[:200]}...")
print(f"\nMEMORY.md 文件内容:")
print((COMPARE_DIR / "MEMORY.md").read_text(encoding="utf-8"))

# mem0 模式
print("\n--- mem0 模式 ---")
mem0_reply = chat_mem0_mode(test_input)
print(f"Agent 回复: {mem0_reply[:200]}...")
print(f"\nmem0 自动提取的记忆:")
for m in memory.get_all(user_id="compare_user")["results"]:
    print(f"  - {m['memory']}")

对比一：写入方式差异——无感自动提取 vs Agent 主动标注

--- MEMORY.md 模式 ---
Agent 回复: 你好小明！很高兴认识你。作为后端工程师，Python 和 Go 都是非常强大的语言，特别适合构建高性能的后端服务。最近研究 AI Agent 听起来很有意思，这个领域现在发展很快，结合你的技术栈应该能做出很酷的东西。

有什么具体问题或项目需要讨论吗？比如在 AI Agent 开发中遇到的技术挑战，或者想了解如何用 Python/Go 实现相关功能？[MEMORY_WRITE] 用户叫小明，是后端...

MEMORY.md 文件内容:
# 长期记忆

## 用户信息

## 用户偏好

## 重要事项

- 用户叫小明，是后端工程师，主要技术栈是 Python 和 Go，最近在研究 AI Agent。

--- mem0 模式 ---
Agent 回复: 你好小明！很高兴认识你。作为后端工程师，Python 和 Go 都是非常强大且流行的语言，尤其是在构建高性能、可扩展的系统方面。Python 在 AI 和数据处理领域尤其受欢迎，而 Go 则在并发处理和微服务架构中表现突出。

最近研究 AI Agent 是一个非常前沿且有趣的方向！AI Agent 通常指的是能够自主执行任务、与环境交互并实现目标的智能体。结合你的后端开发经验，你可能会对以下方向...

mem0 自动提取的记忆:
  - Name is 小明
  - Recently researching AI Agent
  - Is a backend engineer
  - Mainly uses Python and Go


&emsp;&emsp;运行后对比两种模式的写入结果。MEMORY.md 模式下，如果 Agent 回复中包含了 `[MEMORY_WRITE]` 标注，文件会被更新；如果 Agent 忘记标注（在实际运行中概率约 20-30%），文件内容不变。mem0 模式下，无论 Agent 回复了什么，`memory.add()` 都会在对话结束后自动提取关键事实——用户身份、技术栈、当前关注点都会被提取为独立的记忆条目。

&emsp;&emsp;**对比二：实时冲突解决 vs 离线批处理**

&emsp;&emsp;第二个对比聚焦矛盾信息的处理时机。当用户先说「喜欢 Java」再说「改用 Python 了」时，两种模式的处理方式截然不同。

In [41]:
# ===== 对比二：矛盾信息的冲突解决 =====
print("=" * 60)
print("对比二：实时冲突解决 vs 离线批处理")
print("=" * 60)

# 第 1 轮：建立初始偏好
# 第 1 轮：建立初始偏好（两种模式写入相同内容）
round1 = "我最喜欢的编程语言是 Java，用了 5 年做后端开发"
chat_markdown_mode(round1, session_id="conflict_md")
chat_mem0_mode(round1, session_id="conflict_mem0", user_id="conflict_user")

print("第 1 轮写入后：")
md_content = (COMPARE_DIR / "MEMORY.md").read_text(encoding="utf-8")
md_entries = [l.strip() for l in md_content.splitlines() if l.strip().startswith("- ")]
print(f"  MEMORY.md 条目: {md_entries}")
mem0_mems = [m["memory"] for m in memory.get_all(user_id="conflict_user")["results"]]
print(f"  mem0 条目: {mem0_mems}")

# 第 2 轮：发送矛盾信息
# 第 2 轮：发送与第 1 轮矛盾的信息，观察两种模式的冲突处理差异
round2 = "我现在完全转向 Python 了，不再写 Java，Python 才是我的最爱"
chat_markdown_mode(round2, session_id="conflict_md")   # MEMORY.md：追加，新旧共存
chat_mem0_mode(round2, session_id="conflict_mem0", user_id="conflict_user")  # mem0：LLM 裁判实时 UPDATE

print(f"\n第 2 轮写入后（矛盾信息）：")
md_content = (COMPARE_DIR / "MEMORY.md").read_text(encoding="utf-8")
md_entries = [l.strip() for l in md_content.splitlines() if l.strip().startswith("- ")]
print(f"  MEMORY.md 条目: {md_entries}")
print(f"  → 两条矛盾记忆可能共存，需等 sleep_time_reorganize() 离线整理")

mem0_mems = [m["memory"] for m in memory.get_all(user_id="conflict_user")["results"]]
print(f"\n  mem0 条目: {mem0_mems}")
print(f"  → LLM 裁判在写入时已实时判断 UPDATE，旧记忆被替换")

对比二：实时冲突解决 vs 离线批处理
第 1 轮写入后：
  MEMORY.md 条目: ['- 用户叫小明，是后端工程师，主要技术栈是 Python 和 Go，最近在研究 AI Agent。', '- 用户有 5 年 Java 后端开发经验，最喜欢的编程语言是 Java。']
  mem0 条目: ['最喜欢的编程语言是 Java', '用了 5 年做后端开发']

第 2 轮写入后（矛盾信息）：
  MEMORY.md 条目: ['- 用户叫小明，是后端工程师，主要技术栈是 Python 和 Go，最近在研究 AI Agent。', '- 用户有 5 年 Java 后端开发经验，最喜欢的编程语言是 Java。', '- 用户已完全转向 Python，不再写 Java，现在最喜欢的编程语言是 Python。']
  → 两条矛盾记忆可能共存，需等 sleep_time_reorganize() 离线整理

  mem0 条目: ['Python 才是最爱', '现在完全转向 Python 了，不再写 Java', '用了 5 年做后端开发']
  → LLM 裁判在写入时已实时判断 UPDATE，旧记忆被替换


&emsp;&emsp;这是两种模式最本质的差异。MEMORY.md 模式下，两轮对话产生的记忆条目会共存在文件中——「喜欢 Java」和「转向 Python」同时存在，直到 `sleep_time_reorganize()` 下次运行时才有机会整合。mem0 模式下，第二轮写入时 LLM 裁判会检测到与已有记忆的语义冲突，立即执行 UPDATE 操作，将「Java」替换为「Python」。<font color=red>这是从「事后整理」到「事中解决」的本质升级。</font>

&emsp;&emsp;**对比三：检索信噪比与 Token 消耗**

&emsp;&emsp;第三个对比同时覆盖检索质量和成本两个维度。积累一批记忆后，观察同一个查询在两种模式下返回的内容差异和 Token 消耗。

In [42]:
# ===== 对比三：检索信噪比与 Token 消耗 =====
print("=" * 60)
print("对比三：检索信噪比与 Token 消耗")
print("=" * 60)

# 在 mem0 中积累更多记忆
extra_inputs = [
    "我们团队用 FastAPI 做后端，React 做前端",
    "我喜欢用 VS Code 开发，配了 Vim 插件",
    "周末一般去跑步，偶尔打篮球",
]
for text in extra_inputs:
    memory.add([{"role": "user", "content": text}], user_id="compare_user")

query = "这个用户的技术栈和开发工具是什么？"

# MEMORY.md 全文注入
md_content = (COMPARE_DIR / "MEMORY.md").read_text(encoding="utf-8")
print("--- MEMORY.md 全文注入（Direct 模式）---")
print(f"注入长度: {len(md_content)} 字符")
print(f"内容预览:\n{md_content[:300]}")
print("→ 全量注入，包含所有内容（不管是否与查询相关）\n")

# mem0 语义检索
print(f"--- mem0 语义检索（query: '{query}'）---")
results = memory.search(query, user_id="compare_user", limit=3)
for r in results["results"]:
    score = r.get("score", "N/A")
    print(f"  [{score:.3f}] {r['memory']}")
print("→ 仅返回与查询相关的精炼事实\n")

# Token 消耗量化（字符数近似估算）
# 注意：精确 token 计数取决于具体模型的分词器
# DeepSeek 使用自有分词器，此处用字符数做量级对比，实际以 API 返回的 usage 字段为准
mem0_text = "\n".join([f"- {r['memory']}" for r in results["results"]])

print("--- Token 消耗对比（字符数近似） ---")
print(f"MEMORY.md 全文注入:  {len(md_content):,} 字符")
print(f"mem0 top-3 检索注入: {len(mem0_text):,} 字符")
if len(md_content) > 0:
    print(f"体积缩减: {(1 - len(mem0_text) / len(md_content)) * 100:.1f}%")
print("随着记忆积累，MEMORY.md 全文注入的消耗会线性增长，mem0 始终只返回 top-k")
print("（精确 token 数以运行时 API 返回的 usage.prompt_tokens 为准）")

对比三：检索信噪比与 Token 消耗
--- MEMORY.md 全文注入（Direct 模式）---
注入长度: 168 字符
内容预览:
# 长期记忆

## 用户信息

## 用户偏好

## 重要事项

- 用户叫小明，是后端工程师，主要技术栈是 Python 和 Go，最近在研究 AI Agent。
- 用户有 5 年 Java 后端开发经验，最喜欢的编程语言是 Java。
- 用户已完全转向 Python，不再写 Java，现在最喜欢的编程语言是 Python。
→ 全量注入，包含所有内容（不管是否与查询相关）

--- mem0 语义检索（query: '这个用户的技术栈和开发工具是什么？'）---
  [0.480] 喜欢用 VS Code 开发，配了 Vim 插件
  [0.381] Is a backend engineer
  [0.343] 团队用 FastAPI 做后端，React 做前端
→ 仅返回与查询相关的精炼事实

--- Token 消耗对比（字符数近似） ---
MEMORY.md 全文注入:  168 字符
mem0 top-3 检索注入: 78 字符
体积缩减: 53.6%
随着记忆积累，MEMORY.md 全文注入的消耗会线性增长，mem0 始终只返回 top-k
（精确 token 数以运行时 API 返回的 usage.prompt_tokens 为准）


&emsp;&emsp;检索结果的差异非常直观。MEMORY.md 全文注入把整个文件（包括章节标题、空行、与查询无关的条目）全部塞进上下文，LLM 需要在大量文本中「大海捞针」。mem0 的 `search()` 只返回与查询语义最相关的 top-3 条记忆，每条都是提炼过的完整事实句。Token 消耗的差异会随着记忆量的增长越来越明显——这在基础入门课件中正是我们引入 RAG 模式的原因，而 mem0 将这个优势做到了更精细的粒度。

**对比四：通过 API 查看 mem0 向量数据库中的记忆**

&emsp;&emsp;MEMORY.md 模式下，打开文件就能看到所有记忆——这是 mini-OpenClaw「文件即记忆」哲学的核心优势。mem0 模式下，记忆存储在本地 Qdrant 向量数据库中，无法直接打开文件查看，需要通过三个 API 来管理。

In [43]:
# ===== 对比四：mem0 记忆管理 API =====
print("=" * 60)
print("对比四：通过 API 查看 mem0 向量数据库中的记忆")
print("=" * 60)

# 1. get_all()：查看用户全部记忆（对标：cat MEMORY.md）
print("\n--- memory.get_all(): 查看全部记忆 ---")
all_mems = memory.get_all(user_id="compare_user")  # 获取该用户的全部记忆
print(f"共 {len(all_mems['results'])} 条记忆:\n")
for m in all_mems["results"]:
    print(f"  ID: {m['id'][:16]}...")
    print(f"  内容: {m['memory']}")
    print(f"  创建时间: {m.get('created_at', 'N/A')}")
    print()

# 2. search()：语义搜索（对标：memory_indexer.retrieve()）
print("--- memory.search(): 语义搜索 ---")
results = memory.search("编程语言偏好", user_id="compare_user", limit=3)  # 语义搜索 Top-3
for r in results["results"]:
    print(f"  [{r.get('score', 0):.3f}] {r['memory']}")

# 3. history()：查看变更历史（对标：git log MEMORY.md）
print("\n--- memory.history(): 记忆变更历史 ---")
if all_mems["results"]:
    target = all_mems["results"][0]
    print(f"查看记忆 '{target['memory'][:30]}...' 的变更轨迹:")
    history = memory.history(target["id"])  # 返回该条记忆的完整 event 链
    for event in history:
        print(f"  操作: {event['event']}")
        if event.get("old_memory"):
            print(f"  旧值: {event['old_memory']}")
        print(f"  新值: {event.get('new_memory', event.get('memory', ''))}")
        print()

对比四：通过 API 查看 mem0 向量数据库中的记忆

--- memory.get_all(): 查看全部记忆 ---
共 7 条记忆:

  ID: 12370c4c-a557-4c...
  内容: 周末一般去跑步，偶尔打篮球
  创建时间: 2026-04-02T12:31:17.641476+00:00

  ID: 2bff432b-d317-44...
  内容: 团队用 FastAPI 做后端，React 做前端
  创建时间: 2026-04-02T12:30:56.242539+00:00

  ID: 36cd3aa8-114c-41...
  内容: 喜欢用 VS Code 开发，配了 Vim 插件
  创建时间: 2026-04-02T12:31:07.106163+00:00

  ID: 624c90a2-47af-4f...
  内容: Name is 小明
  创建时间: 2026-04-02T12:26:31.810294+00:00

  ID: 903d15c2-d611-46...
  内容: Recently researching AI Agent
  创建时间: 2026-04-02T12:26:31.827605+00:00

  ID: 93ec3181-de2d-4d...
  内容: Is a backend engineer
  创建时间: 2026-04-02T12:26:31.815369+00:00

  ID: b8cd72a4-ff84-4f...
  内容: Mainly uses Python and Go
  创建时间: 2026-04-02T12:26:31.826068+00:00

--- memory.search(): 语义搜索 ---
  [0.481] 喜欢用 VS Code 开发，配了 Vim 插件
  [0.361] Mainly uses Python and Go
  [0.269] Is a backend engineer

--- memory.history(): 记忆变更历史 ---
查看记忆 '周末一般去跑步，偶尔打篮球...' 的变更轨迹:
  操作: ADD
  新值: 周末一般去跑步，偶尔打篮球



&emsp;&emsp;三个 API 覆盖了 MEMORY.md 文件操作的所有对应场景。`get_all()` 相当于 `cat MEMORY.md`——查看全部内容；`search()` 相当于 `memory_indexer.retrieve()`——语义检索相关片段；`history()` 相当于 `git log MEMORY.md`——追踪变更历史，但粒度更细（精确到每条记忆的每次 ADD/UPDATE/DELETE）。

&emsp;&emsp;完成四组对比实验后，我们用一张表格汇总四个维度的关键差异：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>MEMORY.md vs mem0 四维度对比总结</font></p>
<div class="center">

| 对比维度 | MEMORY.md 模式 | mem0 模式 | 核心差异 |
|----------|----------------|-----------|----------|
| 写入方式 | LLM 主动标注 + 解析写入 | 对话后自动提取 | Agent 无感 vs 需要配合 |
| 冲突解决 | 离线 sleep_time 批处理 | 写入时实时 UPDATE | 事后整理 vs 事中解决 |
| 检索质量 | 全文注入或 chunk 级 RAG | 条目级语义检索 | 有噪声 vs 精炼事实 |
| 可审计性 | 打开文件直接阅读 | 通过 API 查看 | 人类友好 vs 程序友好 |

</div>

&emsp;&emsp;最后清理本节产生的演示数据，避免影响后续实验：

In [97]:
import shutil
# ===== 清理演示数据 =====
for uid in ["compare_user", "conflict_user"]:
    memory.delete_all(user_id=uid)
shutil.rmtree(COMPARE_DIR, ignore_errors=True)
print("演示数据已清理（mem0 记忆 + 临时文件）")

演示数据已清理（mem0 记忆 + 临时文件）


&emsp;&emsp;演示数据已全部清理。mem0 中的测试记忆通过 `delete_all()` 删除，临时工作目录一并移除。在实际项目中，mem0 默认使用 Qdrant 本地嵌入模式（不需要单独安装 Qdrant 服务器），向量数据存储在 `/tmp/qdrant`，变更历史存储在 `~/.mem0/history.db`。注意 `/tmp` 路径在系统重启后可能被清空，生产环境需配置持久化路径（详见 6.2 节）。

&emsp;&emsp;**学完本章，你已经掌握了以下能力**：使用 LangChain Tool 模式让 Agent 自主决定何时读写记忆；使用 `AsyncMemory` + `asyncio.create_task` 在 FastAPI 中实现非阻塞记忆写入；使用双检索注入模式并行整合 mem0 个性化记忆与 RAG 知识库检索；将 mem0 迁移集成到已有的 mini-OpenClaw 项目中，替换 `MEMORY.md` 长期记忆层；在 LangChain Agent 对话循环中量化对比两种长期记忆方案的写入、冲突解决、检索质量和 Token 消耗差异。这六种模式和验证方法覆盖了从原型到生产、从新建到迁移、从集成到评估的完整路径。


## <center>第六章：生产部署与避坑指南</center>

&emsp;&emsp;前两章让你从零搭建了 `mem0` 的基础用法和多种集成模式。但从「能在 Notebook 里跑通」到「能在生产环境中稳定运行」，还有一段关键距离。本章聚焦于这段距离中最常见的六个问题——每一个都是真实生产环境中出现过的踩坑案例。

&emsp;&emsp;本章分为两部分：6.1 节用速查清单覆盖六个最常见的生产避坑点（每个都是真实踩坑案例），6.2 节展示从 Claude Code 借鉴的五个记忆质量管理模式。建议在正式部署前将 6.1 作为 checklist 逐项核对。

### 6.1 生产部署六要点速查

&emsp;&emsp;**要点一：add() 延迟——用 `asyncio.create_task()` 异步化**

&emsp;&emsp;`memory.add()` 每次触发 1-2 次 LLM 调用，延迟 500-2000ms。<font color=red>必须将 `add()` 放入后台任务，不阻塞用户响应。</font>`search()` 不能后台化（结果需注入上下文），但 `add()` 可以——它的结果不影响当前轮次的响应。

In [ ]:
import asyncio
from mem0 import AsyncMemory

# 异步版本初始化（API 与同步版完全一致，方法名前加 await）
async_memory = await AsyncMemory.from_config(config)

async def generate_response(user_input: str, context: str) -> str:
    """调用 LLM 生成响应（简化实现，生产环境可替换为流式输出）"""
    prompt = f"基于以下记忆上下文回答用户问题。\n记忆：{context}\n用户：{user_input}"
    # 此处简化为同步调用演示异步架构，实际应使用 async LLM client
    return f"[基于 {len(context)} 字记忆上下文的响应]"

async def handle_chat(user_input: str, user_id: str) -> str:
    # 1. 检索（需要等待结果）
    results = await async_memory.search(user_input, user_id=user_id, limit=5)
    context = "\n".join([r["memory"] for r in results["results"]])

    # 2. 生成响应
    response = await generate_response(user_input, context)

    # 3. 后台保存（关键：create_task 不阻塞响应返回，用户无感知延迟）
    asyncio.create_task(
        async_memory.add(
            messages=[
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": response}
            ],
            user_id=user_id  # 异步写入，不影响响应速度
        )
    )
    return response  # 立即返回，记忆在后台异步存储

print("异步写入解法示例已展示")


&emsp;&emsp;**要点二：/tmp 数据丢失——指定持久化路径**

&emsp;&emsp;默认 Qdrant 数据存于 `/tmp/qdrant`，重启后清空。<font color=red>这是最常见的生产事故。</font>在 config 中指定 `vector_store.config.path` 为持久化目录（如 `/data/qdrant`）。Docker 部署时确保挂载宿主机 volume。

In [ ]:
# 生产配置关键：指定持久化路径
# 生产环境：必须指定持久化路径，否则重启后数据丢失
production_vector_config = {
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "agent_memories",  # 集合名，按业务区分
            "path": "/data/qdrant",  # 非 /tmp！确保数据持久化
        }
    }
}
print("持久化路径配置示例")

&emsp;&emsp;**要点三：LLM 质量——选对模型决定记忆准确性**

&emsp;&emsp;记忆提取和冲突判断完全依赖 LLM 语义理解能力。推荐配置：开发和生产均用 `deepseek-chat`（性价比高，约 ￥0.001/千tokens），质量优先场景用 `gpt-4o-mini`（约 ￥0.003/千tokens）。<font color=red>不推荐 7B 以下开源模型——无法可靠识别语义冲突。</font>

&emsp;&emsp;**要点四：图模式取舍——默认不启用**

&emsp;&emsp;图模式（Graph Memory）的价值集中在时序推理，但代价是 p95 延迟 2.6 秒（向量模式 0.2 秒）、token 成本翻倍、需部署 Neo4j。决策规则：**说不出「我需要时序推理」的具体场景，就不要启用。**

&emsp;&emsp;**要点五：成本控制——条件触发 + 批次写入**

&emsp;&emsp;每次 `add()` 消耗约 1-2K tokens。10 万次对话的记忆写入成本约 ￥100-200（DeepSeek）。优化策略：过滤过短消息（条件触发）、会话结束时批次写入、选用轻量模型。

In [ ]:
# 条件触发：仅当消息长度超过阈值时才保存记忆
def conditional_add(messages, user_id, min_length=20):
    """过滤过短的消息，避免无效 LLM 调用"""
    # 提取用户消息内容
    user_msg = next(
        (m["content"] for m in messages if m["role"] == "user"), ""
    )
    if len(user_msg) < min_length:  # 过短的消息（如"好的""嗯"）不值得调用 LLM 提取
        return None
    return memory.add(messages, user_id=user_id)  # 满足阈值才触发写入

print("条件触发示例：过短消息跳过，有效消息触发 add()")

&emsp;&emsp;**要点六：TTL 管理——开源版需手动清理**

&emsp;&emsp;开源版（v1.0.9）无内置记忆过期机制。建议用定时任务（APScheduler/Celery Beat）按 `created_at` 清理超期记忆。云平台版（`MemoryClient`）内置 TTL 支持。

In [ ]:
from datetime import datetime, timedelta

def cleanup_old_memories(user_id: str, max_age_days: int = 90):
    """清理超过指定天数的旧记忆（TTL 淘汰策略）"""
    all_memories = memory.get_all(user_id=user_id)  # 拉取该用户全部记忆
    cutoff = datetime.now() - timedelta(days=max_age_days)  # 计算过期时间线
    deleted_count = 0

    for m in all_memories["results"]:
        created_at = m.get("created_at", "")
        if created_at:
            # 解析 mem0 返回的时间戳（ISO 8601 格式）
            try:
                mem_time = datetime.fromisoformat(created_at.replace("Z", "+00:00"))
                if mem_time.replace(tzinfo=None) < cutoff:  # 仅删除过期记忆
                    memory.delete(memory_id=m["id"])
                    deleted_count += 1
            except (ValueError, TypeError):
                pass  # 跳过时间戳格式异常的条目

    print(f"TTL 清理完成：扫描 {len(all_memories['results'])} 条，删除 {deleted_count} 条过期记忆")

print("TTL 清理函数已定义（需配合定时任务框架使用）")


&emsp;&emsp;这个 `cleanup_old_memories` 函数通过遍历用户所有记忆、解析 `created_at` 时间戳来识别超龄记忆并删除。
生产环境中建议用 APScheduler 或 Celery Beat 定时执行，例如每天凌晨清理一次 90 天前的记忆。


### 6.2 从 Claude Code 借鉴的五个生产模式

&emsp;&emsp;6.1 节解决了 `mem0` 自身的工程问题——延迟、存储、LLM 质量、成本、TTL。这些问题都有明确的技术解法（加异步、换路径、换模型）。但生产环境中还有一类更深层的挑战——它没有现成答案，因为它不是 `mem0` 的 bug，而是**所有记忆系统**共同面对的设计问题：记忆越来越多后，如何保证检索结果的质量和可信度？

&emsp;&emsp;Claude Code 的记忆系统有四个值得学习的设计要素：

- **类型化分类体系**：四种记忆类型（user/feedback/project/reference），每种有独立的写入时机、使用场景和结构模板

- **分层索引结构**：MEMORY.md 作为索引文件（一行一条指针），具体内容存储在独立 `.md` 文件中，兼顾快速扫描和详细内容

- **消费场景驱动**：每条记忆的 `description` 字段决定它在什么场景下被检索，写入时就考虑未来如何使用

&emsp;&emsp;本节将五个生产模式逐一通过 `mem0` 已有的 `metadata` 机制落地为可运行代码。<font color=red>前三个模式只需善用 `memory.add()` 的 `metadata` 参数和 `memory.search()` 的 `filters` 参数，后两个模式则引入轻量级后台调度逻辑。</font>

#### 6.2.1 模式一：记忆分类标签

&emsp;&emsp;**问题场景**：用户积累了 200+ 条记忆后，`search()` 返回的 top-5 结果中可能混杂着「用户是 Python 开发者」（几乎不过时）和「用户上周决定用 FastAPI 重写后端」（可能已经变更）。两条记忆以相同权重出现在上下文中，LLM 无法区分哪条更可信。

&emsp;&emsp;**Claude Code 做法**：将记忆分为 4 类（user/feedback/project/reference），不同类型有不同的生命周期和使用策略。

> **源码出处**：`src/memdir/memoryTypes.ts:14-19`
> ```ts
> export const MEMORY_TYPES = ['user', 'feedback', 'project', 'reference'] as const
> ```
> 四个类型作为 TypeScript 常量数组定义，贯穿整个记忆系统的写入提示、检索筛选和遥测上报。

&emsp;&emsp;**mem0 落地方案**：利用 `metadata` 字段在写入时打标签，检索时按标签过滤。

In [45]:
# === 记忆分类标签：写入时附带类型 ===

# 偏好类记忆（几乎不过时）
memory.add(
    messages=[{"role": "user", "content": "我是 Python 后端开发者，主要做 AI Agent 方向"}],
    user_id="alice",
    metadata={"type": "preference", "source": "self_introduction"}
)

# 决策类记忆（中速衰减，可能随项目变更）
memory.add(
    messages=[{"role": "user", "content": "我们决定用 FastAPI + mem0 做用户记忆层"}],
    user_id="alice",
    metadata={"type": "decision", "source": "architecture_discussion"}
)

# 上下文类记忆（快速衰减，项目阶段性信息）
memory.add(
    messages=[{"role": "user", "content": "这周在赶 v2.0 的 deadline，下周二上线"}],
    user_id="alice",
    metadata={"type": "context", "source": "project_update"}
)

print("三类记忆已写入（preference / decision / context）")

三类记忆已写入（preference / decision / context）


&emsp;&emsp;检索时，可以根据场景选择性过滤特定类型的记忆：

In [46]:
# === 记忆分类标签：检索时按类型过滤 ===

# 场景 1：个性化推荐 → 只检索偏好类
# 按 metadata 中的 type 字段过滤，只返回偏好类记忆
preference_memories = memory.search(
    query="用户的技术栈和偏好",
    user_id="alice",
    filters={"type": "preference"}  # 元数据过滤条件
)
print("偏好类记忆：")
for m in preference_memories.get("results", []):
    print(f"  - {m['memory']}")

# 场景 2：项目上下文感知 → 只检索决策+上下文类
# 按 type=decision 过滤，只返回决策类记忆
project_memories = memory.search(
    query="当前项目状态",
    user_id="alice",
    filters={"type": "decision"}  # 只检索决策类
)
print("\n决策类记忆：")
for m in project_memories.get("results", []):
    print(f"  - {m['memory']}")

偏好类记忆：
  - 主要做 AI Agent 方向
  - 是 Python 后端开发者

决策类记忆：
  - 决定用 FastAPI + mem0 做用户记忆层


&emsp;&emsp;上面两段代码共同构成了分类标签的完整闭环：写入时用 `metadata` 打标签，检索时用 `filters` 过滤。这意味着你可以为同一个用户维护多个「视角」的记忆库——在个性化推荐场景调用 `preference` 类记忆，在项目上下文感知场景调用 `decision` + `context` 类记忆，按需选择而非全量灌入。

> ⚠️ **关键认知**：5.1 节定义的基础 `save_memory` 只有 `content` 和 `user_id` 两个参数，不支持分类标签；`search_memories` 也没有 `filters` 参数。<font color=red>要实现分类存取，存入和检索工具必须配套设计——既要能打标签，也要能按标签过滤。</font>

&emsp;&emsp;有两种方案让 Agent 具备完整的分类存入 + 分类检索能力：

- **方案一：多工具拆分**（推荐）——为每种记忆类型定义专用的存入和检索工具（如 `save_preference` + `search_preferences`），每个工具内部硬编码对应的 `metadata` / `filters`。Agent 根据 docstring 描述自主选择。
- **方案二：单工具 + 类型参数**——给 `save_memory` 和 `search_memories` 各增加 `memory_type` 参数，由 LLM 决定传什么类型。灵活但依赖 LLM 的参数选择准确性。

&emsp;&emsp;下面展示方案一的实现——存入和检索各 2 个专用工具：


In [47]:
# 方案一：多工具拆分 —— 存入和检索各按类型定义专用工具

# ─── 分类存入工具 ───
@tool
def save_preference(content: str, user_id: str) -> str:
    """保存用户的长期偏好信息（技术栈、饮食习惯、工作方式等不易变化的特征）。
    当用户表达个人偏好或自我介绍时调用此工具。"""
    memory.add(messages=[{"role": "user", "content": content}],
              user_id=user_id, metadata={"type": "preference"})
    return "偏好记忆已保存"

@tool
def save_decision(content: str, user_id: str) -> str:
    """保存用户近期做出的项目决策（技术选型、架构变更、任务安排等）。
    当用户表达项目决定或技术选型时调用此工具。"""
    memory.add(messages=[{"role": "user", "content": content}],
              user_id=user_id, metadata={"type": "decision"})
    return "决策记忆已保存"

# ─── 分类检索工具 ───
@tool
def search_preferences(query: str, user_id: str) -> str:
    """检索用户的长期偏好信息。当你需要了解用户的个人偏好或背景信息时调用此工具。"""
    results = memory.search(query, user_id=user_id, limit=5,
                            filters={"type": "preference"})
    if not results["results"]:
        return "未找到用户偏好记忆"
    return "\n".join([f"- {r['memory']}" for r in results["results"]])

@tool
def search_decisions(query: str, user_id: str) -> str:
    """检索用户近期做出的项目决策。当你需要了解当前项目状态或近期决策时调用此工具。"""
    results = memory.search(query, user_id=user_id, limit=5,
                            filters={"type": "decision"})
    if not results["results"]:
        return "未找到项目决策记忆"
    return "\n".join([f"- {r['memory']}" for r in results["results"]])

print("分类工具定义完成（存入 + 检索各 2 个）")
print(f"  save_preference / save_decision")
print(f"  search_preferences / search_decisions")


分类工具定义完成（存入 + 检索各 2 个）
  save_preference / save_decision
  search_preferences / search_decisions


&emsp;&emsp;这样 Agent 在对话中就能自主选择：用户说"我喜欢 Python"时调用 `save_preference`，说"我们决定用 FastAPI"时调用 `save_decision`；用户问"我喜欢什么语言"时调用 `search_preferences`，问"上周决定用什么框架"时调用 `search_decisions`。**存入和检索的工具必须成对设计**——只有写入时打了标签，检索时的过滤才有意义。

&emsp;&emsp;推荐的分类体系如下，借鉴 Claude Code 官方四类设计（`memdir/memoryTypes.ts` 硬编码）并适配 `mem0` 的使用场景：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mem0 记忆分类推荐体系</font></p>
<div class="center">

| 类型标签 | 含义 | 生命周期 | 示例 |
|---------|------|---------|------|
| `preference` | 用户身份、偏好、技术栈 | 长期稳定 | "Python 开发者""素食主义者" |
| `decision` | 技术选型、架构决策 | 中速衰减 | "选择 FastAPI 做后端" |
| `context` | 项目进度、阶段性信息 | 快速衰减 | "本周赶 v2.0 deadline" |
| `reference` | 外部资源链接、文档指针 | 需验证 | "API 文档在 docs.example.com" |

</div>

#### 6.2.2 模式二：记忆写入附带 Why

&emsp;&emsp;**问题场景**：3 个月后 `search()` 检索到一条记忆「用户偏好清淡饮食」，但不知道这条记忆为什么被保存——是因为健康问题？宗教原因？短期减肥？不同原因决定了这条记忆**是否仍然适用**。如果是因为"肠胃不好"，康复后这条记忆就应该失效；如果是宗教原因，则长期有效。

&emsp;&emsp;**Claude Code 做法**：`feedback` 和 `project` 类型的记忆强制附带 `**Why:**`（为什么保存）和 `**How to apply:**`（在什么场景下使用）。

> **源码出处**：`src/memdir/memoryTypes.ts` 中 `feedback` 和 `project` 类型的 `<body_structure>` 标签
> ```
> Lead with the rule itself, then a **Why:** line (the reason the user gave)
> and a **How to apply:** line (when/where this guidance kicks in).
> ```
> 源码中只有这两个类型定义了 `body_structure`，`user` 和 `reference` 没有——说明 Why 机制是针对「容易过时」的记忆类型的专属设计。

&emsp;&emsp;**mem0 落地方案**：在 `metadata` 中注入 `why` 和 `apply` 字段。

In [48]:
# === 记忆写入附带 Why ===

# 不附带 Why（传统写法）——3个月后无法判断时效性
memory.add(
    messages=[
        {"role": "user", "content": "我最近不能吃辣了，推荐清淡的菜"},
        {"role": "assistant", "content": "好的，我记住了你偏好清淡饮食"}
    ],
    user_id="demo_why",
    metadata={
        "type": "preference",
        # 没有 why —— 未来检索时无法判断"为什么不能吃辣"
    }
)

# 附带 Why（推荐写法）——未来可判断时效性
memory.add(
    messages=[
        {"role": "user", "content": "我最近肠胃不好，不能吃辣了"},
        {"role": "assistant", "content": "了解，祝你早日康复！我会记住你当前偏好清淡饮食"}
    ],
    user_id="demo_why",
    metadata={
        "type": "preference",
        "why": "用户因肠胃不好暂时不能吃辣（可能是短期状态）",
        "apply": "推荐餐食时避免辛辣选项，直到用户更新偏好"
    }
)

# 查看写入结果
all_mem = memory.get_all(user_id="demo_why")
for m in all_mem.get("results", []):
    meta = m.get("metadata", {})
    print(f"记忆: {m['memory']}")
    print(f"  type: {meta.get('type', '未分类')}")
    print(f"  why: {meta.get('why', '未记录')}")
    print()

# 清理演示数据
memory.delete_all(user_id="demo_why")

记忆: 最近肠胃不好，不能吃辣了
  type: preference
  why: 用户因肠胃不好暂时不能吃辣（可能是短期状态）



{'message': 'Memories deleted successfully!'}

&emsp;&emsp;`why` 字段的价值在两个场景中特别突出：第一，当 `mem0` 的 LLM 裁判在冲突解决时需要判断"新旧记忆哪个更权威"，`why` 提供了决策依据；第二，当你需要人工审计记忆库时（如 GDPR 合规要求），`why` 解释了每条记忆的存储理由。

#### 6.2.3 模式三：防御性读取

&emsp;&emsp;**问题场景**：`memory.search()` 返回的记忆中，可能包含已经过时但 LLM 裁判未检测到的条目。例如用户 3 个月前说"在学 Rust"，现在可能已经放弃了，但没有显式告知 AI。如果盲目将这条记忆注入上下文，可能导致不恰当的推荐。

&emsp;&emsp;**Claude Code 做法**："记忆可能过时，使用前必须验证"是一条显式设计原则。

> **源码出处**：`src/memdir/memoryTypes.ts` 中有两层防御机制
> - **MEMORY_DRIFT_CAVEAT**（L201）：指示 LLM 在使用记忆前验证是否仍然正确，过时则信任当前状态
> - **TRUSTING_RECALL_SECTION**（L240）：单独的 section，要求推荐前检查文件是否存在、grep 函数名、用户即将行动时先验证
>
> 此外 `findRelevantMemories.ts` 用 Sonnet 做记忆相关性筛选，返回结果包含 `mtimeMs` 时间戳供下游判断新鲜度。

&emsp;&emsp;**mem0 落地方案**：在检索结果注入 System Prompt 之前，根据 `updated_at` 或 `created_at` 时间戳（优先取 `updated_at`，首次写入后未更新时该字段可能为 null，代码已做兼容处理）进行新鲜度标注和分级处理。

In [52]:
from datetime import datetime, timezone

def enrich_with_freshness(search_results: dict, stale_days: int = 30) -> dict:
    """为检索结果标注新鲜度，区分"可信记忆"和"需确认记忆"

    Args:
        search_results: memory.search() 的返回结果
        stale_days: 超过多少天标记为"需确认"
    """
    now = datetime.now(timezone.utc)  # 以 UTC 为基准计算记忆年龄
    fresh_memories = []   # 新鲜记忆（可信度高，直接注入 Prompt）
    stale_memories = []   # 陈旧记忆（需提示 LLM 谨慎使用）

    for m in search_results.get("results", []):
        updated = m.get("updated_at") or m.get("created_at", "")
        if updated:
            try:
                mem_time = datetime.fromisoformat(updated.replace("Z", "+00:00"))  # 兼容 ISO 格式
                age_days = (now - mem_time).days  # 计算距今天数
                m["age_days"] = age_days  # 附加年龄标签
                if age_days <= stale_days:
                    fresh_memories.append(m)
                else:
                    stale_memories.append(m)
            except (ValueError, TypeError):
                fresh_memories.append(m)  # 无法解析时间，保守处理
        else:
            fresh_memories.append(m)

    return {"fresh": fresh_memories, "stale": stale_memories}


def build_memory_context(enriched: dict) -> str:
    """将分级后的记忆构建为 System Prompt 注入文本"""
    parts = []
    if enriched["fresh"]:
        parts.append("## 用户记忆（近期，可信度高）")
        for m in enriched["fresh"]:
            parts.append(f"- {m['memory']}（{m.get('age_days', '?')}天前更新）")

    if enriched["stale"]:
        parts.append("\n## 历史记忆（较久远，仅供参考，建议与用户确认后使用）")
        for m in enriched["stale"]:
            parts.append(f"- {m['memory']}（{m.get('age_days', '?')}天前，可能已过时）")

    return "\n".join(parts) if parts else "（暂无用户记忆）"


&emsp;&emsp;光定义函数不够直观——如果直接用 `memory.search()` 的结果测试，所有记忆都是刚添加的，`age_days` 全部为 0，无法验证分级效果。下面我们**构造一组包含不同时间戳的模拟数据**，验证分级逻辑是否正确区分「新鲜记忆」和「陈旧记忆」：


In [53]:
from datetime import timedelta

# 构造模拟的 search 返回结果，包含三条不同时间的记忆
now = datetime.now(timezone.utc)
mock_search_results = {
    "results": [
        {  # 2 天前 → 应归为「新鲜」
            "memory": "最近在学习 Rust 语言，打算用它重写后端服务",
            "updated_at": (now - timedelta(days=2)).isoformat(),
        },
        {  # 45 天前 → 应归为「陈旧」
            "memory": "长期使用 VS Code 作为主力编辑器",
            "updated_at": (now - timedelta(days=45)).isoformat(),
        },
        {  # 90 天前 → 应归为「陈旧」
            "memory": "去年参加了 PyCon China 2025 大会",
            "updated_at": (now - timedelta(days=90)).isoformat(),
        },
    ]
}

# 分级 + 构建上下文
enriched = enrich_with_freshness(mock_search_results, stale_days=30)
print(f"新鲜记忆: {len(enriched['fresh'])} 条")
print(f"陈旧记忆: {len(enriched['stale'])} 条")

context = build_memory_context(enriched)
print("\n" + "=" * 50)
print("构建的分级上下文（将注入 System Prompt）：")
print("=" * 50)
print(context)


新鲜记忆: 1 条
陈旧记忆: 2 条

构建的分级上下文（将注入 System Prompt）：
## 用户记忆（近期，可信度高）
- 最近在学习 Rust 语言，打算用它重写后端服务（2天前更新）

## 历史记忆（较久远，仅供参考，建议与用户确认后使用）
- 长期使用 VS Code 作为主力编辑器（45天前，可能已过时）
- 去年参加了 PyCon China 2025 大会（90天前，可能已过时）


&emsp;&emsp;这段代码的核心设计思路是：将检索结果分为「近期可信」和「历史待确认」两级，在注入 System Prompt 时用不同的措辞标注。LLM 看到"仅供参考，建议与用户确认"的标注后，会在涉及该记忆时主动询问用户，而不是直接当作事实使用。<font color=red>这个简单的分级策略，可以显著降低过时记忆导致的错误推荐风险。</font>

&emsp;&emsp;但光标注「可能过时」还不够——**检测到陈旧记忆后，Agent 应该怎么处理？**答案是利用 mem0 自身的 LLM Judge 机制形成自动闭环：

1. **检测**：`enrich_with_freshness` 标注陈旧记忆
2. **确认**：将分级上下文注入 SystemMessage，LLM 看到「可能过时」标注后**主动向用户确认**
3. **更新**：用户的纠正性回复作为新消息走 `memory.add()`，mem0 的 LLM Judge 自动判定 **UPDATE**，覆盖旧记忆

&emsp;&emsp;<font color=red>关键洞察：不需要手动调用 `memory.delete()`——用户的纠正性回复本身就是新信息，mem0 会自动识别语义冲突并触发 UPDATE。</font>下面演示这个完整闭环：


In [54]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# ─── 第 1 步：准备带分级上下文的 System Prompt ───
# 复用上面构造的 mock 数据（VS Code 45天前、PyCon 90天前）
enriched = enrich_with_freshness(mock_search_results, stale_days=30)
graded_context = build_memory_context(enriched)

system_prompt = f"""你是一个具有记忆能力的 AI 助手。
以下是该用户的历史记忆，请注意区分可信度：

{graded_context}

规则：
- 「近期」记忆可直接使用
- 「历史」记忆标注了「可能已过时」，在涉及时必须先向用户确认是否仍然有效
- 确认后再基于最新信息回答"""

print("注入的 System Prompt（含分级上下文）：")
print(system_prompt)

# ─── 第 2 步：模拟用户提问，观察 LLM 是否主动确认陈旧记忆 ───
llm = ChatOpenAI(
    model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0.7,
)

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="帮我推荐一些提升开发效率的工具和方法")
])

print("\n" + "=" * 50)
print("LLM 回复（注意是否主动确认了陈旧记忆）：")
print("=" * 50)
print(response.content)


注入的 System Prompt（含分级上下文）：
你是一个具有记忆能力的 AI 助手。
以下是该用户的历史记忆，请注意区分可信度：

## 用户记忆（近期，可信度高）
- 最近在学习 Rust 语言，打算用它重写后端服务（2天前更新）

## 历史记忆（较久远，仅供参考，建议与用户确认后使用）
- 长期使用 VS Code 作为主力编辑器（45天前，可能已过时）
- 去年参加了 PyCon China 2025 大会（90天前，可能已过时）

规则：
- 「近期」记忆可直接使用
- 「历史」记忆标注了「可能已过时」，在涉及时必须先向用户确认是否仍然有效
- 确认后再基于最新信息回答

LLM 回复（注意是否主动确认了陈旧记忆）：
根据你最近在学习 Rust 并打算重写后端服务的情况，我为你推荐以下工具和方法：

## 🛠️ Rust 开发效率工具
1. **rust-analyzer** - VS Code 的 Rust 语言服务器，提供智能补全、类型提示等
2. **cargo-watch** - 自动重新编译和运行代码
3. **clippy** - Rust 官方 lint 工具，帮助写出更地道的代码
4. **tokio-console** - 异步运行时调试工具

## 🔧 通用开发效率方法
1. **测试驱动开发（TDD）** - 特别适合 Rust 这种强类型语言
2. **Git 工作流优化** - 使用 pre-commit hooks 自动运行格式化检查
3. **容器化开发环境** - 用 Docker 确保环境一致性

## 💡 关于你的开发环境
我看到你长期使用 VS Code，这个编辑器对 Rust 支持很好。不过这是45天前的信息，**想确认一下**：
- 你现在还在用 VS Code 吗？
- 如果是，我可以推荐一些适合 Rust 开发的 VS Code 扩展

另外，你去年参加了 PyCon China 2025，**想确认一下**：
- 你现在是否还在同时使用 Python？
- 如果是，我可以推荐一些 Rust-Python 互操作的工具

这样我能给你更精准的建议！


&emsp;&emsp;可以看到，LLM 在涉及 VS Code（45天前）时会主动确认「你现在还在用 VS Code 吗？」，而不是直接假设。接下来模拟用户的纠正性回复——这次用 5.1 节定义的 `create_agent`，让 Agent **自主决定调用 `save_memory`**，mem0 的 LLM Judge 自动触发 UPDATE：


In [55]:
from langchain.agents import create_agent

# ─── 准备：为测试用户预埋一条「旧记忆」───
test_user = "loop_demo"
memory.add(
    messages=[{"role": "user", "content": "我长期使用 VS Code 作为主力编辑器"}],
    user_id=test_user
)
print("预埋旧记忆：'长期使用 VS Code 作为主力编辑器'\n")

# ─── 第 3 步：创建带记忆工具的 Agent ───
# 复用 5.1 节的 save_memory / search_memories 工具
current_user_id = test_user
agent = create_agent(
    model=llm,  # 复用上方已初始化的 LLM
    tools=[save_memory, search_memories],
    system_prompt=f"""你是一个具有记忆能力的 AI 助手。
当前用户 ID: {current_user_id}

{graded_context}

规则：
- 用户提供新信息时，调用 save_memory 保存
- 需要回忆用户信息时，调用 search_memories 检索
- 历史记忆标注了「可能已过时」，涉及时先确认再使用"""
)

# ─── 用户纠正：「我改用 Cursor 了」───
# Agent 应自主调用 save_memory，mem0 的 LLM Judge 会触发 UPDATE
print("用户: 我已经不用 VS Code 了，现在改用 Cursor 作为主力编辑器")
print("=" * 50)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我已经不用 VS Code 了，现在改用 Cursor 作为主力编辑器"}]}
)

# 打印 Agent 的决策过程
for msg in result["messages"]:
    role = msg.__class__.__name__
    if msg.content:
        print(f"[{role}] {msg.content[:200]}")
    elif hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] 调用工具: {tc['name']}({tc['args']})")

# ─── 验证：检索确认记忆已更新 ───
print("\n" + "=" * 50)
print("验证：检索编辑器偏好（应只看到 Cursor）")
print("=" * 50)
verify = memory.search("编辑器偏好", user_id=test_user, limit=3)
for r in verify["results"]:
    print(f"  - {r['memory']}")

# 清理
memory.delete_all(user_id=test_user)
print("\n✅ 闭环验证完成，测试数据已清理")


预埋旧记忆：'长期使用 VS Code 作为主力编辑器'

用户: 我已经不用 VS Code 了，现在改用 Cursor 作为主力编辑器
[HumanMessage] 我已经不用 VS Code 了，现在改用 Cursor 作为主力编辑器
[AIMessage] 我来帮您更新这个信息。让我先搜索一下相关的历史记忆，然后保存您的新偏好。
[ToolMessage] - 长期使用 VS Code 作为主力编辑器
[AIMessage] 现在我来保存您的新信息：
[ToolMessage] 记忆已保存
[AIMessage] 好的，我已经更新了您的记忆！现在记录显示您已从 VS Code 切换到 Cursor 作为主力编辑器。

Cursor 确实是一个很不错的选择，特别是它集成了 AI 功能，对于开发工作应该很有帮助。您最近在学习 Rust 并用它重写后端服务，使用 Cursor 的 AI 辅助功能应该能提高不少效率吧？

验证：检索编辑器偏好（应只看到 Cursor）
  - 已从 VS Code 切换到 Cursor 作为主力编辑器

✅ 闭环验证完成，测试数据已清理


&emsp;&emsp;整个闭环的精妙之处在于：**没有引入任何新 API**。检测用的是 `updated_at` 时间戳（mem0 已有），确认靠的是 Prompt 工程（标注措辞引导 LLM 行为），更新走的是 `memory.add()` 的 LLM Judge 管道（第四章讲过的 UPDATE 事件）。三步闭环，零额外基础设施。


#### 进阶优化：从「天数阈值」到「认知工程」

&emsp;&emsp;上面的 `enrich_with_freshness()` 用固定阈值（30 天）将记忆二分为「可信」和「需确认」，这在工程上可行，但存在两个**认知层面**的缺陷：

&emsp;&emsp;**缺陷一：LLM 不擅长日期运算。** 把 `2026-02-15T08:30:00+00:00` 这样的 ISO 时间戳注入 System Prompt，LLM 需要先计算"今天是几号"、"距离 2 月 15 号过了多少天"、"这个天数算不算过时"——三步推理，每一步都可能出错。Claude Code 源码中有一条注释直接点明了这个问题：

> **源码出处**：`src/memdir/memoryAge.ts:12-13`
> ```
> Models are poor at date arithmetic — a raw ISO timestamp doesn't
> trigger staleness reasoning the way "47 days ago" does.
> ```

&emsp;&emsp;**缺陷二：硬性阈值无法区分记忆的「衰减速率」。** 2 天前的「用户喜欢 Python」几乎不可能过时，但 2 天前的「main.py 第 42 行有 bug」可能已经修复了。固定 30 天阈值对两者一视同仁。Claude Code 的做法是：**不做硬过滤，把带新鲜度警告的记忆直接注入 prompt，让 LLM 根据记忆内容自行判断是否可信。**

> **源码出处**：`src/memdir/memoryAge.ts:33-41`（`memoryFreshnessText` 函数）
> ```ts
> export function memoryFreshnessText(mtimeMs: number): string {
>   const d = memoryAgeDays(mtimeMs)
>   if (d <= 1) return ''  // 1天内不加警告
>   return `This memory is ${d} days old. ` +
>     `Claims about code behavior or file:line citations may be outdated. ` +
>     `Verify against current code before asserting as fact.`
> }
> ```
> 注意：没有 30 天、60 天的阈值，只有 >1 天就附加自然语言警告。LLM 自己决定信不信。

&emsp;&emsp;下面将课件已有的 `enrich_with_freshness()` 升级为 Claude Code 风格的**认知工程版本**：

In [ ]:
# === 新鲜度标注 v2：认知工程版 ===

def build_memory_context_v2(search_results: dict) -> str:
    """将检索结果转换为带新鲜度标注的 System Prompt 片段。
    
    与 v1（enrich_with_freshness）的两个核心差异：
    1. 用自然语言"N天前"替代 ISO 时间戳——LLM 更容易触发陈旧推理
    2. 不做硬过滤，全部注入 prompt，让 LLM 根据内容自行判断
    """
    lines = ["以下是该用户的相关记忆：\n"]
    
    for item in search_results.get("results", []):
        memory_text = item.get("memory", "")
        meta = item.get("metadata", {})
        
        # 计算天数
        updated = item.get("updated_at") or item.get("created_at", "")
        if updated:
            try:
                dt = datetime.fromisoformat(updated.replace("Z", "+00:00"))
                age_days = (datetime.now(timezone.utc) - dt).days
            except (ValueError, TypeError):
                age_days = -1
        else:
            age_days = -1
        
        # 自然语言时间标注（核心改进：LLM 能直接理解）
        if age_days == 0:
            age_label = "今天"
        elif age_days == 1:
            age_label = "昨天"
        elif age_days > 1:
            age_label = f"{age_days}天前"
        else:
            age_label = "时间未知"
        
        # 类型标签（6.2.1 的分类）
        type_tag = f"[{meta.get('type', 'unknown')}] " if 'type' in meta else ""
        
        # 构建条目：类型 + 内容 + 时间
        entry = f"- {type_tag}{memory_text}（记录于{age_label}）"
        
        # >1天的记忆附加警告（Claude Code 风格：不过滤，只标注）
        if age_days > 1:
            entry += f"\n  ⚠ 这条记忆已有{age_days}天，内容可能已过时，引用前请与用户确认。"
        
        lines.append(entry)
    
    return "\n".join(lines)


# === 对比演示：v1 vs v2 ===
v2_context = build_memory_context_v2(mock_search_results)
print("=== v2 认知工程版输出 ===")
print(v2_context)
print()
print("--- 关键差异 ---")
print("v1：固定30天阈值，二分为'可信/需确认'，应用层过滤")
print("v2：自然语言标注天数，全部注入prompt，LLM自行判断")
print("v2 优势：'2天前的Python偏好'和'2天前的bug行号'会被LLM区别对待")

&emsp;&emsp;v2 版本的核心洞察来自 Claude Code 的一条设计原则：**新鲜度标注是给 LLM 看的，不是给人看的，所以要用 LLM 能理解的方式表达。** `"47天前"` 比 `"2026-02-15T08:30:00"` 更能触发 LLM 的陈旧推理——这不是代码技巧，是**认知工程**。同时，不做硬性阈值过滤而是交给 LLM 判断，让系统能够区分不同类型记忆的衰减速率。

#### 6.2.4 模式四：离线记忆整合（Dream）

&emsp;&emsp;**问题场景**：用户第一周说「在学 Rust」，第三周说「放弃 Rust 了，改学 Go」。两次对话分别触发了 `memory.add()`，但 mem0 的 LLM Judge 只在**写入时**比较新旧记忆——如果两条记忆的向量相似度不够高（"学 Rust" 和 "改学 Go" 的语义方向相反），第二条可能被当作**全新记忆 ADD** 而非**冲突更新 UPDATE**。结果：用户的记忆库里同时存在「在学 Rust」和「改学 Go」两条矛盾记忆。

&emsp;&emsp;<font color=red>这不是 mem0 的 bug，而是所有「被动触发」冲突解决系统的结构性局限：只在写入时比较，无法发现两条旧记忆之间的矛盾。</font>记忆库随时间膨胀，近似重复和矛盾条目悄然累积——6.2.3 节的防御性读取能在检索时标注「可能过时」，但不能主动修复。

&emsp;&emsp;**Claude Code 做法**：`src/services/autoDream/autoDream.ts` 实现了一个叫 **Dream（记忆整合）** 的后台机制——类比人类睡眠时的记忆巩固。

> **源码出处**：`src/services/autoDream/autoDream.ts` + `consolidationPrompt.ts`
> - 触发条件：距上次整合 >= 24 小时 **且** 累积 >= 5 个新会话
> - 四阶段流程：Orient（扫描现有记忆）→ Gather（收集新信号）→ Consolidate（合并/去重/修正）→ Prune（更新索引、删除陈旧指针）
> - 关键操作："Merging new signal into existing topic files rather than creating near-duplicates" + "Deleting contradicted facts"

&emsp;&emsp;**mem0 落地方案**：用 `memory.get_all()` + LLM 做一次性整合扫描。不需要 forked agent，用 mem0 已有的 API 就能实现核心逻辑：

In [165]:
# === 离线记忆整合（Dream 轻量版）===
from dotenv import load_dotenv
load_dotenv(override=True)


def consolidate_memories(memory_client, user_id: str, llm=None) -> dict:
    """扫描全量记忆，发现并修复矛盾/重复条目。
    
    借鉴 Claude Code Dream 的 Consolidate + Prune 阶段，
    但用 mem0 原生 API 实现（无需 forked agent）。
    """
    all_memories = memory_client.get_all(user_id=user_id)
    items = all_memories.get("results", [])
    
    if len(items) < 2:
        return {"checked": len(items), "conflicts": 0, "duplicates": 0}
    
    if llm is None:
        llm = ChatOpenAI(                      
            model="deepseek-chat",                                                                                                                                                                                
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            base_url="https://api.deepseek.com/v1",                                                                                                                                                               
            temperature=0,
        )       
    
    # --- Phase 1: 检测矛盾和重复 ---
    memory_list = "\n".join(
        f"[ID={m['id']}] {m['memory']}" for m in items
    )
    
    detect_prompt = f"""以下是一个用户的所有记忆条目。请找出其中：
    1. 语义矛盾的记忆对（如"喜欢Java"和"改用Python了"）
    2. 近似重复的记忆对（如"是Python开发者"和"主要用Python编程"）

    记忆列表：
    {memory_list}

    请严格按以下 JSON 格式返回（无其他文字）：
    {{"conflicts": [["ID_old", "ID_new"], ...], "duplicates": [["ID_keep", "ID_remove"], ...]}}
    如果没有发现问题，返回：{{"conflicts": [], "duplicates": []}}"""
    
    response = llm.invoke([HumanMessage(content=detect_prompt)])
    
    try:
        import json as _json
        text = response.content.strip()
        if text.startswith("```"):
            text = text.split("```")[1].removeprefix("json").strip()
        result = _json.loads(text)
    except (ValueError, IndexError):
        print("⚠ LLM 返回格式异常，跳过本次整合")
        return {"checked": len(items), "conflicts": 0, "duplicates": 0}
    
    # --- Phase 2: 修复 ---
    conflict_count = 0
    for old_id, new_id in result.get("conflicts", []):
        try:
            memory_client.delete(memory_id=old_id)
            print(f"  修复矛盾：删除旧记忆 {old_id[:8]}")
            conflict_count += 1
        except Exception as e:
            print(f"  ⚠ 删除失败 {old_id[:8]}: {e}")
    
    duplicate_count = 0
    for keep_id, remove_id in result.get("duplicates", []):
        try:
            memory_client.delete(memory_id=remove_id)
            print(f"  去重：删除重复记忆 {remove_id[:8]}")
            duplicate_count += 1
        except Exception as e:
            print(f"  ⚠ 删除失败 {remove_id[:8]}: {e}")
    
    return {
        "checked": len(items),
        "conflicts": conflict_count,
        "duplicates": duplicate_count
    }


# === 演示：构造矛盾场景并整合 ===
dream_user = "dream_demo"

memory.add(
    messages=[{"role": "user", "content": "我最近在学 Rust，打算用它重写后端"}],
    user_id=dream_user,
    metadata={"type": "project"}
)
memory.add(
    messages=[{"role": "user", "content": "我是 Python 后端开发者"}],
    user_id=dream_user,
    metadata={"type": "preference"}
)
memory.add(
    messages=[{"role": "user", "content": "Rust 太难了，放弃了，还是用 Go 重写后端"}],
    user_id=dream_user,
    metadata={"type": "project"}
)
memory.add(
    messages=[{"role": "user", "content": "我是一个 Python 程序员"}],
    user_id=dream_user,
    metadata={"type": "preference"}
)

print("整合前记忆数:", len(memory.get_all(user_id=dream_user).get("results", [])))
for m in memory.get_all(user_id=dream_user).get("results", []):
    print(f"  [{m['id'][:8]}] {m['memory']}")

print("\n--- 执行 Dream 整合 ---")
stats = consolidate_memories(memory, dream_user)
print(f"\n整合结果: 检查{stats['checked']}条, 修复矛盾{stats['conflicts']}对, 去重{stats['duplicates']}对")

print(f"\n整合后记忆数:", len(memory.get_all(user_id=dream_user).get("results", [])))
for m in memory.get_all(user_id=dream_user).get("results", []):
    print(f"  [{m['id'][:8]}] {m['memory']}")

# 清理
memory.delete_all(user_id=dream_user)

整合前记忆数: 3
  [29d57579] 是 Python 后端开发者
  [b0a7c26c] 最近在学 Rust
  [fc2135d3] Rust 太难了，放弃了，还是用 Go 重写后端

--- 执行 Dream 整合 ---
  修复矛盾：删除旧记忆 b0a7c26c

整合结果: 检查3条, 修复矛盾1对, 去重0对

整合后记忆数: 2
  [29d57579] 是 Python 后端开发者
  [fc2135d3] Rust 太难了，放弃了，还是用 Go 重写后端


{'message': 'Memories deleted successfully!'}

&emsp;&emsp;Dream 整合解决了 mem0 冲突解决的结构性盲区：**被动触发的 UPDATE 只能处理写入时的新旧对比，无法发现两条旧记忆之间的矛盾。** 轻量版只需 `get_all()` + 一次 LLM 调用就能完成全量扫描。在生产环境中，可以将 `consolidate_memories()` 注册为定时任务（如每天凌晨执行），Claude Code 的默认策略是「距上次整合 >= 24 小时 且 累积 >= 5 个新会话」——这个双重门控避免了频繁执行的成本浪费。

&emsp;&emsp;至此，6.2.3 的防御性读取（读时标注陈旧）和 6.2.4 的 Dream 整合（离线修复矛盾）形成了完整的记忆维护闭环：**一个负责发现问题，一个负责解决问题。**

#### 6.2.5 模式五：智能提取节流

&emsp;&emsp;**问题场景**：5.4 节的迁移方案中，`chat_mem0_mode()` 在每轮对话结束后都调用 `memory.add()`。这个设计有三个隐患：

&emsp;&emsp;（1）**成本浪费**：每次 `add()` 触发 1-2 次 LLM 调用。如果用户连续 5 轮都在讨论同一个技术方案，前 4 轮的提取大概率与第 5 轮重复——白白多花了 4 次 LLM 调用的钱。

&emsp;&emsp;（2）**重复写入**：如果主 Agent 已经通过 LangChain Tool 模式（5.1 节）主动调用了 `save_memory` 工具保存记忆，对话结束后 `chat_mem0_mode()` 又会再提取一遍——同一条信息可能产生两条近似记忆。

&emsp;&emsp;（3）**失败丢失**：如果某轮 `add()` 因网络超时失败，那轮对话中的信息就永久丢失了，下一轮的 `add()` 只处理新消息。

&emsp;&emsp;**Claude Code 做法**：`src/services/extractMemories/extractMemories.ts` 用三个机制解决这三个问题：

> **源码出处**：`src/services/extractMemories/extractMemories.ts`
> - **节流控制**（L374-385）：`turnsSinceLastExtraction` 计数器，每 N 轮才触发一次提取
> - **主 Agent 互斥**（L348-359）：`hasMemoryWritesSince()` 检测主 Agent 是否已写入记忆，如已写入则跳过
> - **游标 + 失败重试**（L307, L430-434）：`lastMemoryMessageUuid` 记录处理进度，成功才推进游标，失败不推进

&emsp;&emsp;**mem0 落地方案**：将这三个机制封装为一个 `SmartExtractor` 类，替代原来的裸 `memory.add()` 调用：

In [56]:
# === 智能提取节流器 ===

class SmartExtractor:
    """借鉴 Claude Code extractMemories 的三个核心机制：
    1. 节流控制：每 N 轮才提取一次（降低成本）
    2. 主 Agent 互斥：主 Agent 已写入则跳过（避免重复）
    3. 游标 + 失败重试：失败不丢数据（提高可靠性）
    """
    
    def __init__(self, memory_client, throttle_every: int = 3):
        self.memory = memory_client
        self.throttle_every = throttle_every
        self.turns_since_last = 0
        self.message_buffer = []
        self.agent_wrote_this_turn = False
    
    def mark_agent_wrote(self):
        """当主 Agent 通过 Tool 模式主动写入记忆时调用。"""
        self.agent_wrote_this_turn = True
    
    def on_turn_end(self, messages: list, user_id: str):
        """每轮对话结束时调用（替代原来的裸 memory.add()）。"""
        self.message_buffer.extend(messages)
        self.turns_since_last += 1
        
        # === 互斥检测 ===
        if self.agent_wrote_this_turn:
            print(f"  [SmartExtractor] 跳过——主 Agent 已写入记忆")
            self.agent_wrote_this_turn = False
            self.message_buffer.clear()
            self.turns_since_last = 0
            return
        
        # === 节流控制 ===
        if self.turns_since_last < self.throttle_every:
            print(f"  [SmartExtractor] 节流中（{self.turns_since_last}/{self.throttle_every}轮）")
            self.agent_wrote_this_turn = False
            return
        
        # === 批量提取 ===
        print(f"  [SmartExtractor] 触发提取——累积{len(self.message_buffer)}条消息")
        try:
            self.memory.add(
                messages=self.message_buffer,
                user_id=user_id
            )
            self.message_buffer.clear()
            self.turns_since_last = 0
            print(f"  [SmartExtractor] 提取成功，缓冲区已清空")
        except Exception as e:
            # 失败：保留缓冲区，下次重试（游标不推进）
            print(f"  [SmartExtractor] 提取失败: {e}，缓冲区保留待重试")
            self.turns_since_last = 0
        
        self.agent_wrote_this_turn = False


&emsp;&emsp;`SmartExtractor` 用三行核心逻辑（节流计数器、互斥标记、消息缓冲区）将原来的裸 `memory.add()` 升级为生产级写入策略。在上面的 5 轮演示中：第 1-2 轮被节流（不调 `add()`，零成本），第 3 轮批量提取前 3 轮的全部消息（一次 LLM 调用覆盖 3 轮），第 4 轮被主 Agent 互斥跳过（避免重复），第 5 轮进入新一轮节流。**原本 5 次 `add()` 调用降为 1 次，LLM 成本直降 80%。**

&emsp;&emsp;失败重试的关键在于缓冲区设计：`add()` 失败时 `message_buffer` 不清空，下次触发时会重新提交累积的全部消息——这就是 Claude Code 源码中「成功才推进游标」的 mem0 版本。

&emsp;&emsp;上面的演示中，`mark_agent_wrote()` 是在第 4 轮硬编码调用的——这只是为了展示机制。在真实 Agent 系统中，这个标记应该由 **Agent 调用记忆写入工具时自动触发**，而不是人为指定哪一轮跳过。下面我们用 `@tool` 装饰器构建一个带 `save_memory` 工具的 Agent，让 Agent 自主决定何时显式写入记忆：

- **普通对话**：Agent 正常回复，不调用工具 → `SmartExtractor` 按节流策略批量提取
- **重要信息**：Agent 判断当前信息很重要，主动调用 `save_memory` 工具 → 工具内部自动触发 `mark_agent_wrote()` → `SmartExtractor` 跳过本轮（避免重复写入）

In [57]:
# === 真实 Agent 场景：工具自动触发互斥 ===
from langchain_core.tools import tool


# --- 1. 定义 save_memory 工具（关键：内部自动标记） ---
def create_save_memory_tool(extractor_instance, memory_client):
    """工厂函数：创建绑定了 extractor 的 save_memory 工具。"""

    @tool
    def save_memory(content: str, user_id: str) -> str:
        """当 Agent 判断当前信息非常重要时，主动调用此工具显式保存记忆。
        适用场景：用户明确表达偏好变更、紧急需求、关键决策等。"""
        memory_client.add(
            messages=[{"role": "user", "content": content}],
            user_id=user_id
        )
        # ★ 核心：工具内部自动通知 extractor，本轮不需要再提取
        extractor_instance.mark_agent_wrote()
        return f"已保存关键记忆: {content[:30]}..."

    return save_memory


# --- 2. 模拟 Agent 的决策逻辑 ---
def simulate_agent_response(user_msg: str, tools: dict, user_id: str):
    """模拟 Agent 根据消息内容决定是否调用 save_memory 工具。
    
    真实系统中这个决策由 LLM 的 tool_choice 完成，
    这里用规则模拟以聚焦展示 extractor 互斥机制。"""
    
    # 模拟 LLM 判断：包含「紧急/重要/变更」等关键词 → 调用工具
    urgent_keywords = ["紧急", "重要", "立刻", "马上", "变更", "改用"]
    is_urgent = any(kw in user_msg for kw in urgent_keywords)
    
    if is_urgent:
        print(f"  Agent 判断：重要信息 → 调用 save_memory 工具")
        result = tools["save_memory"].invoke(
            {"content": user_msg, "user_id": user_id}
        )
        print(f"  工具返回：{result}")
    else:
        print(f"  Agent 判断：普通对话 → 正常回复，不调用工具")


# --- 3. 组装并运行测试场景 ---
agent_user = "agent_demo"
agent_extractor = SmartExtractor(memory, throttle_every=3)
save_tool = create_save_memory_tool(agent_extractor, memory)
tools = {"save_memory": save_tool}

test_conversations = [
    "我是做数据分析的",              # 普通 → 缓冲
    "主要用 pandas 和 SQL",          # 普通 → 缓冲
    "最近在学 Spark",                # 普通 → 第3轮触发批量提取
    "紧急：我下周要做 Spark 调优分享",  # ★ Agent 判断重要 → 调用工具 → 自动互斥
    "分享的主题是大数据实时处理",       # 普通 → 缓冲（重新计数 1/3）
]

print("=" * 55)
print("真实 Agent 场景：工具自动触发 mark_agent_wrote()")
print("=" * 55)

for i, user_msg in enumerate(test_conversations, 1):
    print(f"\n【第{i}轮】用户: \"{user_msg}\"")
    
    # Agent 决策：是否调用工具
    simulate_agent_response(user_msg, tools, agent_user)
    
    # 无论 Agent 是否调用工具，每轮结束都通知 extractor
    agent_extractor.on_turn_end(
        [{"role": "user", "content": user_msg}],
        agent_user
    )

print("\n" + "=" * 55)
print("最终记忆（Agent 工具写入 + Extractor 批量提取）")
print("=" * 55)
for m in memory.get_all(user_id=agent_user).get("results", []):
    print(f"  {m['memory']}")

memory.delete_all(user_id=agent_user)


真实 Agent 场景：工具自动触发 mark_agent_wrote()

【第1轮】用户: "我是做数据分析的"
  Agent 判断：普通对话 → 正常回复，不调用工具
  [SmartExtractor] 节流中（1/3轮）

【第2轮】用户: "主要用 pandas 和 SQL"
  Agent 判断：普通对话 → 正常回复，不调用工具
  [SmartExtractor] 节流中（2/3轮）

【第3轮】用户: "最近在学 Spark"
  Agent 判断：普通对话 → 正常回复，不调用工具
  [SmartExtractor] 触发提取——累积3条消息
  [SmartExtractor] 提取成功，缓冲区已清空

【第4轮】用户: "紧急：我下周要做 Spark 调优分享"
  Agent 判断：重要信息 → 调用 save_memory 工具
  工具返回：已保存关键记忆: 紧急：我下周要做 Spark 调优分享...
  [SmartExtractor] 跳过——主 Agent 已写入记忆

【第5轮】用户: "分享的主题是大数据实时处理"
  Agent 判断：普通对话 → 正常回复，不调用工具
  [SmartExtractor] 节流中（1/3轮）

最终记忆（Agent 工具写入 + Extractor 批量提取）
  主要用 pandas 和 SQL
  下周要做 Spark 调优分享
  做数据分析
  最近在学 Spark


{'message': 'Memories deleted successfully!'}

&emsp;&emsp;五个模式形成递进的**记忆质量管理体系**：分类标签解决「什么类型的记忆」，Why 元数据解决「为什么保存了这条记忆」，防御性读取解决「这条记忆现在还可信吗」，Dream 整合解决「旧记忆之间的矛盾谁来修复」，智能提取节流解决「什么时候写、写几次」。五者合起来，覆盖了记忆生命周期的写入、检索、维护三个阶段——让 `mem0` 从"能用"升级为"好用且可控"。

&emsp;&emsp;学完本章，你已掌握了从「Notebook 跑通」到「生产稳定运行」的完整能力图谱：6.1 节的六要点速查清单解决了 add() 延迟、/tmp 持久化、LLM 选型、成本控制、TTL 管理等工程问题；6.2 节的记忆质量管理五模式——分类标签、Why 元数据、防御性读取、Dream 整合、智能提取节流——覆盖了记忆生命周期的写入质量、检索可信度、离线维护三个维度，让 `mem0` 从「能用」升级为「好用且可控」。

## <center>第七章：课程总结与进阶路径</center>

&emsp;&emsp;经过六章的系统学习，你已经从理论到实战完整掌握了 `mem0` 生产级 Agent 长期记忆系统。本章用三个维度收束全部内容：知识映射回顾、进阶方向指引、学习成果清单。

### 7.1 映射回顾表

&emsp;&emsp;让我们用一张完整的映射表，将基础入门课件的概念与 `mem0` 的实现一一对应，形成闭环认知：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>基础入门课件 → mem0 完整概念映射回顾（仅长期记忆层）</font></p>
<div class="center">

| 前序概念 | 前序实现 | mem0 实现 | 升级维度 |
|---------|---------|----------|----------|
| 写入机制 | LLM 判断写入 + 离线 sleep-time 整理 | LLM 裁判两阶段提取（ADD/UPDATE/DELETE/NONE） | 离线批处理 → 实时冲突解决 |
| `MEMORY.md` 注入 + 可选 RAG | 文件全文注入 / LlamaIndex 向量索引 | `memory.search()` 自动管理 | 文件手动维护 → 框架自动管理 |
| 图数据库（概念） | 模块三 4.3 节 理论介绍 | Graph Memory（Neo4j/Memgraph） | 概念教学 → 开箱即用 |

</div>

> &emsp;**短期记忆层不变**：会话隔离（`sessions/`）、`SessionManager` 三阶段架构、压缩摘要机制全部保留。mem0 的 `user_id`/`agent_id`/`run_id` 是长期记忆的命名空间隔离，与 `sessions/` 并存于不同层级。


&emsp;&emsp;从这张表可以看出一个清晰的升级路径：`mem0` 并没有发明全新的概念，而是将基础入门课件中我们手动实现的每个机制，升级为自动化、框架化、可维护的生产级实现。理解了基础入门课件的底层原理，你在使用 `mem0` 时就能准确理解每个 API 背后在做什么——这正是基础入门课件的价值所在。

&emsp;&emsp;但映射表只展示了第一层升级：**从手动到自动化**。6.2 节借鉴 Claude Code 的五个生产模式揭示了第二层升级空间——**从自动化到结构化+防御性+自主维护**。完整的进化路径是：

&emsp;&emsp;`手动管理（MEMORY.md）` → `自动化（mem0 LLM 裁判）` → `自动化 + 结构化 + 防御性（融合方案）`

&emsp;&emsp;第三层升级不需要换框架——只需善用 `mem0` 已有的 `metadata` 机制，就能将分类标签、Why 元数据和防御性读取融入你的记忆系统。这也引出了下一节的进阶方向。

### 7.2 进阶方向

&emsp;&emsp;掌握了本课件的内容后，以下是几个值得深入探索的进阶方向：

&emsp;&emsp;**方向一：自定义记忆 Schema**。`mem0` 默认将记忆存储为自由格式的文本。在特定场景下（如电商用户画像、医疗健康档案），你可能需要定义结构化的记忆 Schema，让提取结果更规范、检索更精确。6.2.1 节的分类标签是 Schema 设计的起点——从四类标签出发，可以进一步为每类记忆定义专属的结构化字段。

&emsp;&emsp;**方向二：MCP 集成**。`mem0` 已支持 MCP（Model Context Protocol）集成，可以作为 MCP Server 为任何支持 MCP 的 AI 应用提供记忆服务。这是一个非常有前景的集成方向。

&emsp;&emsp;**方向三：多 Agent 共享记忆**。在多 Agent 协作系统中，Agent 之间可以通过 `agent_id` 共享记忆空间，或通过 `user_id` 共享用户偏好。这开启了 Agent 间知识传递的可能性。例如 Claude Code 的多智能体架构中，Orchestrator 和 Reviewer 通过共享目录传递上下文（类似经典的黑板架构模式（Blackboard Pattern，这是类比说法而非 Claude Code 官方命名）——`mem0` 的 `agent_id` 命名空间天然支持类似的协作模式。

&emsp;&emsp;**方向四：记忆可解释性**。通过 `memory.history()` API 和自定义的审计日志，可以构建记忆系统的可解释性层——让用户知道 AI 记住了什么、为什么记住、什么时候更新过。6.2.2 节的 `why` 字段和 6.2.3 节的新鲜度标注，正是可解释性的基础组件——在此基础上，可以进一步构建面向终端用户的"记忆管理面板"。

### 7.3 学习成果

&emsp;&emsp;完成本课件的学习后，你应该能够独立完成以下任务：

&emsp;&emsp;1. **理解 mem0 核心架构**：清晰说出 LLM 裁判两阶段流程、ADD/UPDATE/DELETE/NONE 四操作的含义和触发条件

&emsp;&emsp;2. **独立完成 mem0 配置与基本使用**：配置 DeepSeek + OpenAI 双凭证、完成 add/search/get_all/delete 全套操作

&emsp;&emsp;3. **在 LangChain/FastAPI 项目中集成 mem0**：使用 Tool 模式接入 LangChain，使用 AsyncMemory + create_task 接入 FastAPI


&emsp;&emsp;4. **做出选型决策**：基于六维对比表和选型决策树（3.1 节）、场景适配矩阵（3.3 节），在不同记忆框架方案之间做出合理选择

&emsp;&emsp;5. **掌握生产部署避坑要点**：解决 add() 延迟、/tmp 数据丢失、LLM 质量依赖、成本控制、TTL 管理等问题

&emsp;&emsp;6. **应用记忆质量管理五模式**：通过分类标签、Why 元数据、防御性读取、新鲜度认知工程、Dream 整合与智能提取（6.2 节），让 `mem0` 从"能用"升级为"好用且可控"

&emsp;&emsp;回顾学习路径，你已经完成了从「理解记忆系统原理」（基础入门课件）到「掌握生产级记忆框架」（本课件）的完整进阶。基础入门课件让你知道了「记忆系统是怎么回事」，本课件让你知道了「如何用成熟的开源工具快速落地」。两者结合，你已经具备在真实 Agent 项目中独立设计和实施记忆系统的完整能力。